In [1]:
import numpy as np
import torch
import torch.nn as nn
import tensorflow as tf
import deepof.model_utils
import deepof.clustering.model_utils_new

# Test parameters matching the full model
batch_size = 2
time_steps = 25
n_nodes = 11
n_edges = 11
features_per_node = 3
features_per_edge = 1
latent_dim = 8

# Create test data
np.random.seed(42)
test_nodes = np.random.randn(batch_size, n_nodes, time_steps, features_per_node).astype(np.float32)
test_edges = np.random.randn(batch_size, n_edges, time_steps, features_per_edge).astype(np.float32)

print(f"Test node data shape: {test_nodes.shape}")
print(f"Test edge data shape: {test_edges.shape}")

Test node data shape: (2, 11, 25, 3)
Test edge data shape: (2, 11, 25, 1)


In [ ]:
def permute_gru_weights(keras_weights):
    """Permutes GRU weights from Keras (z, r, n) to PyTorch (r, z, n) format."""
    W_ih, W_hh, B = keras_weights
    # Keras gate order: z, r, n (update, reset, new/candidate)
    W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
    W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)

    # PyTorch gate order: r, z, n (reset, update, new/candidate)
    W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
    W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)

    # Keras has two bias vectors (input-hidden and recurrent), which are concatenated in B
    B_ih, B_hh = B
    B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
    B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)

    B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
    B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])

    return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt

def transfer_recurrent_decoder_weights(tf_model, pt_model):
    """
    Transfers weights for the full recurrent decoder model.
    """
    # Find layers by type to avoid index issues
    bidi_layers = [l for l in tf_model.layers if isinstance(l, Bidirectional)]
    norm_layers = [l for l in tf_model.layers if isinstance(l, LayerNormalization)]
    conv_layers = [l for l in tf_model.layers if isinstance(l, tf.keras.layers.Conv1D)]
    prob_dec_layer = next(l for l in tf_model.layers if isinstance(l, deepof.model_utils.ProbabilisticDecoder))

    # --- GRU 1 and Norm 1 ---
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(bidi_layers[0].forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1)
    pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(bidi_layers[0].backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1)
    pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)
    pt_model.norm1.weight.data = torch.from_numpy(norm_layers[0].get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm_layers[0].get_weights()[1])

    # --- GRU 2 and Norm 2 ---
    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(bidi_layers[1].forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2)
    pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(bidi_layers[1].backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2)
    pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    pt_model.norm2.weight.data = torch.from_numpy(norm_layers[1].get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm_layers[1].get_weights()[1])

    # --- Conv1D and Norm 3 ---
    # TF Conv1D weights: (kernel_w, kernel_h, in_c, out_c) -> (5, 1, 4*ld, 2*ld)
    # PT Conv1d weights: (out_c, in_c, kernel_w)
    conv_weights_tf = conv_layers[0].get_weights()[0]
    pt_model.conv1d.weight.data = torch.from_numpy(conv_weights_tf).squeeze(1).permute(2, 1, 0)
    pt_model.norm3.weight.data = torch.from_numpy(norm_layers[2].get_weights()[0]); pt_model.norm3.bias.data = torch.from_numpy(norm_layers[2].get_weights()[1])

    # --- Probabilistic Decoder ---
    # TF Dense weights: (in_features, out_features)
    # PT Linear weights: (out_features, in_features)
    prob_dec_weights, prob_dec_bias = prob_dec_layer.time_distributer.get_weights()
    pt_model.prob_decoder.loc_projection.weight.data = torch.from_numpy(prob_dec_weights.T)
    pt_model.prob_decoder.loc_projection.bias.data = torch.from_numpy(prob_dec_bias)


import numpy as np
import torch
import tensorflow as tf
from tensorflow.keras.layers import TimeDistributed, Conv1D, Bidirectional, LayerNormalization, Dense


def _to_torch(x):
    return torch.from_numpy(np.asarray(x))


def _extract_conv1d_weight(conv_layer):
    # Keras Conv1D kernel: (k, in_c, out_c)
    w = conv_layer.get_weights()[0]
    if w.ndim == 3:
        # -> PyTorch Conv1d: (out_c, in_c, k)
        return np.transpose(w, (2, 1, 0))
    elif w.ndim == 4 and w.shape[1] == 1:
        # Sometimes shows as (k, 1, in_c, out_c)
        w = np.squeeze(w, axis=1)
        return np.transpose(w, (2, 1, 0))
    else:
        raise RuntimeError(f"Unexpected Conv1D kernel shape: {w.shape}")


def _extract_gru_raw_weights(gru_layer):
    """
    Return (W_ih, W_hh, B_ih, B_hh) from a Keras GRU layer, normalized to:
      W_ih: (in, 3*units), W_hh: (units, 3*units)
      B_ih: (3*units,), B_hh: (3*units,)
    Handles reset_after=True case where bias can be 2D (2, 3*units).
    """
    ws = gru_layer.get_weights()
    units = gru_layer.units

    if len(ws) == 3:
        W_ih, W_hh, B = ws
        # B can be:
        # - flat (2*3*units,)
        # - 2D (2, 3*units)
        # - flat (3*units,) if reset_after=False (rare)
        if B.ndim == 2 and B.shape[0] == 2 and B.shape[1] == 3 * units:
            B_ih, B_hh = B[0].copy(), B[1].copy()
        elif B.ndim == 1 and B.shape[0] == 2 * 3 * units:
            half = 3 * units
            B_ih, B_hh = B[:half].copy(), B[half:].copy()
        elif B.ndim == 1 and B.shape[0] == 3 * units:
            # No recurrent bias; provide zeros for recurrent bias to match PyTorch expectations
            B_ih, B_hh = B.copy(), np.zeros_like(B)
        else:
            raise RuntimeError(f"Unexpected GRU bias shape: {B.shape}")
    elif len(ws) == 4:
        W_ih, W_hh, B_ih, B_hh = ws
        # Ensure 1D
        B_ih = B_ih.reshape(-1).copy()
        B_hh = B_hh.reshape(-1).copy()
    else:
        raise RuntimeError(f"Unexpected number of GRU weights: {len(ws)}")

    return W_ih, W_hh, B_ih, B_hh


def _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units):
    """
    Keras gate order: z, r, n -> PyTorch: r, z, n.
    Use explicit slicing by units (no np.split to avoid shape issues).
    Returns:
      W_ih_pt: (3h, in), W_hh_pt: (3h, h), B_ih_pt: (3h,), B_hh_pt: (3h,)
    """
    # W_ih: (in, 3*units)
    W_ih_z = W_ih[:, 0 * units:1 * units]
    W_ih_r = W_ih[:, 1 * units:2 * units]
    W_ih_n = W_ih[:, 2 * units:3 * units]
    W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1).T  # -> (3*units, in)

    # W_hh: (units, 3*units)
    W_hh_z = W_hh[:, 0 * units:1 * units]
    W_hh_r = W_hh[:, 1 * units:2 * units]
    W_hh_n = W_hh[:, 2 * units:3 * units]
    W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1).T  # -> (3*units, units)

    # Biases: (3*units,)
    B_ih_z = B_ih[0 * units:1 * units]
    B_ih_r = B_ih[1 * units:2 * units]
    B_ih_n = B_ih[2 * units:3 * units]
    B_hh_z = B_hh[0 * units:1 * units]
    B_hh_r = B_hh[1 * units:2 * units]
    B_hh_n = B_hh[2 * units:3 * units]

    B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])  # (3*units,)
    B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])  # (3*units,)

    return W_ih_pt, W_hh_pt, B_ih_pt, B_hh_pt


def transfer_recurrent_block_weights(tf_block_model, pt_block):
    """
    Minimal transfer for a single recurrent block (node or edge):
      - TimeDistributed(Conv1D)
      - TimeDistributed(Bidirectional(GRU)) x 2
      - LayerNormalization x 2
    """
    # Conv1D (inside TimeDistributed)
    td_convs = [l for l in tf_block_model.layers if isinstance(l, TimeDistributed) and isinstance(l.layer, Conv1D)]
    assert len(td_convs) >= 1, "No TimeDistributed(Conv1D) found in recurrent block"
    conv_td = td_convs[0]
    conv_w_pt = _extract_conv1d_weight(conv_td.layer)
    pt_block.conv1d.weight.data = _to_torch(conv_w_pt)
    if len(conv_td.layer.get_weights()) > 1 and pt_block.conv1d.bias is not None:
        pt_block.conv1d.bias.data = _to_torch(conv_td.layer.get_weights()[1])

    # Two TimeDistributed(Bidirectional(GRU))
    td_bis = [l for l in tf_block_model.layers if isinstance(l, TimeDistributed) and isinstance(l.layer, Bidirectional)]
    assert len(td_bis) >= 2, f"Expected 2 TimeDistributed(Bidirectional) GRUs, got {len(td_bis)}"
    bi1, bi2 = td_bis[0].layer, td_bis[1].layer

    # Norm layers
    norms = [l for l in tf_block_model.layers if isinstance(l, LayerNormalization)]
    assert len(norms) >= 2, f"Expected 2 LayerNormalization layers, got {len(norms)}"

    # First BiGRU -> pt_block.gru1 + norm1
    units1 = bi1.forward_layer.units
    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi1.forward_layer)
    W_ih_f, W_hh_f, B_ih_f, B_hh_f = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units1)
    pt_block.gru1.weight_ih_l0.data = _to_torch(W_ih_f)
    pt_block.gru1.weight_hh_l0.data = _to_torch(W_hh_f)
    pt_block.gru1.bias_ih_l0.data = _to_torch(B_ih_f)
    pt_block.gru1.bias_hh_l0.data = _to_torch(B_hh_f)

    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi1.backward_layer)
    W_ih_b, W_hh_b, B_ih_b, B_hh_b = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units1)
    pt_block.gru1.weight_ih_l0_reverse.data = _to_torch(W_ih_b)
    pt_block.gru1.weight_hh_l0_reverse.data = _to_torch(W_hh_b)
    pt_block.gru1.bias_ih_l0_reverse.data = _to_torch(B_ih_b)
    pt_block.gru1.bias_hh_l0_reverse.data = _to_torch(B_hh_b)

    pt_block.norm1.weight.data = _to_torch(norms[0].get_weights()[0])
    pt_block.norm1.bias.data = _to_torch(norms[0].get_weights()[1])

    # Second BiGRU -> pt_block.gru2 + norm2
    units2 = bi2.forward_layer.units
    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi2.forward_layer)
    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units2)
    pt_block.gru2.weight_ih_l0.data = _to_torch(W_ih_f2)
    pt_block.gru2.weight_hh_l0.data = _to_torch(W_hh_f2)
    pt_block.gru2.bias_ih_l0.data = _to_torch(B_ih_f2)
    pt_block.gru2.bias_hh_l0.data = _to_torch(B_hh_f2)

    W_ih, W_hh, B_ih, B_hh = _extract_gru_raw_weights(bi2.backward_layer)
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = _permute_gru_keras_to_torch(W_ih, W_hh, B_ih, B_hh, units2)
    pt_block.gru2.weight_ih_l0_reverse.data = _to_torch(W_ih_b2)
    pt_block.gru2.weight_hh_l0_reverse.data = _to_torch(W_hh_b2)
    pt_block.gru2.bias_ih_l0_reverse.data = _to_torch(B_ih_b2)
    pt_block.gru2.bias_hh_l0_reverse.data = _to_torch(B_hh_b2)

    pt_block.norm2.weight.data = _to_torch(norms[1].get_weights()[0])
    pt_block.norm2.bias.data = _to_torch(norms[1].get_weights()[1])


def transfer_censnet_weights(tf_layer, pt_layer):
    """
    Spektral CensNetConv -> CensNetConvPT:
      Expected TF order: node_kernel, edge_kernel, node_weights, edge_weights, node_bias, edge_bias
    """
    kn_tf, ke_tf, pn_tf, pe_tf, bn_tf, be_tf = tf_layer.get_weights()
    pt_layer.node_kernel.data = _to_torch(kn_tf)
    pt_layer.edge_kernel.data = _to_torch(ke_tf)

    pn_tf = pn_tf.reshape(-1, 1) if pn_tf.ndim == 1 else pn_tf
    pe_tf = pe_tf.reshape(-1, 1) if pe_tf.ndim == 1 else pe_tf
    pt_layer.node_weights.data = _to_torch(pn_tf)
    pt_layer.edge_weights.data = _to_torch(pe_tf)

    pt_layer.node_bias.data = _to_torch(bn_tf)
    pt_layer.edge_bias.data = _to_torch(be_tf)


def _flatten_keras_tensors(x):
    if isinstance(x, (list, tuple)):
        out = []
        for xi in x:
            out.extend(_flatten_keras_tensors(xi))
        return out
    return [x]

def _layer_from_tensor(t):
    kh = getattr(t, "_keras_history", None)
    return kh[0] if isinstance(kh, tuple) else getattr(kh, "layer", None)

def transfer_recurrent_encoder_weights(tf_model, pt_model, verbose=False):
    try:
        tf_enc = tf_model.get_layer("recurrent_encoder")
    except Exception:
        tf_enc = tf_model

    # Final Dense
    w, b = tf_enc.layers[-1].get_weights()
    pt_model.final_dense.weight.data = torch.from_numpy(w.T)
    pt_model.final_dense.bias.data = torch.from_numpy(b)

    if not getattr(pt_model, "use_gnn", False):
        sub = [l for l in tf_enc.layers if isinstance(l, tf.keras.Model)][0]
        transfer_recurrent_block_weights(sub, pt_model.recurrent_block)
        return

    node_in_ch = int(pt_model.node_recurrent_block.conv1d.in_channels)
    edge_in_ch = int(pt_model.edge_recurrent_block.conv1d.in_channels)
    gnn_tf, _, _, tf_node_block, tf_edge_block = _pick_gnn_pre_embeddings(tf_enc, node_in_ch, edge_in_ch)

    transfer_recurrent_block_weights(tf_node_block, pt_model.node_recurrent_block)
    transfer_recurrent_block_weights(tf_edge_block, pt_model.edge_recurrent_block)
    transfer_censnet_weights(gnn_tf, pt_model.spatial_gnn_block)

    if verbose:
        print(f"Transferred node from {tf_node_block.name} (in_ch={node_in_ch}), edge from {tf_edge_block.name} (in_ch={edge_in_ch})")


In [6]:
import deepof.clustering.models_new


def transfer_vq_weights(tf_model: deepof.models.VectorQuantizer, pt_model: deepof.clustering.models_new.VectorQuantizerPT):
    """Transfers weights from TensorFlow VectorQuantizer to PyTorch version.
    
    Args:
        tf_model: TensorFlow VectorQuantizer model
        pt_model: PyTorch VectorQuantizerPT model
    """
    # Transfer codebook weights
    # TF shape: (embedding_dim, n_components)
    # PT shape: (embedding_dim, n_components)
    # These match, so direct transfer
    codebook_weights = tf_model.codebook.numpy()
    pt_model.codebook.data = torch.from_numpy(codebook_weights)
    
    print(f"✅ Transferred codebook weights: {codebook_weights.shape}")

def transfer_vqvae_weights(tf_vqvae, pt_vqvae):
    """
    Transfer weights from TensorFlow VQVAE to PyTorch VQVAEPT.
    
    Args:
        tf_vqvae: TensorFlow VQVAE instance
        pt_vqvae: PyTorch VQVAEPT instance
    """
    print("Transferring VQ-VAE weights...")
    
    # Transfer encoder weights (assumed already done via encoder-specific transfer functions)
    # transfer_encoder_weights(tf_vqvae.encoder, pt_vqvae.encoder)
    print("  ⚠️  Encoder weights - use encoder-specific transfer function")
    
    # Transfer VQ layer weights
    tf_vq_layer = tf_vqvae.vqvae.get_layer('vector_quantizer')
    transfer_vq_weights(tf_vq_layer, pt_vqvae.vq_layer)
    print("  ✅ VQ layer weights transferred")
    
    # Transfer decoder weights (assumed already done via decoder-specific transfer functions)
    # transfer_decoder_weights(tf_vqvae.decoder, pt_vqvae.decoder)
    print("  ⚠️  Decoder weights - use decoder-specific transfer function")
    
    print("✅ VQ-VAE weight transfer complete!")

In [5]:
# Create TF recurrent block for nodes
tf_node_input = tf.keras.Input(shape=(n_nodes, time_steps, features_per_node))
tf_node_recurrent = deepof.model_utils.get_recurrent_block(
    tf_node_input, 
    latent_dim, 
    gru_unroll=False, 
    bidirectional_merge="concat"
)(tf_node_input)
tf_node_model = tf.keras.Model(inputs=tf_node_input, outputs=tf_node_recurrent)

# Create TF recurrent block for edges
tf_edge_input = tf.keras.Input(shape=(n_edges, time_steps, features_per_edge))
tf_edge_recurrent = deepof.model_utils.get_recurrent_block(
    tf_edge_input,
    latent_dim,
    gru_unroll=False,
    bidirectional_merge="concat"
)(tf_edge_input)
tf_edge_model = tf.keras.Model(inputs=tf_edge_input, outputs=tf_edge_recurrent)

# Test forward pass
tf_node_output = tf_node_model(test_nodes, training=False)
tf_edge_output = tf_edge_model(test_edges, training=False)

print(f"TF node model output shape: {tf_node_output.shape}")
print(f"TF edge model output shape: {tf_edge_output.shape}")
print(f"TF node output sample: {tf_node_output[0, 0, :5].numpy()}")
print(f"TF edge output sample: {tf_edge_output[0, 0, :5].numpy()}")

# Display model structure
print("\nTF Node Model layers:")
for i, layer in enumerate(tf_node_model.layers):
    print(f"  {i}: {layer.name} - {type(layer).__name__}")
    if hasattr(layer, 'layers'):  # If it's a nested model
        for j, sublayer in enumerate(layer.layers):
            print(f"    {j}: {sublayer.name} - {type(sublayer).__name__}")

TF node model output shape: (2, 11, 16)
TF edge model output shape: (2, 11, 16)
TF node output sample: [-0.18209487  1.2535563   0.6659341   1.3584192   1.3101838 ]
TF edge output sample: [ 1.2105745e+00  3.3642778e-01  1.5770061e+00 -5.5604172e-04
 -1.4515566e+00]

TF Node Model layers:
  0: input_1 - InputLayer
  1: model - Functional
    0: input_1 - InputLayer
    1: time_distributed - TimeDistributed
    2: masking - Masking
    3: time_distributed_1 - TimeDistributed
    4: layer_normalization - LayerNormalization
    5: time_distributed_2 - TimeDistributed
    6: layer_normalization_1 - LayerNormalization


In [6]:
# Create PyTorch recurrent blocks
pt_node_recurrent = deepof.clustering.model_utils_new.RecurrentBlockPT(
    input_features=features_per_node, 
    latent_dim=latent_dim
)

pt_edge_recurrent = deepof.clustering.model_utils_new.RecurrentBlockPT(
    input_features=features_per_edge,
    latent_dim=latent_dim
)



# Transfer weights
transfer_recurrent_block_weights(tf_node_model.layers[-1], pt_node_recurrent)
transfer_recurrent_block_weights(tf_edge_model.layers[-1], pt_edge_recurrent)

In [7]:
def compare_recurrent_outputs(tf_model, pt_model, test_data, block_name):
    """Compare TF and PT recurrent block outputs."""
    print(f"\n{'='*50}")
    print(f"Comparing {block_name}")
    print(f"{'='*50}")
    
    # TF forward pass
    tf_output = tf_model(test_data, training=False).numpy()
    
    # PT forward pass
    pt_model.eval()
    with torch.no_grad():
        pt_input = torch.from_numpy(test_data)
        pt_output = pt_model(pt_input).numpy()
    
    # Compare
    max_diff = np.abs(tf_output - pt_output).max()
    mean_diff = np.abs(tf_output - pt_output).mean()
    
    print(f"Output shapes: TF {tf_output.shape}, PT {pt_output.shape}")
    print(f"Max difference: {max_diff:.8f}")
    print(f"Mean difference: {mean_diff:.8f}")
    print(f"TF output sample: {tf_output[0, 0, :5]}")
    print(f"PT output sample: {pt_output[0, 0, :5]}")
    
    # Check if outputs match
    matches = np.allclose(tf_output, pt_output, rtol=1e-5, atol=1e-5)
    print(f"Outputs match: {matches}")
    
    return max_diff, tf_output, pt_output

test_nodes[0,0,0,:]=0.0
test_nodes[1,1,:,1]=0.0
test_nodes[1,:,2,2]=0.0
test_edges[0,0,0,:]=0.0
test_edges[1,1,:,0]=0.0
test_edges[1,:,2,0]=0.0

# Compare node recurrent block
node_diff, tf_node_out, pt_node_out = compare_recurrent_outputs(
    tf_node_model, pt_node_recurrent, test_nodes, "Node Recurrent Block"
)

# Compare edge recurrent block
edge_diff, tf_edge_out, pt_edge_out = compare_recurrent_outputs(
    tf_edge_model, pt_edge_recurrent, test_edges, "Edge Recurrent Block"
)

# Additional detailed comparison
if node_diff > 1e-5 or edge_diff > 1e-5:
    print("\n" + "="*50)
    print("Detailed difference analysis")
    print("="*50)
    
    if node_diff > 1e-5:
        node_diffs = np.abs(tf_node_out - pt_node_out)
        worst_idx = np.unravel_index(np.argmax(node_diffs), node_diffs.shape)
        print(f"Node worst difference at index {worst_idx}:")
        print(f"  TF value: {tf_node_out[worst_idx]}")
        print(f"  PT value: {pt_node_out[worst_idx]}")
        print(f"  Difference: {node_diffs[worst_idx]}")
    
    if edge_diff > 1e-5:
        edge_diffs = np.abs(tf_edge_out - pt_edge_out)
        worst_idx = np.unravel_index(np.argmax(edge_diffs), edge_diffs.shape)
        print(f"Edge worst difference at index {worst_idx}:")
        print(f"  TF value: {tf_edge_out[worst_idx]}")
        print(f"  PT value: {pt_edge_out[worst_idx]}")
        print(f"  Difference: {edge_diffs[worst_idx]}")


Comparing Node Recurrent Block
Output shapes: TF (2, 11, 16), PT (2, 11, 16)
Max difference: 0.00000066
Mean difference: 0.00000013
TF output sample: [-0.16907986  1.2758803   0.68413067  1.3812369   1.332848  ]
PT output sample: [-0.16907997  1.2758803   0.6841305   1.3812369   1.332848  ]
Outputs match: True

Comparing Edge Recurrent Block
Output shapes: TF (2, 11, 16), PT (2, 11, 16)
Max difference: 0.00000083
Mean difference: 0.00000012
TF output sample: [ 1.252241    0.31836933  1.6439373  -0.04645128 -1.5955242 ]
PT output sample: [ 1.2522411   0.3183693   1.6439375  -0.04645133 -1.5955244 ]
Outputs match: True


In [21]:
import numpy as np
import torch
import tensorflow as tf
from spektral.layers import CensNetConv
from deepof.clustering.censNetConv_pt import CensNetConvPT

# Create a simple adjacency matrix for testing
def create_test_adjacency(n_nodes=4):
    """Create a simple test adjacency matrix."""
    adj = np.zeros((n_nodes, n_nodes), dtype=np.float32)  # Ensure float32
    # Create a simple connected graph
    for i in range(n_nodes - 1):
        adj[i, i+1] = 1
        adj[i+1, i] = 1
    # Add one more edge to make it interesting
    if n_nodes > 2:
        adj[0, n_nodes-1] = 1
        adj[n_nodes-1, 0] = 1
    return adj

# Test parameters
n_nodes = 4
n_edges = 4  # For our simple graph
batch_size = 2
input_node_features = 8
input_edge_features = 8
output_channels = 4

adj_matrix = create_test_adjacency(n_nodes)
print("Adjacency matrix:")
print(adj_matrix)

Adjacency matrix:
[[0. 1. 0. 1.]
 [1. 0. 1. 0.]
 [0. 1. 0. 1.]
 [1. 0. 1. 0.]]


In [22]:
# Create TensorFlow CensNetConv layer
tf_layer = CensNetConv(
    node_channels=output_channels,
    edge_channels=output_channels,
    activation='relu'
)

# Preprocess adjacency for TF
tf_lap, tf_edge_lap, tf_inc = tf_layer.preprocess(adj_matrix)
print(f"TF Laplacian shape: {tf_lap.shape}")
print(f"TF Edge Laplacian shape: {tf_edge_lap.shape}")
print(f"TF Incidence shape: {tf_inc.shape}")

# Create test inputs for TF
tf_node_features = tf.random.normal((batch_size, n_nodes, input_node_features))
tf_edge_features = tf.random.normal((batch_size, n_edges, input_edge_features))

# Build the layer by calling it once
tf_output = tf_layer([tf_node_features, (tf_lap, tf_edge_lap, tf_inc), tf_edge_features])
print(f"\nTF output shapes: nodes={tf_output[0].shape}, edges={tf_output[1].shape}")

# Get weights
tf_weights = tf_layer.get_weights()
print(f"\nTF layer weights count: {len(tf_weights)}")
for i, w in enumerate(tf_weights):
    print(f"  Weight {i}: shape {w.shape}")

TF Laplacian shape: (4, 4)
TF Edge Laplacian shape: (4, 4)
TF Incidence shape: (4, 4)

TF output shapes: nodes=(2, 4, 4), edges=(2, 4, 4)

TF layer weights count: 6
  Weight 0: shape (8, 4)
  Weight 1: shape (8, 4)
  Weight 2: shape (8, 1)
  Weight 3: shape (8, 1)
  Weight 4: shape (4,)
  Weight 5: shape (4,)


The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.


In [23]:
# Create PyTorch CensNetConvPT layer
pt_layer = CensNetConvPT(
    node_channels=output_channels,
    edge_channels=output_channels,
    activation='relu'
)

# Initialize with correct dimensions
pt_layer._build(
    node_features_shape=[batch_size, n_nodes, input_node_features],
    edge_features_shape=[batch_size, n_edges, input_edge_features]
)

transfer_censnet_weights(tf_layer, pt_layer)

# Preprocess adjacency for PT - ensure float32
pt_lap, pt_edge_lap, pt_inc = pt_layer.preprocess(torch.tensor(adj_matrix, dtype=torch.float32))
print(f"\nPT Laplacian shape: {pt_lap.shape}")
print(f"PT Edge Laplacian shape: {pt_edge_lap.shape}")
print(f"PT Incidence shape: {pt_inc.shape}")


PT Laplacian shape: torch.Size([4, 4])
PT Edge Laplacian shape: torch.Size([4, 4])
PT Incidence shape: torch.Size([4, 4])


In [24]:
def compare_outputs(tf_layer, pt_layer, test_name, tf_nodes, tf_edges, pt_nodes, pt_edges, 
                    tf_lap, tf_edge_lap, tf_inc, pt_lap, pt_edge_lap, pt_inc):
    """Compare TF and PT outputs for given inputs."""
    print(f"\n{'='*50}")
    print(f"Test: {test_name}")
    print(f"{'='*50}")
    
    # Ensure all PT tensors are float32
    pt_nodes = pt_nodes.float()
    pt_edges = pt_edges.float()
    pt_lap = pt_lap.float()
    pt_edge_lap = pt_edge_lap.float()
    pt_inc = pt_inc.float()
    
    # TF forward pass
    tf_out_nodes, tf_out_edges = tf_layer([tf_nodes, (tf_lap, tf_edge_lap, tf_inc), tf_edges])
    
    # PT forward pass
    pt_layer.eval()
    with torch.no_grad():
        pt_out_nodes, pt_out_edges = pt_layer([pt_nodes, (pt_lap, pt_edge_lap, pt_inc), pt_edges])
    
    # Convert to numpy for comparison
    tf_out_nodes_np = tf_out_nodes.numpy()
    tf_out_edges_np = tf_out_edges.numpy()
    pt_out_nodes_np = pt_out_nodes.numpy()
    pt_out_edges_np = pt_out_edges.numpy()
    
    # Compare
    node_diff = np.abs(tf_out_nodes_np - pt_out_nodes_np).max()
    edge_diff = np.abs(tf_out_edges_np - pt_out_edges_np).max()
    
    print(f"Node output diff: {node_diff:.8f}")
    print(f"Edge output diff: {edge_diff:.8f}")
    print(f"TF node output sample: {tf_out_nodes_np[0, 0, :3]}")
    print(f"PT node output sample: {pt_out_nodes_np[0, 0, :3]}")
    print(f"TF edge output sample: {tf_out_edges_np[0, 0, :3]}")
    print(f"PT edge output sample: {pt_out_edges_np[0, 0, :3]}")
    
    return node_diff, edge_diff

# Test 1: Random inputs
print("Test 1: Random inputs")
tf_nodes_random = tf.random.normal((batch_size, n_nodes, input_node_features), seed=42)
tf_edges_random = tf.random.normal((batch_size, n_edges, input_edge_features), seed=43)
pt_nodes_random = torch.from_numpy(tf_nodes_random.numpy()).float()
pt_edges_random = torch.from_numpy(tf_edges_random.numpy()).float()

compare_outputs(tf_layer, pt_layer, "Random inputs", 
                tf_nodes_random, tf_edges_random, pt_nodes_random, pt_edges_random,
                tf_lap, tf_edge_lap, tf_inc, pt_lap, pt_edge_lap, pt_inc)

# Test 2: Simple inputs (ones)
print("\nTest 2: Ones")
tf_nodes_ones = tf.ones((batch_size, n_nodes, input_node_features))
tf_edges_ones = tf.ones((batch_size, n_edges, input_edge_features))
pt_nodes_ones = torch.ones((batch_size, n_nodes, input_node_features), dtype=torch.float32)
pt_edges_ones = torch.ones((batch_size, n_edges, input_edge_features), dtype=torch.float32)

compare_outputs(tf_layer, pt_layer, "Ones", 
                tf_nodes_ones, tf_edges_ones, pt_nodes_ones, pt_edges_ones,
                tf_lap, tf_edge_lap, tf_inc, pt_lap, pt_edge_lap, pt_inc)

# Test 3: Identity-like pattern
print("\nTest 3: Identity pattern")
tf_nodes_eye = tf.constant(np.eye(input_node_features, dtype=np.float32)[:n_nodes], dtype=tf.float32)
tf_nodes_eye = tf.expand_dims(tf_nodes_eye, 0)
tf_nodes_eye = tf.tile(tf_nodes_eye, [batch_size, 1, 1])
tf_edges_eye = tf.constant(np.eye(input_edge_features, dtype=np.float32)[:n_edges], dtype=tf.float32)
tf_edges_eye = tf.expand_dims(tf_edges_eye, 0)
tf_edges_eye = tf.tile(tf_edges_eye, [batch_size, 1, 1])

pt_nodes_eye = torch.from_numpy(tf_nodes_eye.numpy()).float()
pt_edges_eye = torch.from_numpy(tf_edges_eye.numpy()).float()

compare_outputs(tf_layer, pt_layer, "Identity pattern", 
                tf_nodes_eye, tf_edges_eye, pt_nodes_eye, pt_edges_eye,
                tf_lap, tf_edge_lap, tf_inc, pt_lap, pt_edge_lap, pt_inc)

# Check if preprocessing outputs match
print("\n" + "="*50)
print("Checking preprocessing outputs:")
print("="*50)
print(f"Laplacian match: {np.allclose(tf_lap, pt_lap.numpy())}")
print(f"Edge Laplacian match: {np.allclose(tf_edge_lap, pt_edge_lap.numpy())}")
print(f"Incidence match: {np.allclose(tf_inc, pt_inc.numpy())}")
print(f"Laplacian diff: {np.abs(tf_lap - pt_lap.numpy()).max():.8f}")
print(f"Edge Laplacian diff: {np.abs(tf_edge_lap - pt_edge_lap.numpy()).max():.8f}")
print(f"Incidence diff: {np.abs(tf_inc - pt_inc.numpy()).max():.8f}")

Test 1: Random inputs

Test: Random inputs
DEBUG _propagate_nodes: node_features shape: torch.Size([2, 4, 8])
DEBUG _propagate_nodes: edge_features shape: torch.Size([2, 4, 8])
DEBUG _propagate_nodes: self.node_kernel shape: torch.Size([8, 4])
DEBUG _propagate_nodes: self.edge_weights shape: torch.Size([8, 1])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 8]), b.shape=torch.Size([8, 1])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 4]), b.shape=torch.Size([4, 4])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 8]), b.shape=torch.Size([8, 4])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 8]), b.shape=torch.Size([8, 1])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 4]), b.shape=torch.Size([4, 4])
DEBUG modal_dot_pt: About to einsum with a.shape=torch.Size([2, 4, 8]), b.shape=torch.Size([8, 4])
Node output diff: 0.00000000
Edge output diff: 0.00000000
TF node output sample: [0.19316557 0

# Vector quantizer test

In [6]:
from typing import Any, NewType, Iterable, Tuple
import unittest
import numpy as np
import tcn
import sys
import tensorflow as tf
import tensorflow_probability as tfp
from sklearn.mixture import GaussianMixture
from spektral.layers import CensNetConv
from tensorflow.keras import Input, Model
from tensorflow.keras.initializers import he_uniform
from tensorflow.keras.layers import (
    GRU,
    Bidirectional,
    Dense,
    LayerNormalization,
    RepeatVector,
    TimeDistributed,
)
from tensorflow.keras.optimizers import Nadam
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.distributions import Distribution, TransformedDistribution, Normal
from torch.distributions.transforms import AffineTransform
import time

import deepof.model_utils
import deepof.clustering.model_utils_new
from deepof.clustering.censNetConv_pt import CensNetConvPT
import deepof.utils
from deepof.data_loading import get_dt
import warnings
from deepof.clustering.model_utils_new import ProbabilisticDecoderPT

In [7]:
import deepof.clustering.models_new


class TestRecurrentEncoderTranslation(unittest.TestCase):
    def setUp(self):
        """Set up parameters and create random data that matches model assumptions."""
        tf.keras.backend.clear_session()
        self.latent_dim = 8
        
        # Real data configuration
        self.b = 2  # Batch
        self.t = 25  # Time
        self.n = 1  # Nodes  
        self.f_per_node = 1  # Features per node
        self.f_total = self.n * self.f_per_node  # Total features (33)
        self.e = 1  # Edges
        self.f_edge_per_edge = 1  # Features per edge
        self.f_edge_total = self.e * self.f_edge_per_edge  # Total edge features

        # For encoder, use FLATTENED shapes (as VQVAE does)
        self.input_shape = (self.t, self.f_total)  # (25, 33)
        self.edge_shape = (self.t, self.f_edge_total)  # (25, 11)
        # Create dummy adjacency matrix
        m = np.zeros((self.n, self.n))
        ui = np.triu_indices(self.n)
        num_possible_edges = len(ui[0])
        c = np.random.choice(num_possible_edges, min(self.e, num_possible_edges), replace=False)
        m[ui[0][c], ui[1][c]] = 1
        m += m.T # Make symmetric
        self.adj_matrix = m

        # Check actual edge count
        actual_edge_count = np.sum(np.triu(self.adj_matrix) > 0)
        print(f"Expected edges: {self.e}, Actual edges in adjacency: {actual_edge_count}")
        print(f"Incidence matrix will have shape: (nodes={self.n}, edges={actual_edge_count})")

        # Create random input data in FLATTENED format
        self.x_np = np.random.rand(self.b, self.t, self.f_total).astype(np.float32)
        self.a_np = np.random.rand(self.b, self.t, self.f_edge_total).astype(np.float32)
        
        # For PyTorch, we need to reshape to (B, T, N, F) format
        self.x_np_pt = self.x_np.reshape(self.b, self.t, self.n, self.f_per_node)
        self.a_np_pt = self.a_np.reshape(self.b, self.t, self.e, self.f_edge_per_edge)
        
    def test_forward_pass_gnn(self):
        """Test the GNN-enabled path of the encoder."""
        # Build TF model with FLATTENED input shapes
        tf_model_gnn = deepof.models.get_recurrent_encoder(
            self.input_shape,  # (25, 33)
            self.edge_shape,   # (25, 11)
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=True
        )
        # After creating tf_model_gnn, check the incidence matrix shape
        tf_cens_layer = tf_model_gnn.get_layer('cens_net_conv')
        tf_lap, tf_edge_lap, tf_inc = tf_cens_layer.preprocess(self.adj_matrix)
        print(f"\nTF preprocessing shapes:")
        print(f"  TF Incidence shape: {tf_inc.shape}")
        print(f"  TF Edge Laplacian shape: {tf_edge_lap.shape}")

        # Debug TF model structure
        print("\nDEBUG: TensorFlow model layers:")
        for i, layer in enumerate(tf_model_gnn.layers):
            print(f"  Layer {i}: {layer.name} - {type(layer).__name__}")
            if hasattr(layer, 'output_shape'):
                print(f"    Output shape: {layer.output_shape}")
            if hasattr(layer, 'get_weights'):
                weights = layer.get_weights()
                if weights:
                    print(f"    Weight shapes: {[w.shape for w in weights]}")
        
        # Build PT model with UNFLATTENED input shapes 
        pt_input_shape = (self.t, self.n, self.f_per_node)  # (25, 11, 3)
        pt_edge_shape = (self.t, self.e, self.f_edge_per_edge)  # (25, 11, 1)
        
        pt_model_gnn = deepof.clustering.models_new.RecurrentEncoderPT(
            pt_input_shape, 
            pt_edge_shape, 
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=True
        )
        pt_model_gnn.eval()

        print(f"\nPT preprocessing shapes:")
        print(f"  PT Incidence shape: {pt_model_gnn.incidence.shape}")
        print(f"  PT Edge Laplacian shape: {pt_model_gnn.edge_laplacian.shape}")
        print(f"  PT num_edges attribute: {pt_model_gnn.num_edges}")

        # Run a single "dummy" forward pass on the PyTorch model to initialize weights
        #with torch.no_grad():
        #    pt_model_gnn(torch.from_numpy(self.x_np_pt), torch.from_numpy(self.a_np_pt))

        # Transfer weights from TF to PT
        transfer_recurrent_encoder_weights(tf_model_gnn, pt_model_gnn)

        # Execute and compare the outputs
        # TF gets flattened input
        tf_start = time.time()
        tf_output = tf_model_gnn([self.x_np, self.a_np], training=False).numpy()
        tf_end = time.time()
        
        # PT gets unflattened input
        pt_start = time.time()
        with torch.no_grad():
            pt_output = pt_model_gnn(
                torch.from_numpy(self.x_np_pt), 
                torch.from_numpy(self.a_np_pt)
            ).detach().numpy()
        pt_end = time.time()

        print(f"Tensorflow execution time: {tf_end-tf_start:.4f}s")
        print(f"Pytorch execution time: {pt_end-pt_start:.4f}s")
        print(f"TF output shape: {tf_output.shape}")
        print(f"PT output shape: {pt_output.shape}")
        print(f"TF output sample: {tf_output[0, :5]}")
        print(f"PT output sample: {pt_output[0, :5]}")
        print(f"Max absolute difference: {np.abs(tf_output - pt_output).max():.8f}")

        np.testing.assert_allclose(tf_output, pt_output, rtol=1e-5, atol=1e-4)
        print("✅ `RecurrentEncoderPT` (GNN path) translation test PASSED!")

    def test_forward_pass_no_gnn(self):
        """Test the non-GNN path of the encoder."""
        # Build TF model with FLATTENED input shapes
        tf_model_no_gnn = deepof.models.get_recurrent_encoder(
            self.input_shape,  # (25, 33)
            self.edge_shape,   # (25, 11)
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=False
        )
        
        # Build PT model with UNFLATTENED input shapes
        pt_input_shape = (self.t, self.n, self.f_per_node)  # (25, 11, 3)
        pt_edge_shape = (self.t, self.e, self.f_edge_per_edge)  # (25, 11, 1)
        
        pt_model_no_gnn = deepof.clustering.models_new.RecurrentEncoderPT(
            pt_input_shape, 
            pt_edge_shape, 
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=False
        )
        pt_model_no_gnn.eval()

        # Transfer weights
        transfer_recurrent_encoder_weights(tf_model_no_gnn, pt_model_no_gnn)

        # Execute and compare
        # TF gets flattened input
        tf_output = tf_model_no_gnn([self.x_np, self.a_np], training=False).numpy()
        
        # PT gets unflattened input
        with torch.no_grad():
            pt_output = pt_model_no_gnn(
                torch.from_numpy(self.x_np_pt), 
                torch.from_numpy(self.a_np_pt)
            ).detach().numpy()
        
        print(f"TF output shape: {tf_output.shape}")
        print(f"PT output shape: {pt_output.shape}")
        print(f"TF output sample: {tf_output[0, :5]}")
        print(f"PT output sample: {pt_output[0, :5]}")
        print(f"Max absolute difference: {np.abs(tf_output - pt_output).max():.8f}")

        np.testing.assert_allclose(tf_output, pt_output, rtol=1e-5, atol=1e-5)
        print("✅ `RecurrentEncoderPT` (non-GNN path) translation test PASSED!")

# To run:
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestRecurrentEncoderTranslation)
runner.run(suite)

test_forward_pass_gnn (__main__.TestRecurrentEncoderTranslation)
Test the GNN-enabled path of the encoder. ... 

Expected edges: 1, Actual edges in adjacency: 1
Incidence matrix will have shape: (nodes=1, edges=1)

TF preprocessing shapes:
  TF Incidence shape: (1, 1)
  TF Edge Laplacian shape: (1, 1)

DEBUG: TensorFlow model layers:
  Layer 0: input_1 - InputLayer
    Output shape: [(None, 25, 1)]
  Layer 1: input_2 - InputLayer
    Output shape: [(None, 25, 1)]
  Layer 2: tf.compat.v1.transpose - TFOpLambda
    Output shape: (1, 25, None)
  Layer 3: tf.compat.v1.transpose_2 - TFOpLambda
    Output shape: (1, 25, None)
  Layer 4: tf.reshape - TFOpLambda
    Output shape: (1, 25, 1, None)
  Layer 5: tf.reshape_1 - TFOpLambda
    Output shape: (1, 25, 1, None)
  Layer 6: tf.compat.v1.transpose_1 - TFOpLambda
    Output shape: (None, 1, 25, 1)
  Layer 7: tf.compat.v1.transpose_3 - TFOpLambda
    Output shape: (None, 1, 25, 1)
  Layer 8: model - Functional
    Output shape: (None, 1, 16)
    Weight shapes: [(5, 1, 16), (16, 48), (16, 48), (2, 48), (16, 48), (16, 48), (2, 48), (32,), (32,), (32, 24),

FAIL
test_forward_pass_no_gnn (__main__.TestRecurrentEncoderTranslation)
Test the non-GNN path of the encoder. ... 

Expected edges: 1, Actual edges in adjacency: 1
Incidence matrix will have shape: (nodes=1, edges=1)


ok

FAIL: test_forward_pass_gnn (__main__.TestRecurrentEncoderTranslation)
Test the GNN-enabled path of the encoder.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_15976\2578460083.py", line 121, in test_forward_pass_gnn
    np.testing.assert_allclose(tf_output, pt_output, rtol=1e-5, atol=1e-4)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\numpy\testing\_private\utils.py", line 1504, in assert_allclose
    assert_array_compare(compare, actual, desired, err_msg=str(err_msg),
  File "C:\Users\Petron\AppData\Local\Programs\Python\Python310\lib\contextlib.py", line 79, in inner
    return func(*args, **kwds)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\numpy\testing\_private\utils.py", line 797, in assert_array_compare
    raise AssertionError(msg)
AssertionError: 
Not equal to tolerance rtol=1e-05, atol=0.0001

Misma


DEBUG RecurrentEncoderPT forward:
  Input x shape: torch.Size([2, 25, 1, 1])
  Input a shape: torch.Size([2, 25, 1, 1])
  use_gnn: False
  Final output: torch.Size([2, 8])
TF output shape: (2, 8)
PT output shape: (2, 8)
TF output sample: [ 0.35755023  2.0263896  -1.8466074   0.04479006  2.5075285 ]
PT output sample: [ 0.3575502   2.0263898  -1.8466076   0.04479012  2.507528  ]
Max absolute difference: 0.00000095
✅ `RecurrentEncoderPT` (non-GNN path) translation test PASSED!


<unittest.runner.TextTestResult run=2 errors=0 failures=1>

In [8]:
class VectorQuantizer(tf.keras.models.Model):
    """Vector quantizer layer.

    Quantizes the input vectors into a fixed number of clusters using L2 norm. Based on
    https://arxiv.org/pdf/1509.03700.pdf, and adapted for clustering using https://arxiv.org/abs/1806.02199.
    Implementation based on https://keras.io/examples/generative/vq_vae/.

    """

    def __init__(
        self, n_components, embedding_dim, beta, kmeans_loss: float = 0.0, **kwargs
    ):
        """Initialize the VQ layer.

        Args:
            n_components (int): number of embeddings to use
            embedding_dim (int): dimensionality of the embeddings
            beta (float): beta value for the loss function
            kmeans_loss (float): regularization parameter for the Gram matrix
            **kwargs: additional arguments for the parent class

        """
        super(VectorQuantizer, self).__init__(**kwargs)
        self.embedding_dim = embedding_dim
        self.n_components = n_components
        self.beta = beta
        self.kmeans = kmeans_loss

        # Initialize the VQ codebook
        w_init = tf.random_uniform_initializer()
        self.codebook = tf.Variable(
            initial_value=w_init(
                shape=(self.embedding_dim, self.n_components), dtype="float32"
            ),
            trainable=True,
            name="vqvae_codebook",
        )

    def call(self, x):  # pragma: no cover
        """Compute the VQ layer.

        Args:
            x (tf.Tensor): input tensor

        Returns:
                x (tf.Tensor): output tensor
        """
        # Compute input shape and flatten, keeping the embedding dimension intact
        input_shape = tf.shape(x)

        # Add a disentangling penalty to the embeddings
        if self.kmeans:
            kmeans_loss = deepof.model_utils.compute_kmeans_loss(
                x, weight=self.kmeans, batch_size=input_shape[0]
            )
            self.add_loss(kmeans_loss)
            self.add_metric(kmeans_loss, name="kmeans_loss")

        flattened = tf.reshape(x, [-1, self.embedding_dim])

        # Quantize input using the codebook
        encoding_indices = tf.cast(
            self.get_code_indices(flattened, return_soft_counts=False), tf.int32
        )
        soft_counts = self.get_code_indices(flattened, return_soft_counts=True)

        encodings = tf.one_hot(encoding_indices, self.n_components)

        quantized = tf.matmul(encodings, self.codebook, transpose_b=True)
        quantized = tf.reshape(quantized, input_shape)

        # Compute vector quantization loss, and add it to the layer
        commitment_loss = self.beta * tf.reduce_mean(
            (tf.stop_gradient(quantized) - x) ** 2
        )
        codebook_loss = tf.reduce_mean((quantized - tf.stop_gradient(x)) ** 2)
        self.add_loss(commitment_loss + codebook_loss)

        # Straight-through estimator (copy gradients through the undiferentiable layer)
        # This approach has been reported to have issues for clustering, so we use add an extra
        # reconstruction loss to ensure that the gradients can flow through the encoder.
        # quantized = x + tf.stop_gradient(quantized - x)

        return quantized, soft_counts

    # noinspection PyTypeChecker
    def get_code_indices(
        self, flattened_inputs, return_soft_counts=False
    ):  # pragma: no cover
        """Getter for the code indices at any given time.

        Args:
            flattened_inputs (tf.Tensor): flattened input tensor (encoder output)
            return_soft_counts (bool): whether to return soft counts based on the distance to the codes, instead of the code indices

        Returns:
            encoding_indices (tf.Tensor): code indices tensor with cluster assignments.
        """
        # Compute L2-norm distance between inputs and codes at a given time
        similarity = tf.matmul(flattened_inputs, self.codebook)
        distances = (
            tf.reduce_sum(flattened_inputs**2, axis=1, keepdims=True)
            + tf.reduce_sum(self.codebook**2, axis=0)
            - 2 * similarity
        )

        if return_soft_counts:
            # Compute soft counts based on the distance to the codes
            similarity = (1 / distances) ** 2
            soft_counts = similarity / tf.expand_dims(
                tf.reduce_sum(similarity, axis=1), axis=1
            )
            return soft_counts

        # Return index of the closest code
        encoding_indices = tf.argmin(distances, axis=1)
        return encoding_indices

In [9]:
class VectorQuantizerPT(nn.Module):
    """PyTorch Vector quantizer layer."""

    def __init__(
        self,
        n_components: int,
        embedding_dim: int,
        beta: float,
        kmeans_loss: float = 0.0,
    ):
        super(VectorQuantizerPT, self).__init__()
        self.embedding_dim = embedding_dim
        self.n_components = n_components
        self.beta = beta
        self.kmeans = kmeans_loss

        self.codebook = nn.Parameter(
            torch.empty(self.embedding_dim, self.n_components).uniform_(0, 1)
        )
        
        # Store individual loss components for inspection (but not for summing)
        self.last_commitment_loss = None
        self.last_codebook_loss = None
        self.last_kmeans_loss = None
        self.last_vq_loss = None

    def forward(self, x: torch.Tensor, return_losses: bool = True):
        """
        Args:
            x: Input tensor of shape (..., embedding_dim)
               Typically (batch_size, embedding_dim) from encoder
        """
        input_shape = x.shape
        
        losses = {}

        # Flatten to 2D
        flattened = x.reshape(-1, self.embedding_dim)

        # Kmeans loss on flattened 2D tensor
        if self.kmeans and return_losses:
            kmeans_loss_val = deepof.clustering.model_utils_new.compute_kmeans_loss_pt(flattened, self.kmeans)
            losses['kmeans_loss'] = kmeans_loss_val
            self.last_kmeans_loss = kmeans_loss_val.item()

        # Get encodings
        encoding_indices = self.get_code_indices(
            flattened, return_soft_counts=False
        ).long()
        soft_counts = self.get_code_indices(flattened, return_soft_counts=True)

        encodings = F.one_hot(encoding_indices, self.n_components).float()
        quantized = torch.matmul(encodings, self.codebook.T)
        quantized = quantized.reshape(input_shape)

        if return_losses:
            commitment_loss = self.beta * torch.mean(
                (quantized.detach() - x) ** 2
            )
            codebook_loss = torch.mean((quantized - x.detach()) ** 2)
            vq_loss = commitment_loss + codebook_loss
            
            # Store the COMBINED vq_loss in the losses dict (matching TF behavior)
            losses['vq_loss'] = vq_loss
            
            # Store individual components for inspection/logging only
            self.last_commitment_loss = commitment_loss.item()
            self.last_codebook_loss = codebook_loss.item()
            self.last_vq_loss = vq_loss.item()

        if return_losses:
            return quantized, soft_counts, losses
        else:
            return quantized, soft_counts

    def get_code_indices(
        self, flattened_inputs: torch.Tensor, return_soft_counts: bool = False
    ) -> torch.Tensor:
        similarity = torch.matmul(flattened_inputs, self.codebook)
        distances = (
            torch.sum(flattened_inputs**2, dim=1, keepdim=True)
            + torch.sum(self.codebook**2, dim=0)
            - 2 * similarity
        )

        if return_soft_counts:
            similarity = (1 / distances) ** 2
            soft_counts = similarity / torch.sum(similarity, dim=1, keepdim=True)
            return soft_counts

        encoding_indices = torch.argmin(distances, dim=1)
        return encoding_indices

In [10]:
def transfer_vq_weights(tf_model: VectorQuantizer, pt_model: VectorQuantizerPT):
    """Transfers weights from TensorFlow VectorQuantizer to PyTorch version.
    
    Args:
        tf_model: TensorFlow VectorQuantizer model
        pt_model: PyTorch VectorQuantizerPT model
    """
    # Transfer codebook weights
    # TF shape: (embedding_dim, n_components)
    # PT shape: (embedding_dim, n_components)
    # These match, so direct transfer
    codebook_weights = tf_model.codebook.numpy()
    pt_model.codebook.data = torch.from_numpy(codebook_weights)
    
    print(f"✅ Transferred codebook weights: {codebook_weights.shape}")

In [26]:
class TestVectorQuantizerTranslation(unittest.TestCase):
    def setUp(self):
        """Set up the models and transfer weights."""
        tf.keras.backend.clear_session()
        
        self.n_components = 16
        self.embedding_dim = 8
        self.beta = 0.25
        self.kmeans_weight = 0.1
        self.batch_size = 32  # More realistic batch size
        
        # Create TF model
        self.tf_model = VectorQuantizer(
            n_components=self.n_components,
            embedding_dim=self.embedding_dim,
            beta=self.beta,
            kmeans_loss=self.kmeans_weight,
        )
        
        # Create PT model
        self.pt_model = VectorQuantizerPT(
            n_components=self.n_components,
            embedding_dim=self.embedding_dim,
            beta=self.beta,
            kmeans_loss=self.kmeans_weight,
        )
        self.pt_model.eval()
        
        # Initialize TF model with a forward pass (2D input as in real model!)
        dummy_input = tf.random.normal((self.batch_size, self.embedding_dim))
        _ = self.tf_model(dummy_input)
        
        # Transfer weights
        transfer_vq_weights(self.tf_model, self.pt_model)
        
        # Create test data - 2D as encoder outputs!
        np.random.seed(42)
        self.np_input_2d = np.random.randn(
            self.batch_size, self.embedding_dim
        ).astype(np.float32)
        
        # Also create 3D data to test flexibility
        self.np_input_3d = np.random.randn(
            4, 10, self.embedding_dim
        ).astype(np.float32)

    def test_codebook_initialization(self):
        """Test that codebook weights are transferred correctly."""
        tf_codebook = self.tf_model.codebook.numpy()
        pt_codebook = self.pt_model.codebook.detach().cpu().numpy()
        
        np.testing.assert_allclose(tf_codebook, pt_codebook, rtol=1e-6, atol=1e-6)
        print("✅ Codebook initialization test PASSED!")

    def test_code_indices_2d(self):
        """Test code indices with 2D input (main use case)."""
        tf_input = tf.constant(self.np_input_2d)
        tf_flattened = tf.reshape(tf_input, [-1, self.embedding_dim])
        tf_indices = self.tf_model.get_code_indices(
            tf_flattened, return_soft_counts=False
        ).numpy()
        
        pt_input = torch.from_numpy(self.np_input_2d)
        pt_flattened = pt_input.reshape(-1, self.embedding_dim)
        with torch.no_grad():
            pt_indices = self.pt_model.get_code_indices(
                pt_flattened, return_soft_counts=False
            ).cpu().numpy()
        
        np.testing.assert_array_equal(tf_indices, pt_indices)
        print("✅ Code indices 2D test PASSED!")

    def test_soft_counts_2d(self):
        """Test soft counts with 2D input."""
        tf_input = tf.constant(self.np_input_2d)
        tf_flattened = tf.reshape(tf_input, [-1, self.embedding_dim])
        tf_soft = self.tf_model.get_code_indices(
            tf_flattened, return_soft_counts=True
        ).numpy()
        
        pt_input = torch.from_numpy(self.np_input_2d)
        pt_flattened = pt_input.reshape(-1, self.embedding_dim)
        with torch.no_grad():
            pt_soft = self.pt_model.get_code_indices(
                pt_flattened, return_soft_counts=True
            ).cpu().numpy()
        
        np.testing.assert_allclose(tf_soft, pt_soft, rtol=1e-5, atol=1e-6)
        print("✅ Soft counts 2D test PASSED!")

    def test_quantized_output_2d(self):
        """Test quantized output with 2D input."""
        tf_input = tf.constant(self.np_input_2d)
        tf_quantized, tf_soft = self.tf_model(tf_input, training=False)
        
        pt_input = torch.from_numpy(self.np_input_2d)
        with torch.no_grad():
            pt_quantized, pt_soft, _ = self.pt_model(pt_input, return_losses=True)
        
        np.testing.assert_allclose(
            tf_quantized.numpy(), 
            pt_quantized.cpu().numpy(), 
            rtol=1e-5, 
            atol=1e-6
        )
        np.testing.assert_allclose(
            tf_soft.numpy(), 
            pt_soft.cpu().numpy(), 
            rtol=1e-5, 
            atol=1e-6
        )
        print("✅ Quantized output 2D test PASSED!")

    def test_losses_2d(self):
        """Test that losses match between TF and PT with 2D input."""
        tf_input = tf.constant(self.np_input_2d)
        
        with tf.GradientTape():
            tf_quantized, _ = self.tf_model(tf_input, training=True)
            tf_losses = self.tf_model.losses
        
        tf_total_loss = sum(tf_losses).numpy()
        
        pt_input = torch.from_numpy(self.np_input_2d)
        pt_input.requires_grad = True
        
        pt_quantized, pt_soft, pt_losses_dict = self.pt_model(pt_input, return_losses=True)
        
        # Sum all PT losses (now only kmeans_loss and vq_loss)
        pt_total_loss = sum(pt_losses_dict.values()).item()
        
        print(f"\nTF Losses: {[loss.numpy() for loss in tf_losses]}")
        print(f"PT Losses: {[(k, v.item()) for k, v in pt_losses_dict.items()]}")
        print(f"\nTF Total Loss: {tf_total_loss:.6f}")
        print(f"PT Total Loss: {pt_total_loss:.6f}")
        
        # Also print individual components for debugging
        print(f"\nPT Individual Components (for reference):")
        print(f"  Commitment Loss: {self.pt_model.last_commitment_loss:.6f}")
        print(f"  Codebook Loss: {self.pt_model.last_codebook_loss:.6f}")
        
        # Allow slightly larger tolerance for loss comparison due to numerical differences
        np.testing.assert_allclose(tf_total_loss, pt_total_loss, rtol=1e-4, atol=1e-5)
        print("\n✅ Losses 2D test PASSED!")

    def test_full_forward_pass_timing(self):
        """Test full forward pass and compare timing."""
        print("\n" + "="*50)
        print("TIMING COMPARISON (2D Input)")
        print("="*50)
        
        # TF timing
        tf_input = tf.constant(self.np_input_2d)
        tf_start = time.time()
        for _ in range(100):
            tf_quantized, tf_soft = self.tf_model(tf_input, training=False)
        tf_end = time.time()
        tf_time = tf_end - tf_start
        
        # PT timing
        pt_input = torch.from_numpy(self.np_input_2d)
        with torch.no_grad():
            pt_start = time.time()
            for _ in range(100):
                pt_quantized, pt_soft, _ = self.pt_model(pt_input, return_losses=True)
            pt_end = time.time()
        pt_time = pt_end - pt_start
        
        print(f"TensorFlow execution time (100 iters): {tf_time:.4f}s")
        print(f"PyTorch execution time (100 iters): {pt_time:.4f}s")
        print(f"Speedup: {tf_time/pt_time:.5f}x")
        print("="*50)
        
        # Final output comparison
        tf_quantized_final, tf_soft_final = self.tf_model(tf_input, training=False)
        with torch.no_grad():
            pt_quantized_final, pt_soft_final, _ = self.pt_model(pt_input, return_losses=True)
        
        np.testing.assert_allclose(
            tf_quantized_final.numpy(), 
            pt_quantized_final.cpu().numpy(), 
            rtol=1e-5, 
            atol=1e-6
        )
        print("✅ Full forward pass timing test PASSED!")


# Run the tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestVectorQuantizerTranslation)
runner.run(suite)

test_code_indices_2d (__main__.TestVectorQuantizerTranslation)
Test code indices with 2D input (main use case). ... ok
test_codebook_initialization (__main__.TestVectorQuantizerTranslation)
Test that codebook weights are transferred correctly. ... ok
test_full_forward_pass_timing (__main__.TestVectorQuantizerTranslation)
Test full forward pass and compare timing. ... 

✅ Transferred codebook weights: (8, 16)
✅ Code indices 2D test PASSED!
✅ Transferred codebook weights: (8, 16)
✅ Codebook initialization test PASSED!
✅ Transferred codebook weights: (8, 16)

TIMING COMPARISON (2D Input)


ERROR
test_losses_2d (__main__.TestVectorQuantizerTranslation)
Test that losses match between TF and PT with 2D input. ... ERROR
test_quantized_output_2d (__main__.TestVectorQuantizerTranslation)
Test quantized output with 2D input. ... ERROR
test_soft_counts_2d (__main__.TestVectorQuantizerTranslation)
Test soft counts with 2D input. ... ok

ERROR: test_full_forward_pass_timing (__main__.TestVectorQuantizerTranslation)
Test full forward pass and compare timing.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_5600\3598182899.py", line 165, in test_full_forward_pass_timing
    pt_quantized, pt_soft, _ = self.pt_model(pt_input, return_losses=True)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\torch\nn\modules\module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "c:\Users\Petron\Desktop\Python_Projects\Deep

✅ Transferred codebook weights: (8, 16)
✅ Transferred codebook weights: (8, 16)
✅ Transferred codebook weights: (8, 16)
✅ Soft counts 2D test PASSED!


<unittest.runner.TextTestResult run=6 errors=3 failures=0>

# Full VQVAE

In [11]:
# Cell 1: PyTorch VQVAE Model (Corrected for actual input dimensions)

import deepof.clustering.models_new


class VQVAEPT(nn.Module):
    """
    PyTorch implementation of the VQ-VAE model adapted to the DeepOF setting.
    
    Note: This version handles the actual DeepOF input format where:
    - x: (B, T, node_features) - flattened node features
    - a: (B, T, edge_features) - flattened edge features
    """
    
    def __init__(
        self,
        input_shape: tuple,
        edge_feature_shape: tuple,
        adjacency_matrix: np.ndarray,
        latent_dim: int,
        n_components: int,
        beta: float = 1.0,
        kmeans_loss: float = 0.0,
        use_gnn: bool = True,
        encoder_type: str = "recurrent",
        interaction_regularization: float = 0.0,
    ):
        """Initialize a VQ-VAE model.

        Args:
            input_shape (tuple): Shape of the input (time_steps, node_features).
            edge_feature_shape (tuple): Shape of edge features (time_steps, edge_features).
            adjacency_matrix (np.ndarray): Adjacency matrix for GNN.
            latent_dim (int): Dimensionality of the latent space.
            n_components (int): Number of embeddings (clusters) in the codebook.
            beta (float): Beta parameter of the VQ loss.
            kmeans_loss (float): Regularization parameter for the Gram matrix.
            use_gnn (bool): Whether to use GNN in encoder.
            encoder_type (str): Type of encoder ("recurrent", "TCN", or "transformer").
            interaction_regularization (float): Regularization parameter for interactions.
        """
        super().__init__()
        
        time_steps, n_nodes, n_features_per_node = input_shape
        self.input_n_nodes = n_nodes
        self.input_n_features_per_node = n_features_per_node
        self.window_size = time_steps
        self.latent_dim = latent_dim
        self.n_components = n_components
        self.encoder_type = encoder_type
        
        # Initialize encoder based on type
        if encoder_type == "recurrent":
            self.encoder = deepof.clustering.models_new.RecurrentEncoderPT(
                input_shape=input_shape,
                edge_feature_shape=edge_feature_shape,
                adjacency_matrix=adjacency_matrix,
                latent_dim=latent_dim,
                use_gnn=use_gnn,
                interaction_regularization=interaction_regularization,
            )
            decoder_output_features = n_nodes * n_features_per_node
            self.decoder = deepof.clustering.models_new.RecurrentDecoderPT(
                output_shape=(time_steps, decoder_output_features),
                latent_dim=latent_dim,
            )
        elif encoder_type == "TCN":
            self.encoder = deepof.clustering.models_new.TCNEncoderPT(
                input_shape=input_shape,
                edge_feature_shape=edge_feature_shape,
                adjacency_matrix=adjacency_matrix,
                latent_dim=latent_dim,
                use_gnn=use_gnn,
                interaction_regularization=interaction_regularization,
            )
            decoder_output_features = n_nodes * n_features_per_node
            self.decoder = deepof.clustering.models_new.TCNDecoderPT(
                output_shape=(time_steps, decoder_output_features),
                latent_dim=latent_dim,
            ) 
        elif encoder_type == "transformer":
            self.encoder = deepof.clustering.models_new.TFMEncoderPT(
                input_shape=input_shape,
                edge_feature_shape=edge_feature_shape,
                adjacency_matrix=adjacency_matrix,
                latent_dim=latent_dim,
                use_gnn=use_gnn,
            )
            decoder_output_features = n_nodes * n_features_per_node
            self.decoder = deepof.clustering.models_new.TFMDecoderPT(
                output_shape=(time_steps, decoder_output_features),
                latent_dim=latent_dim,
                num_layers=2,
                num_heads=8,
                dff=128,
                dropout_rate=0.2,
                teacher_forcing_mode="zeros",
                input_dropout_p=0.5,
                self_attn_diag_only=False,
            )
        else:
            raise ValueError(f"Unknown encoder_type: {encoder_type}")
        
        # Initialize Vector Quantizer
        self.vq_layer = VectorQuantizerPT(
            n_components=n_components,
            embedding_dim=latent_dim,
            beta=beta,
            kmeans_loss=kmeans_loss,
        )

    def forward(
        self,
        x: torch.Tensor,
        a: torch.Tensor,
        return_losses: bool = True,
        return_all_outputs: bool = False,
    ):
        """
        Forward pass through the VQ-VAE model.
        
        Args:
            x: Input node features (B, T, N, F)
            a: Input edge features (B, T, E, F_edge)
            return_losses: Whether to compute and return VQ losses
            return_all_outputs: Whether to return all intermediate outputs
        """
        # Encode
        encoder_output = self.encoder(x, a)  # Shape: (B, latent_dim)
        
        # Vector Quantization
        if return_losses:
            quantized_latents, soft_counts, vq_losses = self.vq_layer(
                encoder_output, return_losses=True
            )
        else:
            quantized_latents, soft_counts = self.vq_layer(
                encoder_output, return_losses=False
            )
            vq_losses = {}
        
        # Prepare input for decoder (flatten spatial dimensions for teacher forcing)
        B, T, N, F = x.shape
        x_for_decoder = x.view(B, T, N * F)  # Flatten to (B, T, node_features)
        
        # Decode from QUANTIZED latents (main path)
        encoding_reconstruction_dist = self.decoder(quantized_latents, x_for_decoder)
        
        # Decode from ORIGINAL encoder output (bypass path for gradient flow)
        reconstruction_dist = self.decoder(encoder_output, x_for_decoder)
        
        # Handle transformer decoder outputs (which return attention weights)
        if self.encoder_type == "transformer":
            if isinstance(encoding_reconstruction_dist, tuple):
                encoding_reconstruction_dist = encoding_reconstruction_dist[0]
            if isinstance(reconstruction_dist, tuple):
                reconstruction_dist = reconstruction_dist[0]
        
        if return_all_outputs:
            return (
                encoding_reconstruction_dist,
                reconstruction_dist,
                quantized_latents,
                soft_counts,
                encoder_output,
                vq_losses if return_losses else None,
            )
        else:
            if return_losses:
                return encoding_reconstruction_dist, reconstruction_dist, vq_losses
            else:
                return encoding_reconstruction_dist, reconstruction_dist

    @torch.no_grad()
    def encode(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """Inference-only: Get encoder output. Equivalent to TF 'encoder' model."""
        return self.encoder(x, a)

    @torch.no_grad()
    def group(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """Inference-only: Get quantized latents. Equivalent to TF 'grouper' model."""
        encoder_output = self.encoder(x, a)
        quantized, _ = self.vq_layer(encoder_output, return_losses=False)
        return quantized

    @torch.no_grad()
    def soft_group(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """Inference-only: Get soft cluster assignments. Equivalent to TF 'soft_grouper' model."""
        encoder_output = self.encoder(x, a)
        _, soft_counts = self.vq_layer(encoder_output, return_losses=False)
        return soft_counts

    @torch.no_grad()
    def reconstruct(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """Full reconstruction from input through VQ-VAE."""
        encoding_recon_dist, _ = self.forward(x, a, return_losses=False)
        return encoding_recon_dist.mean
    
    def get_codebook_usage(self, data_loader, max_samples: int = 10000):
        """Compute codebook usage statistics over a dataset."""
        self.eval()
        all_indices = []
        all_soft_counts = []
        samples_seen = 0
        
        with torch.no_grad():
            for x, a, *_ in data_loader:
                x = x.to(next(self.parameters()).device)
                a = a.to(next(self.parameters()).device)
                
                encoder_output = self.encoder(x, a)
                flattened = encoder_output.reshape(-1, self.latent_dim)
                
                indices = self.vq_layer.get_code_indices(
                    flattened, return_soft_counts=False
                )
                soft_counts = self.vq_layer.get_code_indices(
                    flattened, return_soft_counts=True
                )
                
                all_indices.append(indices.cpu())
                all_soft_counts.append(soft_counts.cpu())
                
                samples_seen += x.size(0)
                if samples_seen >= max_samples:
                    break
        
        all_indices = torch.cat(all_indices)
        all_soft_counts = torch.cat(all_soft_counts)
        
        usage_counts = torch.bincount(all_indices, minlength=self.n_components)
        
        # Compute perplexity
        avg_probs = all_soft_counts.mean(dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))
        
        return usage_counts, perplexity.item()

In [13]:
def train_vqvae_epoch(
    model: VQVAEPT,
    train_loader,
    optimizer,
    device,
    epoch: int,
):
    """
    Training loop for one epoch of VQ-VAE.
    
    Returns:
        Dictionary of average losses for the epoch
    """
    model.train()
    
    total_loss_sum = 0.0
    encoding_recon_loss_sum = 0.0
    recon_loss_sum = 0.0
    vq_loss_sum = 0.0
    kmeans_loss_sum = 0.0
    n_batches = 0
    
    for batch_idx, (x, a, y) in enumerate(train_loader):
        x, a, y = x.to(device), a.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        encoding_recon_dist, recon_dist, vq_losses = model(
            x, a, return_losses=True, return_all_outputs=False
        )
        
        # Compute reconstruction losses (negative log-likelihood)
        encoding_recon_loss = -encoding_recon_dist.log_prob(y).mean()
        recon_loss = -recon_dist.log_prob(y).mean()
        
        # Get VQ losses
        vq_loss = vq_losses['vq_loss']
        kmeans_loss = vq_losses.get('kmeans_loss', torch.tensor(0.0).to(device))
        
        # Total loss (matching TF implementation)
        total_loss = encoding_recon_loss + recon_loss + vq_loss + kmeans_loss
        
        # Backward pass
        total_loss.backward()
        optimizer.step()
        
        # Accumulate losses
        total_loss_sum += total_loss.item()
        encoding_recon_loss_sum += encoding_recon_loss.item()
        recon_loss_sum += recon_loss.item()
        vq_loss_sum += vq_loss.item()
        kmeans_loss_sum += kmeans_loss.item()
        n_batches += 1
    
    # Compute cluster population
    with torch.no_grad():
        x_sample, a_sample, _ = next(iter(train_loader))
        x_sample, a_sample = x_sample.to(device), a_sample.to(device)
        soft_counts = model.soft_group(x_sample, a_sample)
        cluster_assignments = torch.argmax(soft_counts, dim=1)
        n_populated = len(torch.unique(cluster_assignments))
    
    return {
        'total_loss': total_loss_sum / n_batches,
        'encoding_reconstruction_loss': encoding_recon_loss_sum / n_batches,
        'reconstruction_loss': recon_loss_sum / n_batches,
        'vq_loss': vq_loss_sum / n_batches,
        'kmeans_loss': kmeans_loss_sum / n_batches,
        'n_populated_clusters': n_populated,
    }


@torch.no_grad()
def validate_vqvae_epoch(
    model: VQVAEPT,
    val_loader,
    device,
):
    """
    Validation loop for one epoch of VQ-VAE.
    
    Returns:
        Dictionary of average losses for the epoch
    """
    model.eval()
    
    total_loss_sum = 0.0
    encoding_recon_loss_sum = 0.0
    recon_loss_sum = 0.0
    vq_loss_sum = 0.0
    kmeans_loss_sum = 0.0
    n_batches = 0
    
    for x, a, y in val_loader:
        x, a, y = x.to(device), a.to(device), y.to(device)
        
        # Forward pass
        encoding_recon_dist, recon_dist, vq_losses = model(
            x, a, return_losses=True, return_all_outputs=False
        )
        
        # Compute reconstruction losses
        encoding_recon_loss = -encoding_recon_dist.log_prob(y).mean()
        recon_loss = -recon_dist.log_prob(y).mean()
        
        # Get VQ losses
        vq_loss = vq_losses['vq_loss']
        kmeans_loss = vq_losses.get('kmeans_loss', torch.tensor(0.0).to(device))
        
        # Total loss
        total_loss = encoding_recon_loss + recon_loss + vq_loss + kmeans_loss
        
        # Accumulate losses
        total_loss_sum += total_loss.item()
        encoding_recon_loss_sum += encoding_recon_loss.item()
        recon_loss_sum += recon_loss.item()
        vq_loss_sum += vq_loss.item()
        kmeans_loss_sum += kmeans_loss.item()
        n_batches += 1
    
    # Compute cluster population
    x_sample, a_sample, _ = next(iter(val_loader))
    x_sample, a_sample = x_sample.to(device), a_sample.to(device)
    soft_counts = model.soft_group(x_sample, a_sample)
    cluster_assignments = torch.argmax(soft_counts, dim=1)
    n_populated = len(torch.unique(cluster_assignments))
    
    return {
        'val_total_loss': total_loss_sum / n_batches,
        'val_encoding_reconstruction_loss': encoding_recon_loss_sum / n_batches,
        'val_reconstruction_loss': recon_loss_sum / n_batches,
        'val_vq_loss': vq_loss_sum / n_batches,
        'val_kmeans_loss': kmeans_loss_sum / n_batches,
        'val_n_populated_clusters': n_populated,
    }


def train_vqvae_full(
    model: VQVAEPT,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs: int,
    log_interval: int = 10,
):
    """
    Complete training loop for VQ-VAE.
    
    Args:
        model: The VQVAEPT model
        train_loader: Training data loader
        val_loader: Validation data loader
        optimizer: PyTorch optimizer
        device: Device to train on
        num_epochs: Number of epochs to train
        log_interval: How often to print progress
        
    Returns:
        Dictionary containing training history
    """
    history = {
        'train': [],
        'val': []
    }
    
    for epoch in range(num_epochs):
        # Train
        train_metrics = train_vqvae_epoch(model, train_loader, optimizer, device, epoch)
        history['train'].append(train_metrics)
        
        # Validate
        val_metrics = validate_vqvae_epoch(model, val_loader, device)
        history['val'].append(val_metrics)
        
        # Log progress
        if (epoch + 1) % log_interval == 0 or epoch == 0:
            print(f"\nEpoch {epoch + 1}/{num_epochs}")
            print(f"  Train - Total: {train_metrics['total_loss']:.4f}, "
                  f"Enc Recon: {train_metrics['encoding_reconstruction_loss']:.4f}, "
                  f"Recon: {train_metrics['reconstruction_loss']:.4f}, "
                  f"VQ: {train_metrics['vq_loss']:.4f}, "
                  f"KMeans: {train_metrics['kmeans_loss']:.4f}, "
                  f"Clusters: {train_metrics['n_populated_clusters']}")
            print(f"  Val   - Total: {val_metrics['val_total_loss']:.4f}, "
                  f"Enc Recon: {val_metrics['val_encoding_reconstruction_loss']:.4f}, "
                  f"Recon: {val_metrics['val_reconstruction_loss']:.4f}, "
                  f"VQ: {val_metrics['val_vq_loss']:.4f}, "
                  f"KMeans: {val_metrics['val_kmeans_loss']:.4f}, "
                  f"Clusters: {val_metrics['val_n_populated_clusters']}")
    
    return history

In [ ]:
def transfer_vqvae_weights(tf_vqvae, pt_vqvae):
    """
    Transfer weights from TensorFlow VQVAE to PyTorch VQVAEPT.
    
    Args:
        tf_vqvae: TensorFlow VQVAE instance
        pt_vqvae: PyTorch VQVAEPT instance
    """
    print("Transferring VQ-VAE weights...")
    
    # Transfer encoder weights (assumed already done via encoder-specific transfer functions)
    # transfer_encoder_weights(tf_vqvae.encoder, pt_vqvae.encoder)
    print("  ⚠️  Encoder weights - use encoder-specific transfer function")
    
    # Transfer VQ layer weights
    tf_vq_layer = tf_vqvae.vqvae.get_layer('vector_quantizer')
    transfer_vq_weights(tf_vq_layer, pt_vqvae.vq_layer)
    print("  ✅ VQ layer weights transferred")
    
    # Transfer decoder weights (assumed already done via decoder-specific transfer functions)
    # transfer_decoder_weights(tf_vqvae.decoder, pt_vqvae.decoder)
    print("  ⚠️  Decoder weights - use decoder-specific transfer function")
    
    print("✅ VQ-VAE weight transfer complete!")

In [52]:
def _flatten_keras_tensors(x):
    if isinstance(x, (list, tuple)):
        out = []
        for xi in x:
            out.extend(_flatten_keras_tensors(xi))
        return out
    return [x]

def _layer_from_tensor(t):
    kh = getattr(t, "_keras_history", None)
    return kh[0] if isinstance(kh, tuple) else getattr(kh, "layer", None)

def _pick_gnn_pre_embeddings(tf_enc, node_in_ch, edge_in_ch):
    # Find CensNetConv and its inbound tensors
    gnn_tf = next(l for l in tf_enc.layers if l.__class__.__name__ == "CensNetConv")
    node = gnn_tf._inbound_nodes[-1]  # use last call
    in_ts = getattr(node, "input_tensors", None)
    if in_ts is None:
        in_ts = getattr(node, "inputs")
    in_ts = _flatten_keras_tensors(in_ts)

    # Among inbound tensors, pick the two with rank 3 and last dim == 2*latent
    cand = [t for t in in_ts if len(t.shape) == 3]
    # Disambiguate by producer: nested Functional input feature dim
    pairs = []
    for t in cand:
        prod = _layer_from_tensor(t)   # nested Functional (recurrent block)
        if isinstance(prod, tf.keras.Model):
            in_last = int(prod.inputs[0].shape[-1])
            pairs.append((t, prod, in_last))

    # Map by input feature dim
    tf_node_pre_t = next(t for t, prod, in_last in pairs if in_last == node_in_ch)
    tf_edge_pre_t = next(t for t, prod, in_last in pairs if in_last == edge_in_ch)
    tf_node_block = _layer_from_tensor(tf_node_pre_t)
    tf_edge_block = _layer_from_tensor(tf_edge_pre_t)
    return gnn_tf, tf_node_pre_t, tf_edge_pre_t, tf_node_block, tf_edge_block

In [7]:
import unittest
import numpy as np
import tensorflow as tf
import torch
import torch.nn as nn
import time
import sys
from io import StringIO


class TestVQVAETranslation(unittest.TestCase):
    
    @classmethod
    def setUpClass(cls):
        """Set up once for all tests."""
        print("\n" + "="*70)
        print(" VQ-VAE TRANSLATION TEST SUITE")
        print("="*70)
    
    def setUp(self):
        """Set up models for testing."""
        tf.keras.backend.clear_session()
        
        # Test parameters
        self.batch_size = 64
        self.time_steps = 25
        self.num_nodes = 11
        self.num_edges = 11
        self.n_node_features = 3
        self.n_edge_features = 1
        self.latent_dim = 6
        self.n_components = 10
        self.beta = 0.25
        self.kmeans_loss = 0.1
        

        # TensorFlow expects (batch, time, total_features)
        self.input_shape_tf = (self.batch_size, self.time_steps, self.num_nodes * self.n_node_features)
        self.edge_feature_shape_tf = (self.batch_size, self.time_steps, self.num_edges * self.n_edge_features)
        
        # PyTorch expects (time, nodes, features_per_node) for input_shape parameter
        self.input_shape_pt = (self.time_steps, self.num_nodes, self.n_node_features)
        self.edge_feature_shape_pt = (self.time_steps, self.num_edges, self.n_edge_features)
        
        # Create dummy adjacency matrix
        m = np.zeros((self.num_nodes, self.num_nodes))
        ui = np.triu_indices(self.num_nodes)
        num_possible_edges = len(ui[0])
        c = np.random.choice(num_possible_edges, min(self.num_edges, num_possible_edges), replace=False)
        m[ui[0][c], ui[1][c]] = 1
        m += m.T # Make symmetric
        self.adjacency_matrix = m

        use_gnn=True
        
        # Create TensorFlow model
        self.tf_model = deepof.models.VQVAE(
            input_shape=self.input_shape_tf,
            edge_feature_shape=self.edge_feature_shape_tf,
            adjacency_matrix=self.adjacency_matrix,
            latent_dim=self.latent_dim,
            n_components=self.n_components,
            beta=self.beta,
            kmeans_loss=self.kmeans_loss,
            use_gnn=use_gnn,  # Simplified for testing
            encoder_type="recurrent",
        )
        
        # Create PyTorch model  
        self.pt_model = deepof.clustering.models_new.VQVAEPT(
            input_shape=self.input_shape_pt,  # Without batch dimension
            edge_feature_shape=self.edge_feature_shape_pt,
            adjacency_matrix=self.adjacency_matrix,
            latent_dim=self.latent_dim,
            n_components=self.n_components,
            beta=self.beta,
            kmeans_loss=self.kmeans_loss,
            use_gnn=use_gnn,
            encoder_type="recurrent",
        )
        self.pt_model.eval()
        
        # Create test data
        np.random.seed(42)
        
        # Generate TensorFlow format data (B, T, total_features)
        self.x_test_tf = np.random.randn(*self.input_shape_tf).astype(np.float32)
        self.a_test_tf = np.random.randn(*self.edge_feature_shape_tf).astype(np.float32)
        self.y_test_tf = np.random.randn(*self.input_shape_tf).astype(np.float32)
        
        # Reshape to PyTorch format (B, T, nodes, features_per_node)
        self.x_test_pt = self.x_test_tf.reshape(
            self.batch_size, self.time_steps, self.num_nodes, self.n_node_features
        )
        self.a_test_pt = self.a_test_tf.reshape(
            self.batch_size, self.time_steps, self.num_edges, self.n_edge_features
        )
        self.y_test_pt = self.y_test_tf.reshape(
            self.batch_size, self.time_steps, self.num_nodes, self.n_node_features
        )
        
        # Initialize TF model with forward pass
        _ = self.tf_model([
            tf.constant(self.x_test_tf),
            tf.constant(self.a_test_tf)
        ], training=False)
        
        # Suppress transfer weight output
        old_stdout = sys.stdout
        sys.stdout = StringIO()
        
        # Transfer VQ weights only (encoder/decoder handled separately)
        transfer_vq_weights(
            self.tf_model.vqvae.get_layer('vector_quantizer'),
            self.pt_model.vq_layer
        )
        
        sys.stdout = old_stdout

    def test_1_model_structure(self):
        """Test that model components exist and have correct structure."""
        # Check TF model
        self.assertIsNotNone(self.tf_model.encoder)
        self.assertIsNotNone(self.tf_model.decoder)
        self.assertIsNotNone(self.tf_model.grouper)
        self.assertIsNotNone(self.tf_model.soft_grouper)
        
        # Check PT model
        self.assertIsNotNone(self.pt_model.encoder)
        self.assertIsNotNone(self.pt_model.decoder)
        self.assertIsNotNone(self.pt_model.vq_layer)

    def test_2_encoder_output_shape(self):
        """Test encoder output shapes match."""
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_encoder_out = self.tf_model.encoder([tf_input, tf_a])
        
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        with torch.no_grad():
            pt_encoder_out = self.pt_model.encoder(pt_input, pt_a)
        
        self.assertEqual(
            tuple(tf_encoder_out.shape),
            tuple(pt_encoder_out.shape),
            "Encoder output shapes don't match"
        )

    def test_3_vq_layer_quantization(self):
        """Test VQ layer produces same quantized outputs."""
        # Create dummy encoder output
        encoder_out = np.random.randn(self.batch_size, self.latent_dim).astype(np.float32)
        
        # TF quantization
        tf_enc = tf.constant(encoder_out)
        tf_quantized, tf_soft = self.tf_model.vqvae.get_layer('vector_quantizer')(tf_enc)
        
        # PT quantization
        pt_enc = torch.from_numpy(encoder_out)
        with torch.no_grad():
            pt_quantized, pt_soft = self.pt_model.vq_layer(pt_enc, return_losses=False)
        
        # Compare quantized outputs
        np.testing.assert_allclose(
            tf_quantized.numpy(),
            pt_quantized.cpu().numpy(),
            rtol=1e-5,
            atol=1e-6
        )
        
        # Compare soft counts
        np.testing.assert_allclose(
            tf_soft.numpy(),
            pt_soft.cpu().numpy(),
            rtol=1e-5,
            atol=1e-6
        )

    def test_4_grouper_outputs(self):
        """Test grouper (quantized latents) outputs match."""
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_grouped = self.tf_model.grouper([tf_input, tf_a])
        
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        with torch.no_grad():
            pt_grouped = self.pt_model.group(pt_input, pt_a)
        
        self.assertEqual(
            tuple(tf_grouped.shape),
            tuple(pt_grouped.shape),
            "Grouper output shapes don't match"
        )

    def test_5_soft_grouper_outputs(self):
        """Test soft_grouper (soft counts) outputs match."""
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_soft = self.tf_model.soft_grouper([tf_input, tf_a])
        
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        with torch.no_grad():
            pt_soft = self.pt_model.soft_group(pt_input, pt_a)
        
        self.assertEqual(
            tuple(tf_soft.shape),
            tuple(pt_soft.shape),
            "Soft grouper output shapes don't match"
        )
        
        pt_soft_sums = pt_soft.sum(dim=1)
        np.testing.assert_allclose(
            pt_soft_sums.cpu().numpy(),
            np.ones(self.batch_size),
            rtol=1e-5,
            atol=1e-6,
            err_msg="Soft counts don't sum to 1"
        )

    def test_6_forward_pass_outputs(self):
        """Test full forward pass output shapes."""
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_output = self.tf_model.vqvae([tf_input, tf_a], training=False)
        
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        with torch.no_grad():
            pt_enc_recon, pt_recon = self.pt_model(pt_input, pt_a, return_losses=False)
        
        # TF output shape: (B, T, total_features) - flattened
        expected_shape_tf = (self.batch_size, self.time_steps, 
                            self.num_nodes * self.n_node_features)
        
        # PT output shape: (B, T, total_features) - also flattened (matches decoder output)
        expected_shape_pt = (self.batch_size, self.time_steps, 
                            self.num_nodes * self.n_node_features)
        
        self.assertEqual(
            tuple(tf_output.mean().shape),
            expected_shape_tf,
            "TF output shape doesn't match expected"
        )
        
        self.assertEqual(
            tuple(pt_enc_recon.mean.shape),
            expected_shape_pt,
            "PT encoding reconstruction shape doesn't match expected"
        )
        
        self.assertEqual(
            tuple(pt_recon.mean.shape),
            expected_shape_pt,
            "PT reconstruction shape doesn't match expected"
        )

    def test_7_loss_computation(self):
        """Test that losses can be computed properly."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        pt_y = torch.from_numpy(self.y_test_pt)
        
        # Flatten y to match decoder output format
        B, T, N, F = pt_y.shape
        pt_y_flat = pt_y.view(B, T, N * F)
        
        self.pt_model.train()
        
        enc_recon_dist, recon_dist, vq_losses = self.pt_model(
            pt_input, pt_a, return_losses=True
        )
        
        # Compute reconstruction losses with flattened ground truth
        enc_recon_loss = -enc_recon_dist.log_prob(pt_y_flat).mean()
        recon_loss = -recon_dist.log_prob(pt_y_flat).mean()
        
        self.assertIn('vq_loss', vq_losses)
        if self.kmeans_loss > 0:
            self.assertIn('kmeans_loss', vq_losses)
        
        self.assertTrue(torch.isfinite(enc_recon_loss).all())
        self.assertTrue(torch.isfinite(recon_loss).all())
        self.assertTrue(torch.isfinite(vq_losses['vq_loss']).all())

    def test_8_backward_pass(self):
        """Test that gradients flow properly through the model."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        pt_y = torch.from_numpy(self.y_test_pt)
        
        # Flatten y to match decoder output format
        B, T, N, F = pt_y.shape
        pt_y_flat = pt_y.view(B, T, N * F)
        
        pt_input.requires_grad = True
        
        self.pt_model.train()
        
        enc_recon_dist, recon_dist, vq_losses = self.pt_model(
            pt_input, pt_a, return_losses=True
        )
        
        # Compute total loss with flattened ground truth
        enc_recon_loss = -enc_recon_dist.log_prob(pt_y_flat).mean()
        recon_loss = -recon_dist.log_prob(pt_y_flat).mean()
        total_loss = enc_recon_loss + recon_loss + sum(vq_losses.values())
        
        total_loss.backward()
        
        self.assertIsNotNone(self.pt_model.vq_layer.codebook.grad)
        self.assertTrue(torch.isfinite(self.pt_model.vq_layer.codebook.grad).all())

    def test_90_cluster_population(self):
        """Test cluster population computation."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        
        with torch.no_grad():
            soft_counts = self.pt_model.soft_group(pt_input, pt_a)
            cluster_assignments = torch.argmax(soft_counts, dim=1)
            unique_clusters = torch.unique(cluster_assignments)
            n_populated = len(unique_clusters)
        
        self.assertGreater(n_populated, 0)
        self.assertLessEqual(n_populated, self.n_components)

    def test_91_inference_methods(self):
        """Test all inference methods work correctly."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        
        self.pt_model.eval()
        
        with torch.no_grad():
            # Test encode
            encoded = self.pt_model.encode(pt_input, pt_a)
            self.assertEqual(encoded.shape, (self.batch_size, self.latent_dim))
            
            # Test group
            quantized = self.pt_model.group(pt_input, pt_a)
            self.assertEqual(quantized.shape, (self.batch_size, self.latent_dim))
            
            # Test soft_group
            soft_counts = self.pt_model.soft_group(pt_input, pt_a)
            self.assertEqual(soft_counts.shape, (self.batch_size, self.n_components))
            
            # Test reconstruct - output is FLATTENED
            reconstruction = self.pt_model.reconstruct(pt_input, pt_a)
            expected_shape = (self.batch_size, self.time_steps, 
                            self.num_nodes * self.n_node_features)
            self.assertEqual(tuple(reconstruction.shape), expected_shape)

    def test_92_return_all_outputs(self):
        """Test return_all_outputs flag."""
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        
        with torch.no_grad():
            outputs = self.pt_model(
                pt_input, pt_a, 
                return_losses=True, 
                return_all_outputs=True
            )
        
        self.assertEqual(len(outputs), 6)
        
        (enc_recon_dist, recon_dist, quantized, soft_counts, 
        encoder_output, vq_losses) = outputs
        
        self.assertEqual(quantized.shape, (self.batch_size, self.latent_dim))
        self.assertEqual(soft_counts.shape, (self.batch_size, self.n_components))
        self.assertEqual(encoder_output.shape, (self.batch_size, self.latent_dim))
        self.assertIsInstance(vq_losses, dict)

    def test_93_timing_comparison(self):
        """Compare inference speed between TF and PT."""
        print("\n" + "="*70)
        print(" TIMING COMPARISON")
        print("="*70)
        
        n_iters = 50
        
        # TensorFlow timing
        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        
        tf_start = time.time()
        for _ in range(n_iters):
            _ = self.tf_model.vqvae([tf_input, tf_a], training=False)
        tf_time = time.time() - tf_start
        
        # PyTorch timing
        pt_input = torch.from_numpy(self.x_test_pt)
        pt_a = torch.from_numpy(self.a_test_pt)
        
        with torch.no_grad():
            pt_start = time.time()
            for _ in range(n_iters):
                _ = self.pt_model(pt_input, pt_a, return_losses=False)
            pt_time = time.time() - pt_start
        
        print(f"TensorFlow: {tf_time:.4f}s ({tf_time/n_iters*1000:.2f}ms per iter)")
        print(f"PyTorch:    {pt_time:.4f}s ({pt_time/n_iters*1000:.2f}ms per iter)")
        print(f"Speedup:    {tf_time/pt_time:.2f}x")
        print("="*70)
        
        self.assertGreater(tf_time, 0)
        self.assertGreater(pt_time, 0)


    def test_encoder_nested_blocks_direct_parity(self):
        print("\n" + "="*70)
        print(" TEST: Encoder nested blocks direct parity")
        print("="*70)
        assert self.pt_model.encoder.use_gnn

        # 1) Transfer weights (uses GNN inbound mapping)
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder)

        # 2) Locate the exact nested blocks feeding CensNetConv
        tf_enc = self.tf_model.encoder.get_layer("recurrent_encoder")
        gnn_tf = next(l for l in tf_enc.layers if l.__class__.__name__ == "CensNetConv")
        node = gnn_tf._inbound_nodes[0]
        in_tensors = getattr(node, "input_tensors", None)
        if in_tensors is None:
            in_tensors = getattr(node, "inputs")
        in_tensors = _flatten_keras_tensors(in_tensors)
        tf_node_pre_t = in_tensors[0]
        tf_edge_pre_t = in_tensors[-1]
        tf_node_block = _layer_from_tensor(tf_node_pre_t)
        tf_edge_block = _layer_from_tensor(tf_edge_pre_t)

        # 3) Build direct-call models for the nested blocks
        #    Note: calling the nested Model on a new Input is fine.
        B, T = self.batch_size, self.time_steps
        N, E = self.num_nodes, self.num_edges
        Fn, Fe = self.n_node_features, self.n_edge_features

        tf_node_inp = tf.keras.Input(shape=(N, T, Fn))
        tf_edge_inp = tf.keras.Input(shape=(E, T, Fe))
        tf_node_direct = tf.keras.Model(tf_node_inp, tf_node_block(tf_node_inp), name="tf_node_direct")
        tf_edge_direct = tf.keras.Model(tf_edge_inp, tf_edge_block(tf_edge_inp), name="tf_edge_direct")

        # 4) Prepare identical inputs (same as you used in the reshape test)
        tf_nodes_in_np = self.x_test_tf.reshape(B, T, N, Fn).transpose(0, 2, 1, 3)
        tf_edges_in_np = self.a_test_tf.reshape(B, T, E, Fe).transpose(0, 2, 1, 3)

        # 5) Forward both TF and PT blocks on the same inputs
        tf_node_pre = tf_node_direct(tf_nodes_in_np, training=False).numpy()
        tf_edge_pre = tf_edge_direct(tf_edges_in_np, training=False).numpy()

        self.pt_model.encoder.eval()
        with torch.no_grad():
            pt_nodes_in = torch.from_numpy(tf_nodes_in_np)
            pt_edges_in = torch.from_numpy(tf_edges_in_np)
            pt_node_pre = self.pt_model.encoder.node_recurrent_block(pt_nodes_in).cpu().numpy()
            pt_edge_pre = self.pt_model.encoder.edge_recurrent_block(pt_edges_in).cpu().numpy()

        # 6) Compare
        def rep(name, a, b, rtol=1e-5, atol=1e-6):
            d = np.abs(a - b)
            print(f"{name}: max {d.max():.8f} | mean {d.mean():.8f}")
            np.testing.assert_allclose(a, b, rtol=rtol, atol=atol, err_msg=f"{name} mismatch")

        rep("Node block output", tf_node_pre, pt_node_pre)
        rep("Edge block output", tf_edge_pre, pt_edge_pre)

        print("✅ Encoder nested blocks direct parity passed")


    def test_000gnn_reshape_and_intermediate_parity(self):
        """
        Compare, step by step, what TF vs PT feed into and get out of the GNN path:
        1) Inputs to the node/edge recurrent blocks (after TF reshapes)
        2) Outputs of the node/edge recurrent blocks (pre-GNN embeddings)
        3) Outputs of the GNN (nodes and edges)
        4) Flatten+concat tensor fed to final Dense
        """
        print("\n" + "="*70)
        print(" TEST: GNN RESHAPE + INTERMEDIATE PARITY")
        print("="*70)
       
        assert self.pt_model.encoder.use_gnn, "This test is for use_gnn=True"

        # 0) Transfer weights
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder)

        tf_enc = self.tf_model.encoder.get_layer("recurrent_encoder")
        node_in_ch = int(self.pt_model.encoder.node_recurrent_block.conv1d.in_channels)
        edge_in_ch = int(self.pt_model.encoder.edge_recurrent_block.conv1d.in_channels)

        gnn_tf, tf_node_pre_t, tf_edge_pre_t, tf_node_block, tf_edge_block = \
            _pick_gnn_pre_embeddings(tf_enc, node_in_ch, edge_in_ch)

        tf_tap = tf.keras.Model(
            inputs=tf_enc.inputs,
            outputs=[tf_node_pre_t, tf_edge_pre_t, gnn_tf.output]
        )

        tf_input = tf.constant(self.x_test_tf)
        tf_a = tf.constant(self.a_test_tf)
        tf_node_pre, tf_edge_pre, tf_gnn_out = tf_tap([tf_input, tf_a], training=False)
        tf_node_pre = tf_node_pre.numpy()
        tf_edge_pre = tf_edge_pre.numpy()
        tf_gnn_nodes = tf_gnn_out[0].numpy()
        tf_gnn_edges = tf_gnn_out[1].numpy()

        # Sanity: the inputs to the recurrent blocks match
        B, T = self.batch_size, self.time_steps
        N, E = self.num_nodes, self.num_edges
        Fn, Fe = self.n_node_features, self.n_edge_features
        tf_nodes_in_np = self.x_test_tf.reshape(B, T, N, Fn).transpose(0, 2, 1, 3)
        tf_edges_in_np = self.a_test_tf.reshape(B, T, E, Fe).transpose(0, 2, 1, 3)

        pt_enc = self.pt_model.encoder
        pt_enc.eval()
        with torch.no_grad():
            pt_nodes_in = torch.from_numpy(tf_nodes_in_np)
            pt_edges_in = torch.from_numpy(tf_edges_in_np)
            node_pre = pt_enc.node_recurrent_block(pt_nodes_in)  # (B, N, 2*ld)
            edge_pre = pt_enc.edge_recurrent_block(pt_edges_in)  # (B, E, 2*ld)
            adj_tuple = (pt_enc.laplacian, pt_enc.edge_laplacian, pt_enc.incidence)
            pt_gnn_nodes, pt_gnn_edges = pt_enc.spatial_gnn_block([node_pre, adj_tuple, edge_pre])

        pt_node_pre_np = node_pre.cpu().numpy()
        pt_edge_pre_np = edge_pre.cpu().numpy()
        pt_gnn_nodes_np = pt_gnn_nodes.cpu().numpy()
        pt_gnn_edges_np = pt_gnn_edges.cpu().numpy()

        def rep(name, a, b, rtol=1e-5, atol=1e-6):
            d = np.abs(a - b)
            print(f"{name}: shape {a.shape} | max {d.max():.8f} | mean {d.mean():.8f}")
            np.testing.assert_allclose(a, b, rtol=rtol, atol=atol, err_msg=f"{name} mismatch")

        print("\nComparing pre-GNN embeddings (recurrent outputs)")
        rep("Nodes pre-GNN", tf_node_pre, pt_node_pre_np)
        rep("Edges pre-GNN", tf_edge_pre, pt_edge_pre_np)

        print("\nComparing GNN outputs")
        rep("GNN nodes", tf_gnn_nodes, pt_gnn_nodes_np)
        rep("GNN edges", tf_gnn_edges, pt_gnn_edges_np)

        # Optional: concat check before final Dense
        tf_concat = np.concatenate(
            [tf_gnn_nodes.reshape(B, -1), tf_gnn_edges.reshape(B, -1)], axis=-1
        )
        pt_concat = np.concatenate(
            [pt_gnn_nodes_np.reshape(B, -1), pt_gnn_edges_np.reshape(B, -1)], axis=-1
        )
        rep("Concat to final Dense", tf_concat, pt_concat, rtol=1e-6, atol=1e-7)

        print("✅ GNN intermediate parity test passed")


    def test_94_encoder_exact_match(self):
        """Test that encoder outputs match exactly between TF and PT."""
        print("\n" + "="*70)
        print(" TEST 94: ENCODER EXACT OUTPUT MATCH")
        print("="*70)
        
        # Transfer encoder weights
        print("Transferring encoder weights...")
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder)
        
        # Use the same random input for both
        tf_input = tf.constant(self.x_test_tf+1e-9)
        tf_a = tf.constant(self.a_test_tf+1e-9)
        
        pt_input = torch.from_numpy(self.x_test_pt+1e-9)
        pt_a = torch.from_numpy(self.a_test_pt+1e-9)
        
        # Get encoder outputs
        tf_encoder_out = self.tf_model.encoder([tf_input, tf_a]).numpy()
        
        with torch.no_grad():
            pt_encoder_out = self.pt_model.encoder(pt_input, pt_a).cpu().numpy()
        
        print(f"TF encoder output shape: {tf_encoder_out.shape}")
        print(f"PT encoder output shape: {pt_encoder_out.shape}")
        print(f"TF output sample: {tf_encoder_out[0, :5]}")
        print(f"PT output sample: {pt_encoder_out[0, :5]}")
        print(f"Max absolute difference: {np.abs(tf_encoder_out - pt_encoder_out).max():.8f}")
        print(f"Mean absolute difference: {np.abs(tf_encoder_out - pt_encoder_out).mean():.8f}")
        
        # Assert close match
        np.testing.assert_allclose(
            tf_encoder_out,
            pt_encoder_out,
            rtol=1e-4,
            atol=1e-5,
            err_msg="Encoder outputs don't match!"
        )
        print("✅ Encoder outputs match!")
        print("="*70)

    def test_95_vq_layer_exact_match(self):
        """Test that VQ layer outputs match exactly between TF and PT."""
        print("\n" + "="*70)
        print(" TEST 95: VQ LAYER EXACT OUTPUT MATCH")
        print("="*70)
        
        # VQ weights already transferred in setUp
        # Create dummy encoder output (same for both)
        np.random.seed(123)
        encoder_out = np.random.randn(self.batch_size, self.latent_dim).astype(np.float32)
        
        # TF VQ layer
        tf_enc = tf.constant(encoder_out)
        tf_vq_layer = self.tf_model.vqvae.get_layer('vector_quantizer')
        tf_quantized, tf_soft = tf_vq_layer(tf_enc, training=False)
        
        # PT VQ layer
        pt_enc = torch.from_numpy(encoder_out)
        with torch.no_grad():
            pt_quantized, pt_soft = self.pt_model.vq_layer(pt_enc, return_losses=False)
        
        # Compare quantized outputs
        tf_quantized_np = tf_quantized.numpy()
        pt_quantized_np = pt_quantized.cpu().numpy()
        
        print(f"TF quantized shape: {tf_quantized_np.shape}")
        print(f"PT quantized shape: {pt_quantized_np.shape}")
        print(f"TF quantized sample: {tf_quantized_np[0, :5]}")
        print(f"PT quantized sample: {pt_quantized_np[0, :5]}")
        print(f"Quantized max diff: {np.abs(tf_quantized_np - pt_quantized_np).max():.8f}")
        print(f"Quantized mean diff: {np.abs(tf_quantized_np - pt_quantized_np).mean():.8f}")
        
        np.testing.assert_allclose(
            tf_quantized_np,
            pt_quantized_np,
            rtol=1e-5,
            atol=1e-6,
            err_msg="VQ quantized outputs don't match!"
        )
        
        # Compare soft counts
        tf_soft_np = tf_soft.numpy()
        pt_soft_np = pt_soft.cpu().numpy()
        
        print(f"TF soft counts shape: {tf_soft_np.shape}")
        print(f"PT soft counts shape: {pt_soft_np.shape}")
        print(f"Soft counts max diff: {np.abs(tf_soft_np - pt_soft_np).max():.8f}")
        print(f"Soft counts mean diff: {np.abs(tf_soft_np - pt_soft_np).mean():.8f}")
        
        np.testing.assert_allclose(
            tf_soft_np,
            pt_soft_np,
            rtol=1e-5,
            atol=1e-6,
            err_msg="VQ soft counts don't match!"
        )
        
        # Verify cluster assignments are identical
        tf_clusters = np.argmax(tf_soft_np, axis=1)
        pt_clusters = np.argmax(pt_soft_np, axis=1)
        match_rate = (tf_clusters == pt_clusters).mean()
        print(f"Cluster assignment match rate: {match_rate*100:.2f}%")
        
        self.assertEqual(match_rate, 1.0, "Cluster assignments should be identical!")
        print("✅ VQ layer outputs match perfectly!")
        print("="*70)

    def test_96_decoder_exact_match(self):
        """Test that decoder outputs match exactly between TF and PT."""
        print("\n" + "="*70)
        print(" TEST 96: DECODER EXACT OUTPUT MATCH")
        print("="*70)
        
        # Transfer decoder weights
        print("Transferring decoder weights...")
        transfer_recurrent_decoder_weights(self.tf_model.decoder, self.pt_model.decoder)
        
        # Create dummy latent code (same for both)
        np.random.seed(456)
        latent = np.random.randn(self.batch_size, self.latent_dim).astype(np.float32)
        
        # TF decoder expects flattened input for teacher forcing
        tf_latent = tf.constant(latent)
        tf_x_input = tf.constant(self.x_test_tf)
        tf_decoder_out = self.tf_model.decoder([tf_latent, tf_x_input], training=False)
        
        # PT decoder also expects flattened input
        pt_latent = torch.from_numpy(latent)
        pt_x_input = torch.from_numpy(self.x_test_pt)
        B, T, N, F = pt_x_input.shape
        pt_x_flat = pt_x_input.view(B, T, N * F)
        
        with torch.no_grad():
            pt_decoder_out = self.pt_model.decoder(pt_latent, pt_x_flat)
        
        # Handle transformer decoder which returns tuple
        if self.pt_model.encoder_type == "transformer":
            if isinstance(tf_decoder_out, tuple):
                tf_decoder_out = tf_decoder_out[0]
            if isinstance(pt_decoder_out, tuple):
                pt_decoder_out = pt_decoder_out[0]
        
        # Compare distribution means
        tf_mean = tf_decoder_out.mean().numpy()
        pt_mean = pt_decoder_out.mean.cpu().numpy()
        
        print(f"TF decoder mean shape: {tf_mean.shape}")
        print(f"PT decoder mean shape: {pt_mean.shape}")
        print(f"TF mean sample: {tf_mean[0, 0, :5]}")
        print(f"PT mean sample: {pt_mean[0, 0, :5]}")
        print(f"Decoder mean max diff: {np.abs(tf_mean - pt_mean).max():.8f}")
        print(f"Decoder mean avg diff: {np.abs(tf_mean - pt_mean).mean():.8f}")
        
        np.testing.assert_allclose(
            tf_mean,
            pt_mean,
            rtol=1e-4,
            atol=1e-5,
            err_msg="Decoder means don't match!"
        )
        
        # Compare distribution scales/stddevs
        # Note: PyTorch distributions use .scale instead of .stddev
        tf_stddev = tf_decoder_out.stddev().numpy()
        samples = pt_decoder_out.sample((10000,))
        stddev_estimate = samples.std(dim=0).cpu().numpy()  #as pt_scale = pt_decoder_out.stddev().cpu().numpy() is not implemented

        print(f"TF stddev sample: {tf_stddev[0, 0, :5]}")
        print(f"PT scale sample: {stddev_estimate[0, 0, :5]}")
        print(f"Decoder scale sum diff: {np.abs(tf_stddev.sum() - stddev_estimate.sum()):.8f}")
        
        np.testing.assert_allclose(
            tf_stddev,
            stddev_estimate,
            rtol=0.1,  #high tolerance, as only sampled
            atol=0.1,
            err_msg="Decoder scales don't match!"
        )
        print("✅ Decoder outputs match!")
        print("="*70)

    def test_97_full_model_exact_match(self):
        """Test that full VQ-VAE outputs match exactly between TF and PT."""
        print("\n" + "="*70)
        print(" TEST 97: FULL MODEL EXACT OUTPUT MATCH")
        print("="*70)
        
        # Transfer all weights
        print("Transferring all model weights...")
        transfer_recurrent_encoder_weights(self.tf_model.encoder, self.pt_model.encoder)
        transfer_vq_weights(
            self.tf_model.vqvae.get_layer('vector_quantizer'),
            self.pt_model.vq_layer
        )
        transfer_recurrent_decoder_weights(self.tf_model.decoder, self.pt_model.decoder)
        print("All weights transferred.")
        
        # Use same random seed for both
        np.random.seed(789)
        x_test = np.random.randn(*self.input_shape_tf).astype(np.float32)
        a_test = np.random.randn(*self.edge_feature_shape_tf).astype(np.float32)
        
        # TF forward pass
        tf_input = tf.constant(x_test)
        tf_a = tf.constant(a_test)
        tf_output = self.tf_model.vqvae([tf_input, tf_a], training=False)
        
        # PT forward pass
        x_test_pt = x_test.reshape(
            self.batch_size, self.time_steps, self.num_nodes, self.n_node_features
        )
        a_test_pt = a_test.reshape(
            self.batch_size, self.time_steps, self.num_edges, self.n_edge_features
        )
        pt_input = torch.from_numpy(x_test_pt)
        pt_a = torch.from_numpy(a_test_pt)
        
        with torch.no_grad():
            pt_enc_recon, pt_recon = self.pt_model(pt_input, pt_a, return_losses=False)
        
        # Handle transformer outputs
        if self.pt_model.encoder_type == "transformer":
            if isinstance(tf_output, tuple):
                tf_output = tf_output[0]
            if isinstance(pt_enc_recon, tuple):
                pt_enc_recon = pt_enc_recon[0]
        
        # Compare encoding reconstruction (main output)
        tf_mean = tf_output.mean().numpy()
        pt_mean = pt_enc_recon.mean.cpu().numpy()
        
        print(f"TF output shape: {tf_mean.shape}")
        print(f"PT output shape: {pt_mean.shape}")
        print(f"TF output sample: {tf_mean[0, 0, :5]}")
        print(f"PT output sample: {pt_mean[0, 0, :5]}")
        print(f"Output mean max diff: {np.abs(tf_mean - pt_mean).max():.8f}")
        print(f"Output mean avg diff: {np.abs(tf_mean - pt_mean).mean():.8f}")
        
        relative_diff = np.abs(tf_mean - pt_mean) / (np.abs(tf_mean) + 1e-8)
        print(f"Output relative diff (mean): {relative_diff.mean():.8f}")
        print(f"Output relative diff (max): {relative_diff.max():.8f}")
        
        # Test encoding reconstruction
        np.testing.assert_allclose(
            tf_mean,
            pt_mean,
            rtol=1e-4,
            atol=1e-4,
            err_msg="Full model encoding reconstruction outputs don't match!"
        )
        
        # Also compare scales
        tf_stddev = tf_output.stddev().numpy()
        samples = pt_enc_recon.sample((10000,))
        pt_stddev_estimate = samples.std(dim=0).cpu().numpy()
        
        print(f"Scale sum diff: {np.abs(tf_stddev.sum() - pt_stddev_estimate.sum()):.8f}")
        
        np.testing.assert_allclose(
            tf_stddev,
            pt_stddev_estimate,
            rtol=0.1, #high tolerance as only sample
            atol=0.1,
            err_msg="Full model scales don't match!"
        )
        
        # Test bypass reconstruction (from original encoder output)
        if self.pt_model.encoder_type == "transformer" and isinstance(pt_recon, tuple):
            pt_recon = pt_recon[0]
        
        pt_recon_mean = pt_recon.mean.cpu().numpy()
        print(f"\nBypass reconstruction shape: {pt_recon_mean.shape}")
        print(f"Bypass differs from main by: {np.abs(tf_mean - pt_recon_mean).mean():.8f}")
        
        # Test that cluster assignments are identical
        print("\nTesting cluster assignments...")
        tf_soft = self.tf_model.soft_grouper([tf_input, tf_a]).numpy()
        with torch.no_grad():
            pt_soft = self.pt_model.soft_group(pt_input, pt_a).cpu().numpy()
        
        # Compare soft probabilities
        print(f"Soft probability max diff: {np.abs(tf_soft - pt_soft).max():.8f}")
        np.testing.assert_allclose(
            tf_soft,
            pt_soft,
            rtol=1e-4,
            atol=1e-5,
            err_msg="Soft cluster probabilities don't match!"
        )
        
        # Compare hard assignments
        tf_clusters = np.argmax(tf_soft, axis=1)
        pt_clusters = np.argmax(pt_soft, axis=1)
        
        cluster_match_rate = (tf_clusters == pt_clusters).mean()
        print(f"Cluster assignment match rate: {cluster_match_rate*100:.2f}%")
        
        self.assertGreaterEqual(
            cluster_match_rate,
            0.99,
            f"Cluster assignments should match at least 99%, got {cluster_match_rate*100:.2f}%"
        )
        
        # Print cluster distribution
        tf_unique, tf_counts = np.unique(tf_clusters, return_counts=True)
        pt_unique, pt_counts = np.unique(pt_clusters, return_counts=True)
        print(f"\nTF cluster distribution: {dict(zip(tf_unique, tf_counts))}")
        print(f"PT cluster distribution: {dict(zip(pt_unique, pt_counts))}")
        
        # Verify both models populate similar number of clusters
        n_tf_clusters = len(tf_unique)
        n_pt_clusters = len(pt_unique)
        print(f"TF populated clusters: {n_tf_clusters}/{self.n_components}")
        print(f"PT populated clusters: {n_pt_clusters}/{self.n_components}")
        
        self.assertEqual(
            n_tf_clusters,
            n_pt_clusters,
            "Number of populated clusters should match!"
        )
        
        print("✅ Full model outputs match!")
        print("="*70)

    @classmethod
    def tearDownClass(cls):
        """Print summary after all tests."""
        print("\n" + "="*70)
        print(" TEST SUMMARY")
        print("="*70)
        print(" ✅ All VQ-VAE translation tests completed!")
        print("="*70 + "\n")


# Run the tests
if __name__ == '__main__':
    runner = unittest.TextTestRunner(verbosity=2)
    suite = unittest.TestLoader().loadTestsFromTestCase(TestVQVAETranslation)
    result = runner.run(suite)
    
    if result.wasSuccessful():
        print(f"\n{'='*70}")
        print(f" ✅ SUCCESS: All {result.testsRun} tests passed!")
        print(f"{'='*70}")

test_000gnn_reshape_and_intermediate_parity (__main__.TestVQVAETranslation)
Compare, step by step, what TF vs PT feed into and get out of the GNN path: ... ERROR
test_1_model_structure (__main__.TestVQVAETranslation)
Test that model components exist and have correct structure. ... ok
test_2_encoder_output_shape (__main__.TestVQVAETranslation)
Test encoder output shapes match. ... ok
test_3_vq_layer_quantization (__main__.TestVQVAETranslation)
Test VQ layer produces same quantized outputs. ... ok
test_4_grouper_outputs (__main__.TestVQVAETranslation)
Test grouper (quantized latents) outputs match. ... ok
test_5_soft_grouper_outputs (__main__.TestVQVAETranslation)
Test soft_grouper (soft counts) outputs match. ... ok
test_6_forward_pass_outputs (__main__.TestVQVAETranslation)
Test full forward pass output shapes. ... ok
test_7_loss_computation (__main__.TestVQVAETranslation)
Test that losses can be computed properly. ... ok
test_8_backward_pass (__main__.TestVQVAETranslation)
Test that g

# Recurrent Block test

In [ ]:
def get_recurrent_block(
    x: tf.Tensor, latent_dim: int, gru_unroll: bool, bidirectional_merge: str
):
    """Build a recurrent embedding block, using a 1D convolution followed by two bidirectional GRU layers.

    Args:
        x (tf.Tensor): Input tensor.
        latent_dim (int): Number of dimensions of the output tensor.
        gru_unroll (bool): whether to unroll the GRU layers. Defaults to False.
        bidirectional_merge (str): how to merge the forward and backward GRU layers. Defaults to "concat".

    Returns:
        tf.keras.models.Model object with the specified architecture.

    """
    encoder = TimeDistributed(
        tf.keras.layers.Conv1D(
            filters=2 * latent_dim,
            kernel_size=5,
            strides=1,  # Increased strides yield shorter sequences
            padding="same",
            activation="relu",
            kernel_initializer=he_uniform(),
            use_bias=False,
        )
    )(x)
    encoder = tf.keras.layers.Masking(mask_value=0.0)(encoder)
    encoder = TimeDistributed(
        Bidirectional(
            GRU(
                2 * latent_dim,
                activation="tanh",
                recurrent_activation="sigmoid",
                return_sequences=True,
                unroll=gru_unroll,
                use_bias=True,
            ),
            merge_mode=bidirectional_merge,
        )
    )(encoder)
    encoder = LayerNormalization()(encoder)
    encoder = TimeDistributed(
        Bidirectional(
            GRU(
                latent_dim,
                activation="tanh",
                recurrent_activation="sigmoid",
                return_sequences=False,
                unroll=gru_unroll,
                use_bias=True,
            ),
            merge_mode=bidirectional_merge,
        )
    )(encoder)
    encoder = LayerNormalization()(encoder)
    

    return tf.keras.models.Model(x, encoder)

In [ ]:
def transfer_recurrent_block_weights(tf_model, pt_model):
    """Transfers weights for the full recurrent block with GRU gate permutation."""
    conv_td, _, gru1_td, norm1, gru2_td, norm2 = tf_model.layers[1:]


    def permute_gru_weights(keras_weights):
        W_ih, W_hh, B = keras_weights
        W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
        W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)
        W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
        W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)
        B_ih, B_hh = B
        B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
        B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)
        B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
        B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])
        return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt

    pt_model.conv1d.weight.data = torch.from_numpy(conv_td.layer.get_weights()[0]).permute(2, 1, 0)
    
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(gru1_td.layer.forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1); pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(gru1_td.layer.backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1); pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)

    pt_model.norm1.weight.data = torch.from_numpy(norm1.get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm1.get_weights()[1])

    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(gru2_td.layer.forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2); pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(gru2_td.layer.backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2); pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    
    pt_model.norm2.weight.data = torch.from_numpy(norm2.get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm2.get_weights()[1])

In [ ]:
class TestRecurrentBlockTranslation(unittest.TestCase):
    def setUp(self):
        """Set up the full models and transfer weights."""
        tf.keras.backend.clear_session()
        self.latent_dim = 8
        self.input_shape = (10, 6, 3) # (T, N, F)
        
        self.tf_model = get_recurrent_block(
            tf.keras.Input(shape=self.input_shape), self.latent_dim, False, "concat"
        )
        self.pt_model = deepof.clustering.model_utils_new.RecurrentBlockPT(self.input_shape[-1], self.latent_dim)
        self.pt_model.eval()

        transfer_recurrent_block_weights(self.tf_model, self.pt_model)
        
        # Create test data WITH MASKING
        self.np_input = np.random.rand(4, *self.input_shape).astype(np.float32)
        # Mask the last two "nodes" for the first sample in the batch
        self.np_input[0, :, :, :] = 0.0

    def test_final_forward_pass_with_masking(self):
        """Test the full block with the pack_padded_sequence masking method."""
        tf_start=time.time()
        tf_output = self.tf_model(self.np_input, training=False)
        tf_end=time.time()
        tf_output_np = tf_output.numpy()
        

        pt_input_tensor = torch.from_numpy(self.np_input)
        with torch.no_grad():
            pt_start = time.time()
            pt_output = self.pt_model(pt_input_tensor)
            pt_end=time.time()
        pt_output_np = pt_output.cpu().numpy()
        print("Tensorflow execution time: " + str(tf_end-tf_start))
        print("Pytorch execution time: " + str(pt_end-pt_start))

        np.testing.assert_allclose(tf_output_np, pt_output_np, rtol=1e-5, atol=1e-5)
        print("✅ Full `RecurrentBlockPT` translation test PASSED!")
        
#To run in Jupyter:
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestRecurrentBlockTranslation)
runner.run(suite)

test_final_forward_pass_with_masking (__main__.TestRecurrentBlockTranslation)
Test the full block with the pack_padded_sequence masking method. ... ok

----------------------------------------------------------------------
Ran 1 test in 1.370s

OK


Tensorflow execution time: 0.050722599029541016
Pytorch execution time: 0.0352330207824707
✅ Full `RecurrentBlockPT` translation test PASSED!


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

# ProbabilisticDecoder Test

In [ ]:
import unittest
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow_probability import layers as tfpl
from tensorflow_probability import distributions as tfd
from tensorflow_probability.python.bijectors import scale as tfb
import torch
import torch.nn as nn
from torch.distributions import Distribution, TransformedDistribution
from torch.distributions.transforms import AffineTransform
import time

In [ ]:
class ProbabilisticDecoder(tf.keras.layers.Layer):
    """Map the reconstruction output of a given decoder to a multivariate normal distribution."""

    def __init__(self, output_data_shape, **kwargs):
        """Initialize the probabilistic decoder."""
        super().__init__(**kwargs)
        self.time_distributer = tf.keras.layers.Dense(
            tfpl.IndependentNormal.params_size(output_data_shape) // 2
        )
        self.probabilistic_decoding = tfpl.DistributionLambda(
            make_distribution_fn=lambda decoded: tfd.Masked(
                tfd.Independent(
                    tfd.Normal(
                        loc=decoded[0], scale=tf.ones_like(decoded[0]),
                        validate_args=False, allow_nan_stats=False,
                    ),
                    reinterpreted_batch_ndims=1,
                ),
                validity_mask=decoded[1],
            ),
            convert_to_tensor_fn="mean",
        )
        self.scaled_probabilistic_decoding = tfpl.DistributionLambda(
            make_distribution_fn=lambda decoded: tfd.Masked(
                tfd.TransformedDistribution(
                    decoded[0], # base distribution
                    tfb.Scale(tf.cast(tf.expand_dims(decoded[1], axis=2), tf.float32)), # bijector
                    name="vae_reconstruction",
                ),
                validity_mask=decoded[1],
            ),
            convert_to_tensor_fn="mean",
        )

    def call(self, inputs):
        hidden, validity_mask = inputs
        loc_params = tf.keras.layers.TimeDistributed(self.time_distributer)(hidden)
        prob_decoded = self.probabilistic_decoding([loc_params, validity_mask])
        scaled_prob_decoded = self.scaled_probabilistic_decoding(
            [prob_decoded, validity_mask]
        )
        return scaled_prob_decoded

In [ ]:
# FIX: Create a subclass that knows how to compute the mean for an Affine transform.
class AffineTransformedDistribution(TransformedDistribution):
    """
    A specific TransformedDistribution for Affine transforms that implements .mean.
    """
    def __init__(self, base_distribution, transform):
        super().__init__(base_distribution, transform)

    @property
    def mean(self):
        """
        Computes the mean of the transformed distribution.
        E[loc + scale * X] = loc + scale * E[X]
        """
        # The transform itself is callable and applies the affine transformation.
        return self.transforms[0](self.base_dist.mean)

class ProbabilisticDecoderPT(nn.Module):
    """
    PyTorch translation of the ProbabilisticDecoder, including scaling transform.
    """
    def __init__(self, hidden_dim: int, data_dim: int):
        super().__init__()
        self.loc_projection = nn.Linear(in_features=hidden_dim, out_features=data_dim)

    def forward(self, hidden: torch.Tensor, validity_mask: torch.Tensor) -> AffineTransformedDistribution:
        B, T, D = hidden.shape
        # Reconstruct mean locations
        loc_params = self.loc_projection(hidden.view(B * T, -1)).reshape(B, T, -1)

        # Define Gaussian distributions with means (init: var=1)
        scale_params = torch.ones_like(loc_params)
        base_dist = torch.distributions.Normal(loc=loc_params, scale=scale_params)

        # Multivariate Gaussian distributions for feature vector
        independent_dist = torch.distributions.Independent(base_dist, 1)
        
        # Define transform to map masked values to 0 (y = 0 + 0 * x) and unmasked-values to themselves (y = 0 + 1.0 * x)
        scale_transform = validity_mask.unsqueeze(-1).to(hidden.dtype)
        transform = AffineTransform(loc=0, scale=scale_transform)
        
        # Returns a custom class instead of the generic one as "mean" functionality otherwise would be missing.
        final_dist = AffineTransformedDistribution(independent_dist, transform)
        return final_dist

In [ ]:
def transfer_probabilistic_decoder_weights(tf_model, pt_model):
    dense_layer = tf_model.time_distributer
    W, b = dense_layer.get_weights()
    pt_model.loc_projection.weight.data = torch.from_numpy(W.T)
    pt_model.loc_projection.bias.data = torch.from_numpy(b)

In [ ]:
class TestProbabilisticDecoderFinal(unittest.TestCase):
    def setUp(self):
        tf.keras.backend.clear_session()
        self.batch_size, self.time_steps, self.hidden_dim, self.data_dim = 4, 10, 32, 5

        # Create TF model
        self.tf_model = ProbabilisticDecoder(output_data_shape=(self.data_dim,))
        
        # Create PyTorch model
        self.pt_model = ProbabilisticDecoderPT(hidden_dim=self.hidden_dim, data_dim=self.data_dim)
        self.pt_model.eval()

        # --- THE FIX: Zero out the input tensor based on the mask ---
        np_hidden_original = np.random.rand(self.batch_size, self.time_steps, self.hidden_dim).astype(np.float32)
        
        # Create a float mask (1.0/0.0)
        self.np_float_mask = np.ones((self.batch_size, self.time_steps), dtype=np.float32)
        self.np_float_mask[0, -1] = 0.0 # Mask last step of first item
        self.np_float_mask[1, 5:] = 0.0 # Mask multiple steps of second item

        # Apply the mask to the input data itself before feeding it to the models
        self.np_hidden_masked = np_hidden_original * self.np_float_mask[:, :, np.newaxis]

        # TF needs a boolean mask for tfd.Masked
        self.np_bool_mask = self.np_float_mask.astype(bool)

        # Build the TF model by calling it once with the masked input
        self.tf_model([tf.constant(self.np_hidden_masked), tf.constant(self.np_bool_mask)])
        
        # Transfer weights
        transfer_probabilistic_decoder_weights(self.tf_model, self.pt_model)
        print("✅ Weights transferred successfully for final test.")

    def test_final_forward_pass(self):
        """Tests that the .mean() of the final transformed distributions are identical."""
        # --- TensorFlow ---
        # Pass the zeroed-out hidden data and the boolean mask
        tf_start=time.time()
        tf_dist = self.tf_model([self.np_hidden_masked, self.np_bool_mask])
        tf_mean_np = tf_dist.mean().numpy()
        tf_end=time.time()


        # --- PyTorch ---
        pt_hidden_tensor = torch.from_numpy(self.np_hidden_masked)
        pt_mask_tensor = torch.from_numpy(self.np_float_mask)
        with torch.no_grad():
            pt_start=time.time()
            pt_dist = self.pt_model(pt_hidden_tensor, pt_mask_tensor)
        pt_mean_np = pt_dist.mean.cpu().numpy()
        pt_end=time.time()

        # --- Verification ---
        np.testing.assert_allclose(tf_mean_np, pt_mean_np, rtol=1e-6, atol=1e-6)
        
        # Check a masked-out part is zero
        self.assertTrue(np.all(pt_mean_np[0, -1, :] == 0.0))
        # Check an un-masked part is not zero
        self.assertFalse(np.all(pt_mean_np[0, 0, :] == 0.0))

        print("Tensorflow execution time: " + str(tf_end-tf_start))
        print("Pytorch execution time: " + str(pt_end-pt_start))
        
        print("\n✅ `ProbabilisticDecoderPT` FINAL translation test PASSED!")

# To run in Jupyter or a script:
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestProbabilisticDecoderFinal)
runner.run(suite)

test_final_forward_pass (__main__.TestProbabilisticDecoderFinal)
Tests that the .mean() of the final transformed distributions are identical. ... ERROR

ERROR: test_final_forward_pass (__main__.TestProbabilisticDecoderFinal)
Tests that the .mean() of the final transformed distributions are identical.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_40932\1502940511.py", line 7, in setUp
    self.tf_model = ProbabilisticDecoder(output_data_shape=(self.data_dim,))
NameError: name 'ProbabilisticDecoder' is not defined. Did you mean: 'ProbabilisticDecoderPT'?

----------------------------------------------------------------------
Ran 1 test in 0.003s

FAILED (errors=1)


<unittest.runner.TextTestResult run=1 errors=1 failures=0>

# Recurrent Decoder Test

In [ ]:
import unittest
import numpy as np
import tcn
import tensorflow as tf
import tensorflow_probability as tfp
from sklearn.mixture import GaussianMixture
from spektral.layers import CensNetConv
from tensorflow.keras import Input, Model
from tensorflow.keras.initializers import he_uniform
from tensorflow.keras.layers import (
    GRU,
    Bidirectional,
    Dense,
    LayerNormalization,
    RepeatVector,
    TimeDistributed,
)
from tensorflow.keras.optimizers import Nadam
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import time
import deepof.model_utils
import deepof.clustering.model_utils_new
from deepof.clustering.censNetConv_pt import CensNetConvPT
import deepof.utils
from deepof.data_loading import get_dt
import warnings
from deepof.clustering.model_utils_new import ProbabilisticDecoderPT
from torch.distributions import Distribution, TransformedDistribution
from torch.distributions.transforms import AffineTransform

In [ ]:
def get_recurrent_decoder(
    input_shape: tuple,
    latent_dim: int,
    gru_unroll: bool = False,
    bidirectional_merge: str = "concat",
):
    """Return a recurrent neural decoder.

    Builds a deep neural network capable of decoding the structured latent space generated by one of the compatible
    classes into a sequence of motion tracking instances, either reconstructing the original
    input, or generating new data from given clusters.

    Args:
        input_shape (tuple): shape of the input data
        latent_dim (int): dimensionality of the latent space
        gru_unroll (bool): whether to unroll the GRU layers. Defaults to False.
        bidirectional_merge (str): how to merge the forward and backward GRU layers. Defaults to "concat".

    Returns:
        keras.Model: a keras model that can be trained to decode the latent space into a series of motion tracking
        sequences.

    """
    # Define and instantiate generator
    g = Input(shape=latent_dim)  # Decoder input, shaped as the latent space
    x = Input(shape=input_shape)  # Encoder input, used to generate an output mask
    validity_mask = tf.math.logical_not(tf.reduce_all(x == 0.0, axis=2))

    generator = RepeatVector(input_shape[0])(g)
    generator = Bidirectional(
        GRU(
            latent_dim,
            activation="tanh",
            recurrent_activation="sigmoid",
            return_sequences=True,
            unroll=gru_unroll,
            use_bias=True,
        ),
        merge_mode=bidirectional_merge,
    )(generator, mask=validity_mask)
    generator = LayerNormalization()(generator)
    generator = Bidirectional(
        GRU(
            2 * latent_dim,
            activation="tanh",
            recurrent_activation="sigmoid",
            return_sequences=True,
            unroll=gru_unroll,
            use_bias=True,
        ),
        merge_mode=bidirectional_merge,
    )(generator)
    generator = LayerNormalization()(generator)
    generator = tf.keras.layers.Conv1D(
        filters=2 * latent_dim,
        kernel_size=5,
        strides=1,
        padding="same",
        activation="relu",
        kernel_initializer=he_uniform(),
        use_bias=False,
    )(generator)
    generator = LayerNormalization()(generator)

    x_decoded = deepof.model_utils.ProbabilisticDecoder(input_shape)(
        [generator, validity_mask]
    )

    return Model([g, x], x_decoded, name="recurrent_decoder")


In [ ]:
class RecurrentDecoderPT(nn.Module):
    """
    A full PyTorch implementation of the recurrent decoder.
    """
    def __init__(self, output_shape: tuple, latent_dim: int, bidirectional_merge: str = "concat"):
        super().__init__()
        self.latent_dim = latent_dim
        self.output_shape = output_shape
        if bidirectional_merge != "concat":
            warnings.warn("Bidirectional merge mode is fixed to 'concat' to correspond with original TensorFlow implementation.")

        # First Bi-GRU layer
        self.gru1 = nn.GRU(
            input_size=latent_dim,
            hidden_size=latent_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.norm1 = nn.LayerNorm(2 * latent_dim, eps=1e-3)

        # Second Bi-GRU layer
        self.gru2 = nn.GRU(
            input_size=2 * latent_dim, # Input from first Bi-GRU
            hidden_size=2 * latent_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.norm2 = nn.LayerNorm(4 * latent_dim, eps=1e-3) # Output of second Bi-GRU is 2 * (2*latent_dim)

        # Convolutional Layer
        self.conv1d = nn.Conv1d(
            in_channels=4 * latent_dim, # Input from second norm layer
            out_channels=2 * latent_dim,
            kernel_size=5,
            padding="same",
            bias=False
        )
        self.norm3 = nn.LayerNorm(2 * latent_dim, eps=1e-3) # Output of Conv1D

        # Probabilistic Layer 
        self.prob_decoder = ProbabilisticDecoderPT(
            hidden_dim=2 * latent_dim, # Input from third norm layer
            data_dim=output_shape[1]
        )

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> TransformedDistribution:
        B, T, _ = x.shape

        # 1. Create the validity mask and sequence lengths from input 'x'
        validity_mask = ~torch.all(x == 0.0, dim=2)
        lengths = validity_mask.sum(dim=1).cpu().to(torch.int64)
        valid_indices = torch.where(lengths > 0)[0]

        # 2. Emulate RepeatVector
        generator = g.unsqueeze(1).expand(-1, T, -1)

        # 3. First Bi-GRU with masking
        gru1_out_full = torch.zeros(B, T, 2 * self.latent_dim, device=g.device, dtype=g.dtype)
        if len(valid_indices) > 0:
            # Apply GRU whilst ignoring masked data
            packed_input_1 = pack_padded_sequence(
                generator[valid_indices], lengths[valid_indices], batch_first=True, enforce_sorted=False
            )
            packed_output_1, _ = self.gru1(packed_input_1)
            unpacked_output_1, _ = pad_packed_sequence(
                packed_output_1, batch_first=True, total_length=T
            )
            gru1_out_full[valid_indices] = unpacked_output_1
        norm1_out = self.norm1(gru1_out_full)

        # 4. Second Bi-GRU with masking (reusing the same mask/lengths)
        gru2_out_full = torch.zeros(B, T, 4 * self.latent_dim, device=g.device, dtype=g.dtype)
        if len(valid_indices) > 0:
            packed_input_2 = pack_padded_sequence(
                norm1_out[valid_indices], lengths[valid_indices], batch_first=True, enforce_sorted=False
            )
            packed_output_2, _ = self.gru2(packed_input_2)
            unpacked_output_2, _ = pad_packed_sequence(
                packed_output_2, batch_first=True, total_length=T
            )
            gru2_out_full[valid_indices] = unpacked_output_2
        norm2_out = self.norm2(gru2_out_full)

        # 5. Convolution Block
        # Conv1d expects (B, C, T), so we permute
        conv_in = norm2_out.permute(0, 2, 1)
        conv_out = F.relu(self.conv1d(conv_in))
        # Permute back to (B, T, C) for LayerNorm
        norm3_in = conv_out.permute(0, 2, 1)
        norm3_out = self.norm3(norm3_in)

        # 6. Final Probabilistic Decoder
        final_dist = self.prob_decoder(norm3_out, validity_mask)

        return final_dist

In [ ]:
# Helper function from the provided example to handle gate order differences
def permute_gru_weights(keras_weights):
    """Permutes GRU weights from Keras (z, r, n) to PyTorch (r, z, n) format."""
    W_ih, W_hh, B = keras_weights
    # Keras gate order: z, r, n (update, reset, new/candidate)
    W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
    W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)

    # PyTorch gate order: r, z, n (reset, update, new/candidate)
    W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
    W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)

    # Keras has two bias vectors (input-hidden and recurrent), which are concatenated in B
    B_ih, B_hh = B
    B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
    B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)

    B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
    B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])

    return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt
    
def transfer_recurrent_decoder_weights(tf_model, pt_model):
    """
    Transfers weights for the full recurrent decoder model.
    """
    # Find layers by type to avoid index issues
    bidi_layers = [l for l in tf_model.layers if isinstance(l, Bidirectional)]
    norm_layers = [l for l in tf_model.layers if isinstance(l, LayerNormalization)]
    conv_layers = [l for l in tf_model.layers if isinstance(l, tf.keras.layers.Conv1D)]
    prob_dec_layer = next(l for l in tf_model.layers if isinstance(l, deepof.model_utils.ProbabilisticDecoder))

    # --- GRU 1 and Norm 1 ---
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(bidi_layers[0].forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1)
    pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(bidi_layers[0].backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1)
    pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)
    pt_model.norm1.weight.data = torch.from_numpy(norm_layers[0].get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm_layers[0].get_weights()[1])

    # --- GRU 2 and Norm 2 ---
    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(bidi_layers[1].forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2)
    pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(bidi_layers[1].backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2)
    pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    pt_model.norm2.weight.data = torch.from_numpy(norm_layers[1].get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm_layers[1].get_weights()[1])

    # --- Conv1D and Norm 3 ---
    # TF Conv1D weights: (kernel_w, kernel_h, in_c, out_c) -> (5, 1, 4*ld, 2*ld)
    # PT Conv1d weights: (out_c, in_c, kernel_w)
    conv_weights_tf = conv_layers[0].get_weights()[0]
    pt_model.conv1d.weight.data = torch.from_numpy(conv_weights_tf).squeeze(1).permute(2, 1, 0)
    pt_model.norm3.weight.data = torch.from_numpy(norm_layers[2].get_weights()[0]); pt_model.norm3.bias.data = torch.from_numpy(norm_layers[2].get_weights()[1])

    # --- Probabilistic Decoder ---
    # TF Dense weights: (in_features, out_features)
    # PT Linear weights: (out_features, in_features)
    prob_dec_weights, prob_dec_bias = prob_dec_layer.time_distributer.get_weights()
    pt_model.prob_decoder.loc_projection.weight.data = torch.from_numpy(prob_dec_weights.T)
    pt_model.prob_decoder.loc_projection.bias.data = torch.from_numpy(prob_dec_bias)

In [ ]:
import deepof.clustering.models_new


class TestRecurrentDecoderTranslation(unittest.TestCase):
    def setUp(self):
        """Set up the full models and transfer weights."""
        tf.keras.backend.clear_session()
        # Make epsilon consistent between TF and PT LayerNorm
        tf.keras.backend.set_epsilon(1e-3)

        self.latent_dim = 16
        self.input_shape = (15, 8)  # (T, Features)
        self.batch_size = 4

        # Instantiate the original full TensorFlow model
        self.tf_model = deepof.models.get_recurrent_decoder(
            input_shape=self.input_shape,
            latent_dim=self.latent_dim,
            bidirectional_merge="concat"
        )

        # Instantiate the full PyTorch model
        self.pt_model = deepof.clustering.models_new.RecurrentDecoderPT(
            output_shape=self.input_shape,
            latent_dim=self.latent_dim
        )
        self.pt_model.eval()

        # Transfer all weights
        transfer_recurrent_decoder_weights(self.tf_model, self.pt_model)

        # Create test data WITH MASKING
        self.np_latent_input = np.random.rand(self.batch_size, self.latent_dim).astype(np.float32)
        self.np_sequence_input = np.random.rand(self.batch_size, *self.input_shape).astype(np.float32)
        # Mask some steps for sample 0
        self.np_sequence_input[0, -3:, :] = 0.0
        # Mask all steps for sample 1
        self.np_sequence_input[1, :, :] = 0.0

    def test_full_forward_pass_with_masking(self):
        """Test the full decoder translation against the original TF model."""
        # TensorFlow execution
        tf_start = time.time()
        tf_output_dist = self.tf_model([self.np_latent_input, self.np_sequence_input], training=False)
        # CORRECTED LINE: Call .mean() on the distribution object first
        tf_output_np = tf_output_dist.mean().numpy()
        tf_end = time.time()


        # PyTorch execution
        pt_latent_tensor = torch.from_numpy(self.np_latent_input)
        pt_sequence_tensor = torch.from_numpy(self.np_sequence_input)
        with torch.no_grad():
            pt_start = time.time()
            pt_dist = self.pt_model(pt_latent_tensor, pt_sequence_tensor)
            # Use the .mean property to get the tensor output
            pt_output = pt_dist.mean
        pt_output_np = pt_output.cpu().numpy()
        pt_end = time.time()

        print("Tensorflow execution time: " + str(tf_end-tf_start))
        print("Pytorch execution time: " + str(pt_end-pt_start))

        # Compare the final tensor outputs
        np.testing.assert_allclose(tf_output_np, pt_output_np, rtol=1e-5, atol=1e-4)
        print("✅ Full `RecurrentDecoderPT` translation test PASSED!")

# To run in a Python script or Jupyter notebook:
if __name__ == '__main__':
    # Add deepof and other necessary imports from the original problem description
    # Then run the test suite
    runner = unittest.TextTestRunner(verbosity=2)
    suite = unittest.TestLoader().loadTestsFromTestCase(TestRecurrentDecoderTranslation)
    runner.run(suite)

test_full_forward_pass_with_masking (__main__.TestRecurrentDecoderTranslation)
Test the full decoder translation against the original TF model. ... ok

----------------------------------------------------------------------
Ran 1 test in 1.550s

OK


Tensorflow execution time: 0.10011792182922363
Pytorch execution time: 0.016722679138183594
✅ Full `RecurrentDecoderPT` translation test PASSED!


# Recurrent Encoder Test

In [ ]:
import unittest
import numpy as np
import tcn
import tensorflow as tf
import tensorflow_probability as tfp
from sklearn.mixture import GaussianMixture
from spektral.layers import CensNetConv
from tensorflow.keras import Input, Model
from tensorflow.keras.initializers import he_uniform
from tensorflow.keras.layers import (
    GRU,
    Bidirectional,
    Dense,
    LayerNormalization,
    RepeatVector,
    TimeDistributed,
)
from tensorflow.keras.optimizers import Nadam
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import time
import deepof.model_utils
import deepof.clustering.model_utils_new
from deepof.clustering.censNetConv_pt import CensNetConvPT
import deepof.utils
from deepof.data_loading import get_dt
import warnings
from deepof.clustering.model_utils_new import ProbabilisticDecoderPT, RecurrentBlockPT
from torch.distributions import Distribution, TransformedDistribution
from torch.distributions.transforms import AffineTransform

In [ ]:
def get_recurrent_encoder(
    input_shape: tuple,
    edge_feature_shape: tuple,
    adjacency_matrix: np.ndarray,
    latent_dim: int,
    use_gnn: bool = True,
    gru_unroll: bool = False,
    bidirectional_merge: str = "concat",
    interaction_regularization: float = 0.0,
):
    """Return a deep recurrent neural encoder.

     Builds a neural network capable of encoding the motion tracking instances into a vector ready to be fed to
    one of the provided structured latent spaces.

    Args:
        input_shape (tuple): shape of the node features for the input data. Should be time x nodes x features.
        edge_feature_shape (tuple): shape of the adjacency matrix to use in the graph attention layers. Should be time x edges x features.
        adjacency_matrix (np.ndarray): adjacency matrix for the mice connectivity graph. Shape should be nodes x nodes.
        latent_dim (int): dimension of the latent space.
        use_gnn (bool): If True, the encoder uses a graph representation of the input, with coordinates and speeds as node attributes, and distances as edge attributes. If False, a regular 3D tensor is used as input.
        gru_unroll (bool): whether to unroll the GRU layers. Defaults to False.
        bidirectional_merge (str): how to merge the forward and backward GRU layers. Defaults to "concat".
        interaction_regularization (float): Regularization parameter for the interaction features.

    Returns:
        keras.Model: a keras model that can be trained to encode motion tracking instances into a vector.

    """
    # Define feature and adjacency inputs
    x = Input(shape=input_shape)
    a = Input(shape=edge_feature_shape)

    if use_gnn:
        x_reshaped = tf.transpose(
            tf.reshape(
                tf.transpose(x),
                [
                    -1,
                    adjacency_matrix.shape[-1],
                    x.shape[1],
                    input_shape[-1] // adjacency_matrix.shape[-1],
                ][::-1],
            )
        )
        a_reshaped = tf.transpose(
            tf.reshape(
                tf.transpose(a),
                [
                    -1,
                    edge_feature_shape[-1],
                    a.shape[1],
                    1,
                ][::-1],
            )
        )


    else:
        x_flat = tf.reshape(x, [-1, input_shape[0], input_shape[1] * input_shape[2]])
        x_reshaped = tf.expand_dims(x_flat, axis=1)

    # Instantiate temporal RNN block
    encoder = deepof.clustering.model_utils_new.get_recurrent_block(
        x_reshaped, latent_dim, gru_unroll, bidirectional_merge
    )(x_reshaped)


    # Instantiate spatial graph block
    if use_gnn:

        # Embed edge features too
        a_encoder = deepof.clustering.model_utils_new.get_recurrent_block(
            a_reshaped, latent_dim, gru_unroll, bidirectional_merge
        )(a_reshaped)
    
        spatial_block = CensNetConv(
            node_channels=latent_dim,
            edge_channels=latent_dim,
            activation="relu",
            node_regularizer=tf.keras.regularizers.l1(interaction_regularization),
        )

        # Process adjacency matrix
        laplacian, edge_laplacian, incidence = spatial_block.preprocess(
            adjacency_matrix
        )

        # Get and concatenate node and edge embeddings
        x_nodes, x_edges = spatial_block(
            [encoder, (laplacian, edge_laplacian, incidence), a_encoder], mask=None
        )
        

        x_nodes = tf.reshape(
            x_nodes,
            [-1, adjacency_matrix.shape[-1] * latent_dim],
        )

        x_edges = tf.reshape(
            x_edges,
            [-1, edge_feature_shape[-1] * latent_dim],
        )

        encoder = tf.concat([x_nodes, x_edges], axis=-1)

    else:
        encoder = tf.squeeze(encoder, axis=1)

    encoder_output = tf.keras.layers.Dense(latent_dim, kernel_initializer="he_uniform")(
        encoder
    )
    
    return Model([x, a], encoder_output, name="recurrent_encoder")

In [ ]:
class RecurrentEncoderPT(nn.Module):
    def __init__(
        self,
        input_shape: tuple,
        edge_feature_shape: tuple,
        adjacency_matrix: np.ndarray,
        latent_dim: int,
        use_gnn: bool = True,
        interaction_regularization: float = 0.0,
    ):
        super().__init__()
        self.use_gnn = use_gnn
        self.num_nodes = adjacency_matrix.shape[0]
        self.latent_dim = latent_dim

        if self.use_gnn:
            # Node path initialization 
            node_feat_per_animal = input_shape[-1] // self.num_nodes
            self.node_recurrent_block = deepof.clustering.model_utils_new.RecurrentBlockPT(
                input_features=node_feat_per_animal, latent_dim=latent_dim
            )

            # Edge path initialization 
            self.edge_recurrent_block = deepof.clustering.model_utils_new.RecurrentBlockPT(
                input_features=1, latent_dim=latent_dim
            )

            self.spatial_gnn_block = CensNetConvPT(
                node_channels=latent_dim,
                edge_channels=latent_dim,
            )
            lap, edge_lap, inc = self.spatial_gnn_block.preprocess(torch.tensor(adjacency_matrix))
            self.register_buffer("laplacian", lap.float())
            self.register_buffer("edge_laplacian", edge_lap.float())
            self.register_buffer("incidence", inc.float())
            
            self.num_edges = edge_feature_shape[1]
            final_dense_in = (self.num_nodes * latent_dim) + (self.num_edges * latent_dim)
            self.final_dense = nn.Linear(final_dense_in, latent_dim)

        else: # Non-GNN path 
            in_features = input_shape[1] * input_shape[2]
            self.recurrent_block = deepof.clustering.model_utils_new.RecurrentBlockPT(
                input_features=in_features, latent_dim=latent_dim
            )
            self.final_dense = nn.Linear(latent_dim, latent_dim)

    def forward(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        B, T, N_nodes_total, F_nodes_total = x.shape
        _, _, E_edges_total, F_edges_total = a.shape

        if self.use_gnn:
            # --- Attempt to replicate the exact TensorFlow reshape logic ---
            
            # 1. Node Path
            F_per_node = F_nodes_total // self.num_nodes
            x_t = x.permute(3, 2, 1, 0)
            target_shape_x = (F_per_node, T, self.num_nodes, -1)
            x_reshaped_t = x_t.reshape(target_shape_x)
            x_reshaped = x_reshaped_t.permute(3, 2, 1, 0)
            
            # 2. Edge Path
            a_t = a.permute(3, 2, 1, 0)
            target_shape_a = (1, T, F_edges_total, -1)
            a_reshaped_t = a_t.reshape(target_shape_a)
            a_reshaped = a_reshaped_t.permute(3, 2, 1, 0)

            # 3. Pass through Recurrent Blocks
            node_output = self.node_recurrent_block(x_reshaped)           
            edge_output = self.edge_recurrent_block(a_reshaped)
            
            # 4. GNN and Final Layers
            adj_tuple = (self.laplacian, self.edge_laplacian, self.incidence)
            x_nodes, x_edges = self.spatial_gnn_block(
                [node_output, adj_tuple, edge_output]
            )
            x_nodes=F.relu(x_nodes)
            x_edges=F.relu(x_edges)
            
            b_prime = x_nodes.shape[0]
            x_nodes_flat = x_nodes.view(b_prime, -1)
            x_edges_flat = x_edges.view(b_prime, -1)
            encoder = torch.cat([x_nodes_flat, x_edges_flat], dim=-1)
            

        else: # Non-GNN path 
            x_reshaped = x.view(B, T, N_nodes_total * F_nodes_total).unsqueeze(1)
            encoder = self.recurrent_block(x_reshaped).squeeze(1)

        return self.final_dense(encoder)

In [ ]:
def transfer_recurrent_block_weights(tf_model, pt_model):
    """Transfers weights for the full recurrent block with GRU gate permutation."""
    conv_td, _, gru1_td, norm1, gru2_td, norm2 = tf_model.layers[1:]


    def permute_gru_weights(keras_weights):
        W_ih, W_hh, B = keras_weights
        W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
        W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)
        W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
        W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)
        B_ih, B_hh = B
        B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
        B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)
        B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
        B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])
        return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt

    pt_model.conv1d.weight.data = torch.from_numpy(conv_td.layer.get_weights()[0]).permute(2, 1, 0)
    
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(gru1_td.layer.forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1); pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(gru1_td.layer.backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1); pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)

    pt_model.norm1.weight.data = torch.from_numpy(norm1.get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm1.get_weights()[1])

    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(gru2_td.layer.forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2); pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(gru2_td.layer.backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2); pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    
    pt_model.norm2.weight.data = torch.from_numpy(norm2.get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm2.get_weights()[1])

    
def transfer_censnet_weights(tf_layer, pt_layer):
    """
    Transfers all six weights from a Spektral CensNetConv layer to the
    corresponding CensNetConvPT layer.
    """
    # Get all weights from the TensorFlow layer
    tf_weights = tf_layer.get_weights()
    
    # Unpack all six weights
    # Order: kernel_node, bias_node, kernel_edge, bias_edge, projector_node, projector_edge
    kn_tf, bn_tf, ke_tf, be_tf, pn_tf, pe_tf = tf_weights
    
    # 1. Transfer Node Kernel
    pt_layer.node_kernel.data = torch.from_numpy(kn_tf)
    
    # 2. Transfer Node Bias
    pt_layer.node_bias.data = torch.from_numpy(bn_tf)
    
    # 3. Transfer Edge Kernel
    pt_layer.edge_kernel.data = torch.from_numpy(ke_tf)
    
    # 4. Transfer Edge Bias
    pt_layer.edge_bias.data = torch.from_numpy(be_tf)
    
    # 5. Transfer Node Projector Weights - ensure shape [features, 1]
    if pn_tf.ndim == 1:
        pn_tf = pn_tf.reshape(-1, 1)
    pt_layer.node_weights.data = torch.from_numpy(pn_tf)
    
    # 6. Transfer Edge Projector Weights - ensure shape [features, 1]
    if pe_tf.ndim == 1:
        pe_tf = pe_tf.reshape(-1, 1)
    pt_layer.edge_weights.data = torch.from_numpy(pe_tf)
    

def transfer_recurrent_encoder_weights(tf_model, pt_model):
    """
    Transfers weights for the full recurrent encoder, finding layers
    by their default names and types to avoid modifying original code.
    """
    # The final dense layer is consistently the last one in the model's layer list.
    final_dense_tf = tf_model.layers[-1]
    final_dense_pt = pt_model.final_dense
    w, b = final_dense_tf.get_weights()
    final_dense_pt.weight.data = torch.from_numpy(w.T)
    final_dense_pt.bias.data = torch.from_numpy(b)

    if pt_model.use_gnn:
        # Keras automatically names nested models 'model', 'model_1', etc., by order of creation.
        # Node recurrent block is created first.
        node_recurrent_model = tf_model.get_layer("model")
        # Edge recurrent block is created second.
        edge_recurrent_model = tf_model.get_layer("model_1")
        # Find the CensNetConv layer by its class type.
        gnn_layer = next(l for l in tf_model.layers if isinstance(l, CensNetConv))

        transfer_recurrent_block_weights(node_recurrent_model, pt_model.node_recurrent_block)
        transfer_recurrent_block_weights(edge_recurrent_model, pt_model.edge_recurrent_block)
        transfer_censnet_weights(gnn_layer, pt_model.spatial_gnn_block)
    else: # Not using GNN
        # There is only one nested model, which Keras names 'model'.
        recurrent_model = tf_model.get_layer("model")
        transfer_recurrent_block_weights(recurrent_model, pt_model.recurrent_block)

In [ ]:
import deepof.clustering.models_new


class TestRecurrentEncoderTranslation(unittest.TestCase):
    def setUp(self):
        """Set up parameters and create random data that matches model assumptions."""
        tf.keras.backend.clear_session()
        self.latent_dim = 8
        
        # Real data configuration
        self.b = 2  # Batch
        self.t = 25  # Time
        self.n = 11  # Nodes  
        self.f_per_node = 3  # Features per node
        self.f_total = self.n * self.f_per_node  # Total features (33)
        self.e = 11  # Edges
        self.f_edge_per_edge = 1  # Features per edge
        self.f_edge_total = self.e * self.f_edge_per_edge  # Total edge features

        # For encoder, use FLATTENED shapes (as VQVAE does)
        self.input_shape = (self.t, self.f_total)  # (25, 33)
        self.edge_shape = (self.t, self.f_edge_total)  # (25, 11)
        # Create dummy adjacency matrix
        m = np.zeros((self.n, self.n))
        ui = np.triu_indices(self.n)
        num_possible_edges = len(ui[0])
        c = np.random.choice(num_possible_edges, min(self.e, num_possible_edges), replace=False)
        m[ui[0][c], ui[1][c]] = 1
        m += m.T # Make symmetric
        self.adj_matrix = m

        # Create random input data in FLATTENED format
        self.x_np = np.random.rand(self.b, self.t, self.f_total).astype(np.float32)
        self.a_np = np.random.rand(self.b, self.t, self.f_edge_total).astype(np.float32)
        
        # For PyTorch, we need to reshape to (B, T, N, F) format
        self.x_np_pt = self.x_np.reshape(self.b, self.t, self.n, self.f_per_node)
        self.a_np_pt = self.a_np.reshape(self.b, self.t, self.e, self.f_edge_per_edge)
        
    def test_forward_pass_gnn(self):
        """Test the GNN-enabled path of the encoder."""
        # Build TF model with FLATTENED input shapes
        tf_model_gnn = deepof.models.get_recurrent_encoder(
            self.input_shape,  # (25, 33)
            self.edge_shape,   # (25, 11)
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=True
        )
        
        # Build PT model with UNFLATTENED input shapes 
        pt_input_shape = (self.t, self.n, self.f_per_node)  # (25, 11, 3)
        pt_edge_shape = (self.t, self.e, self.f_edge_per_edge)  # (25, 11, 1)
        
        pt_model_gnn = deepof.clustering.models_new.RecurrentEncoderPT(
            pt_input_shape, 
            pt_edge_shape, 
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=True
        )
        pt_model_gnn.eval()

        # Run a single "dummy" forward pass on the PyTorch model to initialize weights
        with torch.no_grad():
            pt_model_gnn(torch.from_numpy(self.x_np_pt), torch.from_numpy(self.a_np_pt))

        # Transfer weights from TF to PT
        transfer_recurrent_encoder_weights(tf_model_gnn, pt_model_gnn)

        # Execute and compare the outputs
        # TF gets flattened input
        tf_start = time.time()
        tf_output = tf_model_gnn([self.x_np, self.a_np], training=False).numpy()
        tf_end = time.time()
        
        # PT gets unflattened input
        pt_start = time.time()
        with torch.no_grad():
            pt_output = pt_model_gnn(
                torch.from_numpy(self.x_np_pt), 
                torch.from_numpy(self.a_np_pt)
            ).detach().numpy()
        pt_end = time.time()

        print(f"Tensorflow execution time: {tf_end-tf_start:.4f}s")
        print(f"Pytorch execution time: {pt_end-pt_start:.4f}s")
        print(f"TF output shape: {tf_output.shape}")
        print(f"PT output shape: {pt_output.shape}")
        print(f"TF output sample: {tf_output[0, :5]}")
        print(f"PT output sample: {pt_output[0, :5]}")
        print(f"Max absolute difference: {np.abs(tf_output - pt_output).max():.8f}")

        np.testing.assert_allclose(tf_output, pt_output, rtol=1e-5, atol=1e-4)
        print("✅ `RecurrentEncoderPT` (GNN path) translation test PASSED!")

    def test_forward_pass_no_gnn(self):
        """Test the non-GNN path of the encoder."""
        # Build TF model with FLATTENED input shapes
        tf_model_no_gnn = deepof.models.get_recurrent_encoder(
            self.input_shape,  # (25, 33)
            self.edge_shape,   # (25, 11)
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=False
        )
        
        # Build PT model with UNFLATTENED input shapes
        pt_input_shape = (self.t, self.n, self.f_per_node)  # (25, 11, 3)
        pt_edge_shape = (self.t, self.e, self.f_edge_per_edge)  # (25, 11, 1)
        
        pt_model_no_gnn = deepof.clustering.models_new.RecurrentEncoderPT(
            pt_input_shape, 
            pt_edge_shape, 
            self.adj_matrix, 
            self.latent_dim, 
            use_gnn=False
        )
        pt_model_no_gnn.eval()

        # Transfer weights
        transfer_recurrent_encoder_weights(tf_model_no_gnn, pt_model_no_gnn)

        # Execute and compare
        # TF gets flattened input
        tf_output = tf_model_no_gnn([self.x_np, self.a_np], training=False).numpy()
        
        # PT gets unflattened input
        with torch.no_grad():
            pt_output = pt_model_no_gnn(
                torch.from_numpy(self.x_np_pt), 
                torch.from_numpy(self.a_np_pt)
            ).detach().numpy()
        
        print(f"TF output shape: {tf_output.shape}")
        print(f"PT output shape: {pt_output.shape}")
        print(f"TF output sample: {tf_output[0, :5]}")
        print(f"PT output sample: {pt_output[0, :5]}")
        print(f"Max absolute difference: {np.abs(tf_output - pt_output).max():.8f}")

        np.testing.assert_allclose(tf_output, pt_output, rtol=1e-5, atol=1e-5)
        print("✅ `RecurrentEncoderPT` (non-GNN path) translation test PASSED!")

# To run:
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestRecurrentEncoderTranslation)
runner.run(suite)

test_forward_pass_gnn (__main__.TestRecurrentEncoderTranslation)
Test the GNN-enabled path of the encoder. ... FAIL
test_forward_pass_no_gnn (__main__.TestRecurrentEncoderTranslation)
Test the non-GNN path of the encoder. ... ERROR

ERROR: test_forward_pass_no_gnn (__main__.TestRecurrentEncoderTranslation)
Test the non-GNN path of the encoder.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_36452\118059049.py", line 100, in test_forward_pass_no_gnn
    tf_model_no_gnn = deepof.models.get_recurrent_encoder(
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\deepof\data_loading.py", line 44, in somedec_inner
    response = fn(*args, **kwargs)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\deepof\models.py", line 112, in get_recurrent_encoder
    x_flat = tf.reshape(x, [-1, input_shape[0], input_shape[1] * input_shape[2]])
IndexError: tuple index out of range

FAIL: t

Tensorflow execution time: 0.1810s
Pytorch execution time: 0.0000s
TF output shape: (2, 8)
PT output shape: (2, 8)
TF output sample: [-0.44642478 -1.7641416  -0.05425613  2.625434   -1.5602206 ]
PT output sample: [-0.09484386 -0.6577374  -0.1929313   2.2434304  -0.6872003 ]
Max absolute difference: 2.05209970


<unittest.runner.TextTestResult run=2 errors=1 failures=1>

# Gaussian Mixture Latent

In [16]:
import unittest
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense
import tensorflow_probability as tfp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from typing import List, Tuple, Dict
import time
import deepof.model_utils

tfd = tfp.distributions

In [17]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Nadam
import tensorflow_probability as tfp
from tensorflow.keras.layers import Layer # Assuming ClusterControl inherits from this
from typing import List

# These are placeholders for the external utilities used in the original model
# to make the class definition self-contained and runnable.
class ClusterControl(Layer):
    """Placeholder for the custom deepof.model_utils.ClusterControl layer."""
    def __init__(self, batch_size, n_components, encoding_dim, k, **kwargs):
        super().__init__(**kwargs)
    def call(self, inputs: List[tf.Tensor]) -> tf.Tensor:
        # The layer is pass-through for the latent vector
        return inputs[0]

def compute_kmeans_loss(latent_means: tf.Tensor, weight: float, batch_size: int) -> tf.Tensor:
    """Placeholder for the custom deepof.model_utils.compute_kmeans_loss function."""
    gram_matrix = (tf.transpose(latent_means) @ latent_means) / tf.cast(batch_size, tf.float32)
    s = tf.linalg.svd(gram_matrix, compute_uv=False)
    s = tf.sqrt(tf.maximum(s, 1e-9))
    return weight * tf.reduce_mean(s)

# TensorFlow Probability layers
tfpl = tfp.layers
tfd = tfp.distributions


class GaussianMixtureLatent(tf.keras.models.Model):
    """Gaussian Mixture probabilistic latent space model.

    Used to represent the embedding of motion tracking data in a mixture of Gaussians
    with a provided number of components, with means, covariances and weights.
    Implementation based on VaDE (https://arxiv.org/abs/1611.05148)
    and VaDE-SC (https://openreview.net/forum?id=RQ428ZptQfU).

    """

    def __init__(
        self,
        input_shape: tuple,
        n_components: int,
        latent_dim: int,
        batch_size: int,
        kl_warmup: int = 5,
        kl_annealing_mode: str = "linear",
        mc_kl: int = 100,
        mmd_warmup: int = 15,
        mmd_annealing_mode: str = "linear",
        kmeans_loss: float = 0.0,
        reg_cluster_variance: bool = False,
        **kwargs,
    ):
        """Initialize the Gaussian Mixture Latent layer.

        Args:
            input_shape (tuple): shape of the input data
            n_components (int): number of components in the Gaussian mixture.
            latent_dim (int): dimensionality of the latent space.
            batch_size (int): batch size for training.
            kl_warmup (int): number of epochs to warm up the KL divergence.
            kl_annealing_mode (str): mode to use for annealing the KL divergence. Must be one of "linear" and "sigmoid".
            mc_kl (int): number of Monte Carlo samples to use for computing the KL divergence.
            mmd_warmup (int): number of epochs to warm up the MMD.
            mmd_annealing_mode (str): mode to use for annealing the MMD. Must be one of "linear" and "sigmoid".
            kmeans_loss (float): weight of the Gram matrix regularization loss.
            reg_cluster_variance (bool): whether to penalize uneven cluster variances in the latent space.
            **kwargs: keyword arguments passed to the parent class

        """
        super(GaussianMixtureLatent, self).__init__(**kwargs)
        self.seq_shape = input_shape[0] 
        self.n_components = n_components
        self.latent_dim = latent_dim
        self.batch_size = batch_size
        self.kl_warmup = kl_warmup
        self.kl_annealing_mode = kl_annealing_mode
        self.mc_kl = mc_kl
        self.mmd_warmup = mmd_warmup
        self.mmd_annealing_mode = mmd_annealing_mode
        self.kmeans = kmeans_loss
        self.optimizer = Nadam(learning_rate=1e-3, clipvalue=0.75)
        self.reg_cluster_variance = reg_cluster_variance
        self.pretrain = tf.Variable(0.0, name="pretrain", trainable=False)

        # Initialize GM parameters
        self.c_mu = tf.Variable(
            tf.initializers.GlorotNormal()(shape=[self.n_components, self.latent_dim]),
            name="mu_c",
        )
        self.log_c_sigma = tf.Variable(
            tf.initializers.GlorotNormal()([self.n_components, self.latent_dim]),
            name="log_sigma_c",
        )

        # Initialize the Gaussian Mixture prior with the specified number of components
        self.prior = tf.constant(tf.ones([self.n_components]) * (1 / self.n_components))

        # Initialize layers
        self.z_gauss_mean = Dense(
            tfpl.IndependentNormal.params_size(self.latent_dim) // 2,
            name="cluster_means",
            activation="linear",
            kernel_initializer="glorot_uniform",
            activity_regularizer=None,
        )
        self.z_gauss_var = Dense(
            tfpl.IndependentNormal.params_size(self.latent_dim) // 2,
            name="cluster_variances",
            activation="softplus",
            kernel_initializer="glorot_uniform",
            activity_regularizer=tf.keras.regularizers.l1(0.1),
        )

        self.cluster_control_layer = deepof.model_utils.ClusterControl(
            batch_size=self.batch_size,
            n_components=self.n_components,
            encoding_dim=self.latent_dim,
            k=self.n_components,
        )

        # control KL weight
        self.kl_warm_up_iters = tf.cast(
            self.kl_warmup * (self.seq_shape // self.batch_size), tf.int64
        )
        self._kl_weight = tf.Variable(
            1.0, trainable=False, dtype=tf.float32, name="kl_weight"
        )

    def call(self, inputs, training=False, epsilon=None, return_all_outputs_for_testing=False): # pragma: no cover
        """Compute the output of the layer."""
        z_gauss_mean = self.z_gauss_mean(inputs)
        z_gauss_var = self.z_gauss_var(inputs)

        if epsilon is not None:
            # Use deterministic reparameterization for testing
            z_sample = z_gauss_mean + tf.math.sqrt(tf.math.exp(z_gauss_var)) * epsilon
        else:
            # Original stochastic sampling for production
            z_dist = tfd.MultivariateNormalDiag(
                loc=z_gauss_mean, scale_diag=tf.math.sqrt(tf.math.exp(z_gauss_var))
            )
            z_sample = tf.squeeze(z_dist.sample())

        # Compute embedding probabilities given each cluster
        p_z_c = tf.stack(
            [
                tfd.MultivariateNormalDiag(
                    loc=self.c_mu[i, :],
                    scale_diag=tf.math.exp(self.log_c_sigma)[i, :],
                ).log_prob((z_sample if training else z_gauss_mean))
                + 1e-6
                for i in range(self.n_components)
            ],
            axis=-1,
        )

        # Update prior
        prior = self.prior

        # Compute cluster probabilitie given embedding
        z_cat = tf.math.log(prior + 1e-6) + p_z_c
        z_cat = tf.nn.log_softmax(z_cat, axis=-1)
        z_cat = tf.math.exp(z_cat)

        # Add clustering loss
        loss_clustering = -tf.reduce_sum(
            tf.multiply(z_cat, tf.math.softmax(p_z_c, axis=-1)), axis=-1
        ) * (1.0 - tf.cast(self.pretrain, tf.float32))
        loss_prior = -tf.math.reduce_sum(
            tf.math.xlogy(z_cat, 1e-6 + prior), axis=-1
        ) * (1.0 - tf.cast(self.pretrain, tf.float32))

        #self.add_metric(loss_clustering, name="clustering_loss", aggregation="mean")
        #self.add_metric(loss_prior, name="prior_loss", aggregation="mean")

        # Update KL weight based on the current iteration
        if self.kl_warm_up_iters > 0:
            if self.kl_annealing_mode in ["linear", "sigmoid"]:
                self._kl_weight = tf.cast(
                    tf.keras.backend.min(
                        [self.optimizer.iterations / self.kl_warm_up_iters, 1.0]
                    ),
                    tf.float32,
                )
                if self.kl_annealing_mode == "sigmoid":
                    self._kl_weight = tf.math.sigmoid(
                        (2 * self._kl_weight - 1)
                        / (self._kl_weight - self._kl_weight**2)
                    )
            else:
                raise NotImplementedError(
                    "annealing_mode must be one of 'linear' and 'sigmoid'"
                )
        else:
            self._kl_weight = tf.cast(1.0, tf.float32)

        loss_variational_1 = -1 / 2 * tf.reduce_sum(z_gauss_var + 1, axis=-1)
        loss_variational_2 = tf.math.reduce_sum(
            tf.math.xlogy(z_cat, 1e-6 + z_cat), axis=-1
        )
        kl = loss_variational_1 + loss_variational_2 * (
            1.0 - tf.cast(self.pretrain, tf.float32)
        )
        kl_batch = self._kl_weight * kl

        #self.add_metric(self._kl_weight, aggregation="mean", name="kl_weight")
        #self.add_metric(kl, aggregation="mean", name="kl_divergence")

        #self.add_loss(tf.math.reduce_mean(loss_clustering))
        #self.add_loss(tf.math.reduce_mean(loss_prior))
        #self.add_loss(tf.math.reduce_mean(kl_batch))


        # Calculate metrics for potential return
        hard_groups = tf.math.argmax(z_cat, axis=1)
        max_groups = tf.reduce_max(z_cat, axis=1)
        n_populated = tf.cast(tf.shape(tf.unique(tf.reshape(hard_groups, [-1]))[0])[0], tf.float32)
        confidence = tf.reduce_mean(max_groups)

        z = z_sample if training else z_gauss_mean

        if self.n_components > 1:
            z = self.cluster_control_layer([z, z_cat])

        k_loss = 0.0
        if self.kmeans:
            k_loss = deepof.model_utils.compute_kmeans_loss(z, weight=self.kmeans, batch_size=self.batch_size)
            #self.add_loss(k_loss)
            #self.add_metric(k_loss, name="kmeans_loss")

        # MODIFIED: Add a switch for the return value
        if return_all_outputs_for_testing:
            # In test mode, return all computed values for direct comparison
            return z, z_cat, n_populated, confidence, k_loss
        else:
            # In production mode, use side effects (add_loss/add_metric) and return the original signature
            loss_clustering = -tf.reduce_sum(tf.multiply(z_cat, tf.math.softmax(p_z_c, axis=-1)), axis=-1) * (1.0 - tf.cast(self.pretrain, tf.float32))
            loss_prior = -tf.reduce_sum(tf.math.xlogy(z_cat, 1e-6 + self.prior), axis=-1) * (1.0 - tf.cast(self.pretrain, tf.float32))
            self.add_metric(loss_clustering, name="clustering_loss", aggregation="mean")
            self.add_metric(loss_prior, name="prior_loss", aggregation="mean")

            self.add_metric(self._kl_weight, aggregation="mean", name="kl_weight")
            self.add_metric(kl, aggregation="mean", name="kl_divergence")

            self.add_loss(tf.math.reduce_mean(loss_clustering))
            self.add_loss(tf.math.reduce_mean(loss_prior))
            self.add_loss(tf.math.reduce_mean(kl_batch))

            if self.kmeans:
                self.add_loss(k_loss)
                self.add_metric(k_loss, name="kmeans_loss")

            # ... all other add_loss and add_metric calls from the original ...
            return z, z_cat

In [18]:
from typing import Dict, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

class ClusterControlPT(nn.Module):
    """
    Calculates clustering metrics. This is a pass-through layer for the main
    latent vector `z`, returning it unmodified alongside a dictionary of metrics.
    """
    def __init__(self):
        super().__init__()

    def forward(
        self, z: torch.Tensor, z_cat: torch.Tensor
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Calculates metrics and passes the latent vector `z` through.

        Args:
            z: The latent vector (batch_size, latent_dim).
            z_cat: Cluster probabilities (batch_size, n_components).

        Returns:
            A tuple containing the unmodified `z` and a dictionary of metrics.
        """
        confidence, hard_groups = torch.max(z_cat, dim=1)
        
        # Calculate the number of unique clusters populated in the batch
        num_populated = torch.unique(hard_groups).numel()
        
        metrics = {
            "number_of_populated_clusters": torch.tensor(
                float(num_populated), device=z.device
            ),
            "confidence_in_selected_cluster": torch.mean(confidence),
        }
        
        return z, metrics

def compute_kmeans_loss_pt(latent_means: torch.Tensor, weight: float) -> torch.Tensor:
    """
    Computes a loss based on the singular values of the Gram matrix of the
    latent vectors, encouraging orthogonality.

    Args:
        latent_means: The latent vectors from the model (batch_size, latent_dim).
        weight: The weight to apply to this loss component.

    Returns:
        The calculated scalar loss tensor.
    """
    batch_size = float(latent_means.shape[0])
    gram_matrix = (latent_means.T @ latent_means) / batch_size
    
    # Compute singular values, which are the square roots of the eigenvalues for a symmetric matrix
    singular_values = torch.linalg.svdvals(gram_matrix)
    
    # Clamp to avoid NaN gradients from sqrt(0)
    penalization = torch.sqrt(torch.clamp(singular_values, min=1e-9))
    
    return weight * torch.mean(penalization)


class GaussianMixtureLatentPT(nn.Module):
    """
    PyTorch implementation of the Gaussian Mixture probabilistic latent space model.
    It embeds data into a latent space and models that space as a mixture of Gaussians.
    """
    def __init__(
        self,
        input_dim: int,
        n_components: int,
        latent_dim: int,
        kmeans: float,
        **kwargs,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.n_components = n_components
        self.latent_dim = latent_dim
        self.kmeans_weight = kmeans

        # --- Trainable Parameters for the GMM components ---
        self.gmm_means = nn.Parameter(torch.empty(n_components, latent_dim))
        self.gmm_log_vars = nn.Parameter(torch.empty(n_components, latent_dim))
        nn.init.xavier_normal_(self.gmm_means)
        nn.init.xavier_normal_(self.gmm_log_vars)

        # --- Encoder Layers to produce the latent distribution ---
        self.encoder_mean = nn.Linear(self.input_dim, self.latent_dim)
        self.encoder_log_var = nn.Linear(self.input_dim, self.latent_dim)

        # --- Non-trainable Buffers ---
        self.register_buffer('prior', torch.ones(n_components) / n_components)
        self.register_buffer('pretrain', torch.tensor(0.0))
        
        # --- Helper Layers ---
        self.cluster_control = ClusterControlPT()

    def _encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encodes the input into mean and log-variance of the latent distribution."""
        z_mean = self.encoder_mean(x)
        z_log_var = self.encoder_log_var(x) # Note: softplus is applied in the forward pass
        return z_mean, z_log_var

    def _reparameterize(
        self, mean: torch.Tensor, var: torch.Tensor, epsilon: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Performs reparameterization.
        MODIFIED to exactly replicate the original TF model's non-standard scale calculation.
        """
        # Original TF logic: scale = sqrt(exp(variance))
        # The 'var' input here is the direct output of the softplus activation.
        scale = torch.sqrt(torch.exp(var))
        
        if epsilon is None:
            epsilon = torch.randn_like(scale)
        return mean + scale * epsilon

    def _calculate_posterior(self, z: torch.Tensor) -> torch.Tensor:
        """Calculates the posterior probability p(c|z) for each sample."""
        # MODIFIED: The GMM parameters from TF are log-std-dev, not log-variance.
        # So we just exponentiate them to get the scale.
        gmm_scale = torch.exp(self.gmm_log_vars)

        gmm_dist = Normal(
            loc=self.gmm_means.unsqueeze(0),
            scale=gmm_scale.unsqueeze(0)
        )
        log_p_z_given_c = gmm_dist.log_prob(z.unsqueeze(1)).sum(dim=-1)
        
        log_p_c_given_z = torch.log(self.prior + 1e-9) + log_p_z_given_c
        
        return F.softmax(log_p_c_given_z, dim=-1)

    def forward(
        self, x: torch.Tensor, epsilon: torch.Tensor = None
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        
        z_mean, z_var_raw = self._encode(x)
        z_var = F.softplus(z_var_raw) # Apply activation

        # Pass z_var directly, not z_log_var
        z_sample = self._reparameterize(z_mean, z_var, epsilon)
        # ... rest of the method is the same ...
        z_for_downstream = z_sample if self.training else z_mean
        z_cat = self._calculate_posterior(z_for_downstream)
        z_final, metrics = self.cluster_control(z_for_downstream, z_cat)
        kmeans_loss = torch.tensor(0.0, device=x.device)
        if self.kmeans_weight > 0:
            kmeans_loss = compute_kmeans_loss_pt(z_final, weight=self.kmeans_weight)
        return (z_final, z_cat, metrics["number_of_populated_clusters"], metrics["confidence_in_selected_cluster"], kmeans_loss)

In [19]:
def transfer_gmm_weights(tf_model, pt_model: GaussianMixtureLatentPT):
    """
    Transfers weights from the final TF model to the refactored PT model,
    using the updated attribute names.
    """
    # --- Transfer GMM component parameters ---
    # OLD: pt_model.c_mu
    pt_model.gmm_means.data = torch.from_numpy(tf_model.c_mu.numpy())
    # OLD: pt_model.log_c_sigma
    pt_model.gmm_log_vars.data = torch.from_numpy(tf_model.log_c_sigma.numpy())

    # --- Transfer Encoder layer parameters ---
    tf_mean_weights = tf_model.z_gauss_mean.get_weights()
    # OLD: pt_model.z_gauss_mean
    pt_model.encoder_mean.weight.data = torch.from_numpy(tf_mean_weights[0].T)
    pt_model.encoder_mean.bias.data = torch.from_numpy(tf_mean_weights[1])
    
    tf_var_weights = tf_model.z_gauss_var.get_weights()
    # OLD: pt_model.z_gauss_var
    pt_model.encoder_log_var.weight.data = torch.from_numpy(tf_var_weights[0].T)
    pt_model.encoder_log_var.bias.data = torch.from_numpy(tf_var_weights[1])

In [20]:
class TestGMMFinalSimplified(unittest.TestCase):
    def setUp(self):
        self.input_dim, self.latent_dim, self.n_components, self.batch_size = 64, 16, 5, 4
        self.seq_shape = self.batch_size * 100
        self.kmeans_weight = 0.1

        tf.keras.backend.clear_session()
        # Instantiate the *actual* final TF model
        self.tf_model = GaussianMixtureLatent(
            input_shape=(self.seq_shape, self.input_dim),
            n_components=self.n_components,
            latent_dim=self.latent_dim,
            batch_size=self.batch_size,
            kmeans_loss=self.kmeans_weight
        )
        # Build the model using the test-mode signature
        self.tf_model(
            tf.zeros((1, self.input_dim)), 
            epsilon=tf.zeros((1, self.latent_dim)),
            return_all_outputs_for_testing=True
        )

        # PyTorch model setup remains the same
        self.pt_model = GaussianMixtureLatentPT(
            input_dim=self.input_dim, n_components=self.n_components,
            latent_dim=self.latent_dim, kmeans=self.kmeans_weight
        )
        
        transfer_gmm_weights(self.tf_model, self.pt_model)
        
        self.np_input = np.random.rand(self.batch_size, self.input_dim).astype(np.float32)
        seed = 42
        np.random.seed(seed)
        epsilon_np = np.random.randn(self.batch_size, self.latent_dim).astype(np.float32)
        self.epsilon_tf = tf.convert_to_tensor(epsilon_np)
        self.epsilon_pt = torch.from_numpy(epsilon_np)

    def run_comparison_test(self, training_mode: bool):
        mode_str = "TRAINING" if training_mode else "EVALUATION"
        print(f"\n--- Testing final integration in {mode_str} mode ---")
        
        self.pt_model.train(training_mode)

        tf_start = time.time()
        # Call the TF model with test flags enabled
        tf_z, tf_z_cat, tf_n_pop, tf_conf, tf_kmeans = self.tf_model(
            self.np_input, 
            training=training_mode, 
            epsilon=self.epsilon_tf, 
            return_all_outputs_for_testing=True
        )
        tf_end = time.time()
        
        pt_start = time.time()
        # PyTorch call remains the same
        with torch.no_grad():
            pt_z, pt_z_cat, pt_n_pop, pt_conf, pt_kmeans = self.pt_model(
                torch.from_numpy(self.np_input), epsilon=self.epsilon_pt
            )
        pt_end = time.time()
        
        print("Tensorflow execution time: " + str(tf_end-tf_start))
        print("Pytorch execution time: " + str(pt_end-pt_start))
        
        print("Comparing all outputs...")
        np.testing.assert_allclose(tf_z.numpy(), pt_z.numpy(), rtol=1e-5, atol=1e-5)
        np.testing.assert_allclose(tf_z_cat.numpy(), pt_z_cat.numpy(), rtol=1e-5, atol=1e-5)
        np.testing.assert_allclose(tf_n_pop.numpy(), pt_n_pop.numpy(), rtol=1e-5, atol=1e-5)
        np.testing.assert_allclose(tf_conf.numpy(), pt_conf.numpy(), rtol=1e-5, atol=1e-5)
        np.testing.assert_allclose(tf_kmeans.numpy(), pt_kmeans.numpy(), rtol=1e-5, atol=1e-5)
        print(f"✅ Final integration in {mode_str} mode PASSED!")

    def test_final_pass_train_mode(self):
        self.run_comparison_test(training_mode=True)
    
    def test_final_pass_eval_mode(self):
        self.run_comparison_test(training_mode=False)


# Run the test
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestGMMFinalSimplified)
runner.run(suite)

test_final_pass_eval_mode (__main__.TestGMMFinalSimplified) ... ok
test_final_pass_train_mode (__main__.TestGMMFinalSimplified) ... 


--- Testing final integration in EVALUATION mode ---
Tensorflow execution time: 0.053191423416137695
Pytorch execution time: 0.006725311279296875
Comparing all outputs...
✅ Final integration in EVALUATION mode PASSED!

--- Testing final integration in TRAINING mode ---


ok

----------------------------------------------------------------------
Ran 2 tests in 0.625s

OK


Tensorflow execution time: 0.05488920211791992
Pytorch execution time: 0.0
Comparing all outputs...
✅ Final integration in TRAINING mode PASSED!


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

# Get Vade

In [21]:
import unittest
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense
import tensorflow_probability as tfp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from typing import List, Tuple, Dict, Callable
import time
import deepof.model_utils
from deepof.model_utils import ClusterControl, compute_kmeans_loss, CensNetConv, ProbabilisticDecoder
from deepof.models import get_recurrent_encoder, get_recurrent_decoder, GaussianMixtureLatent, get_TCN_encoder, get_TCN_decoder, get_transformer_encoder, get_transformer_decoder
from deepof.clustering.models_new import RecurrentEncoderPT, RecurrentDecoderPT, GaussianMixtureLatentPT
from tensorflow.keras.layers import (
    GRU,
    Bidirectional,
    Dense,
    LayerNormalization,
    RepeatVector,
    TimeDistributed,
)

tfd = tfp.distributions

In [22]:
def get_vade(
    input_shape: tuple,
    edge_feature_shape: tuple,
    adjacency_matrix: np.ndarray,
    latent_dim: int,
    use_gnn: bool,
    n_components: int,
    batch_size: int = 64,
    kl_warmup: int = 15,
    kl_annealing_mode: str = "sigmoid",
    mc_kl: int = 100,
    kmeans_loss: float = 1.0,
    reg_cluster_variance: bool = False,
    encoder_type: str = "recurrent",
    interaction_regularization: float = 0.0,
):
    """Build a Gaussian mixture variational autoencoder (VaDE) model, adapted to the DeepOF setting.

    Args:
        input_shape (tuple): shape of the input data.
        edge_feature_shape (tuple): shape of the edge feature matrix used for graph representations.
        adjacency_matrix (np.ndarray): adjacency matrix of the connectivity graph to use.
        latent_dim (int): dimensionality of the latent space.
        use_gnn (bool): If True, the encoder uses a graph representation of the input, with coordinates and speeds as node attributes, and distances as edge attributes. If False, a regular 3D tensor is used as input.
        n_components (int): number of components in the Gaussian mixture.
        batch_size (int): batch size for training.
        kl_warmup (int): Number of iterations during which to warm up the KL divergence.
        kl_annealing_mode (str): mode to use for annealing the KL divergence. Must be one of "linear" and "sigmoid".
        mc_kl (int): number of Monte Carlo samples to use for computing the KL divergence.
        kmeans_loss (float): weight of the Gram matrix loss as described in deepof.model_utils.compute_kmeans_loss.
        reg_cluster_variance (bool): whether to penalize uneven cluster variances in the latent space.
        encoder_type (str): type of encoder to use. Can be set to "recurrent" (default), "TCN", or "transformer".
        interaction_regularization (float): weight of the interaction regularization term.

    Returns:
        encoder (tf.keras.Model): connected encoder of the VQ-VAE model. Outputs a vector of shape (latent_dim,).
        decoder (tf.keras.Model): connected decoder of the VQ-VAE model.
        grouper (tf.keras.Model): deep clustering branch of the VQ-VAE model. Outputs a vector of shape (n_components,) for each training instance, corresponding to the soft counts for each cluster.
        vade (tf.keras.Model): complete VaDE model

    """
    if encoder_type == "recurrent":
        encoder = get_recurrent_encoder(
            input_shape=input_shape[1:],
            adjacency_matrix=adjacency_matrix,
            edge_feature_shape=edge_feature_shape[1:],
            latent_dim=latent_dim,
            use_gnn=use_gnn,
            interaction_regularization=interaction_regularization,
        )
        decoder = get_recurrent_decoder(
            input_shape=input_shape[1:], latent_dim=latent_dim
        )

    elif encoder_type == "TCN":
        encoder = get_TCN_encoder(
            input_shape=input_shape[1:],
            adjacency_matrix=adjacency_matrix,
            edge_feature_shape=edge_feature_shape[1:],
            latent_dim=latent_dim,
            use_gnn=use_gnn,
            interaction_regularization=interaction_regularization,
        )
        decoder = get_TCN_decoder(input_shape=input_shape[1:], latent_dim=latent_dim)

    elif encoder_type == "transformer":
        encoder = get_transformer_encoder(
            input_shape[1:],
            edge_feature_shape=edge_feature_shape[1:],
            adjacency_matrix=adjacency_matrix,
            latent_dim=latent_dim,
            use_gnn=use_gnn,
            interaction_regularization=interaction_regularization,
        )
        decoder = get_transformer_decoder(input_shape[1:], latent_dim=latent_dim)

    latent_space = GaussianMixtureLatent(
        input_shape=input_shape[0],
        n_components=n_components,
        latent_dim=latent_dim,
        batch_size=batch_size,
        kl_warmup=kl_warmup,
        kl_annealing_mode=kl_annealing_mode,
        mc_kl=mc_kl,
        kmeans_loss=kmeans_loss,
        reg_cluster_variance=reg_cluster_variance,
        name="gaussian_mixture_latent",
    )

    # Connect encoder and latent space
    inputs = Input(input_shape[1:])
    a = tf.keras.layers.Input(edge_feature_shape[1:], name="encoder_edge_features")
    encoder_outputs = encoder([inputs, a])
    latent, categorical = latent_space(encoder_outputs)
    embedding = tf.keras.Model([inputs, a], latent, name="encoder")
    grouper = tf.keras.Model([inputs, a], categorical, name="grouper")

    # Connect decoder
    vade_outputs = decoder([embedding.outputs, inputs])

    # Instantiate fully connected model
    vade = tf.keras.Model(embedding.inputs, vade_outputs, name="VaDE")

    return embedding, decoder, grouper, vade


In [23]:
import torch
import torch.nn as nn
import numpy as np
from typing import Tuple

# Assume the following translated blocks are imported and available:
# from deepof.clustering.models_new import (
#     RecurrentEncoderPT, RecurrentDecoderPT, GaussianMixtureLatentPT
# )
# And their corresponding TensorFlow versions and weight transfer functions are also available.

class VaDEPT(nn.Module):
    """
    A self-contained PyTorch implementation of the VaDE model.

    This class encapsulates the entire VaDE architecture, including the encoder,
    the Gaussian mixture latent space, and the decoder. It is instantiated with
    all necessary configuration parameters, building its sub-modules internally.
    This provides a clean, single-object interface for the model.
    """
    def __init__(
        self,
        input_shape: tuple,
        edge_feature_shape: tuple,
        adjacency_matrix: np.ndarray,
        latent_dim: int,
        n_components: int,
        use_gnn: bool = True,
        kmeans_loss: float = 1.0,
        interaction_regularization: float = 0.0,
    ):
        """
        Initializes and builds the VaDE model and its components.

        Args:
            input_shape (tuple): Shape of the input node features (Time, Nodes, Features_per_node).
            edge_feature_shape (tuple): Shape of the edge features (Time, Edges, Features_per_edge).
            adjacency_matrix (np.ndarray): Adjacency matrix of the connectivity graph.
            latent_dim (int): Dimensionality of the latent space.
            n_components (int): Number of components in the Gaussian mixture.
            use_gnn (bool): If True, use the GNN-based encoder.
            kmeans_loss (float): Weight of the k-means style loss in the latent space.
            interaction_regularization (float): Regularization for GNN interaction features.
        """
        super().__init__()
        
        # Store key dimensions for internal use (e.g., reshaping in forward pass)
        time_steps, n_nodes, n_features_per_node = input_shape
        self.input_n_nodes = n_nodes
        self.input_n_features_per_node = n_features_per_node

        # 1. Instantiate Encoder
        self.encoder = RecurrentEncoderPT(
            input_shape=input_shape,
            edge_feature_shape=edge_feature_shape,
            adjacency_matrix=adjacency_matrix,
            latent_dim=latent_dim,
            use_gnn=use_gnn,
            interaction_regularization=interaction_regularization,
        )

        # 2. Instantiate Latent Space
        self.latent_space = GaussianMixtureLatentPT(
            input_dim=latent_dim,
            n_components=n_components,
            latent_dim=latent_dim,
            kmeans=kmeans_loss,
        )

        # 3. Instantiate Decoder
        decoder_output_features = n_nodes * n_features_per_node
        self.decoder = RecurrentDecoderPT(
            output_shape=(time_steps, decoder_output_features),
            latent_dim=latent_dim,
        )

    def forward(
        self, x: torch.Tensor, a: torch.Tensor
    ) -> Tuple[torch.distributions.Distribution, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Defines the full forward pass for the VaDE model (training and evaluation).

        Args:
            x (torch.Tensor): Input node features tensor (B, T, N, F_node).
            a (torch.Tensor): Input edge features tensor (B, T, E, F_edge).

        Returns:
            A tuple containing:
            - reconstruction_dist (torch.distributions.Distribution): The output distribution from the decoder.
            - latent (torch.Tensor): The sampled latent representation from the GMM space.
            - categorical (torch.Tensor): The cluster probabilities (soft assignments).
            - kmeans_loss (torch.Tensor): The k-means regularization loss from the latent space.
        """
        # 1. Encode the input to get the pre-latent representation
        encoder_output = self.encoder(x, a)
        
        # 2. Pass through GMM latent space
        latent, categorical, _, _, kmeans_loss, gmm_params = self.latent_space(encoder_output)
        
        # 3. Decode the latent sample back to the original data space
        # Reshape x to (B, T, N*F) for the decoder's masking logic
        B, T, _, _ = x.shape
        x_for_decoder = x.view(B, T, self.input_n_nodes * self.input_n_features_per_node)
        
        reconstruction_dist = self.decoder(latent, x_for_decoder)
        
        return reconstruction_dist, latent, categorical, kmeans_loss

    @torch.no_grad()
    def embed(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """
        Inference-only method to get the latent embedding. Equivalent to the 'embedding' Keras model.

        Args:
            x (torch.Tensor): Input node features tensor.
            a (torch.Tensor): Input edge features tensor.

        Returns:
            torch.Tensor: The latent representation `z`.
        """
        encoder_output = self.encoder(x, a)
        latent, _, _, _, _, _ = self.latent_space(encoder_output)
        return latent

    @torch.no_grad()
    def group(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """
        Inference-only method to get cluster probabilities. Equivalent to the 'grouper' Keras model.

        Args:
            x (torch.Tensor): Input node features tensor.
            a (torch.Tensor): Input edge features tensor.

        Returns:
            torch.Tensor: The soft cluster assignments (categorical probabilities).
        """
        encoder_output = self.encoder(x, a)
        _, categorical, _, _, _, _ = self.latent_space(encoder_output)
        return categorical

In [24]:
def transfer_recurrent_block_weights(tf_model, pt_model):
    """Transfers weights for the full recurrent block with GRU gate permutation."""
    conv_td, _, gru1_td, norm1, gru2_td, norm2 = tf_model.layers[1:]


    def permute_gru_weights(keras_weights):
        W_ih, W_hh, B = keras_weights
        W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
        W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)
        W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
        W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)
        B_ih, B_hh = B
        B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
        B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)
        B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
        B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])
        return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt

    pt_model.conv1d.weight.data = torch.from_numpy(conv_td.layer.get_weights()[0]).permute(2, 1, 0)
    
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(gru1_td.layer.forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1); pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(gru1_td.layer.backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1); pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)

    pt_model.norm1.weight.data = torch.from_numpy(norm1.get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm1.get_weights()[1])

    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(gru2_td.layer.forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2); pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(gru2_td.layer.backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2); pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    
    pt_model.norm2.weight.data = torch.from_numpy(norm2.get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm2.get_weights()[1])

    
def transfer_censnet_weights(tf_layer, pt_layer):
    """
    Transfers all six weights from a Spektral CensNetConv layer to the
    corresponding CensNetConvPT layer.
    """
    # Get all weights from the TensorFlow layer. The order is determined by
    # the layer's build order in Spektral's source code.
    tf_weights = tf_layer.get_weights()

    # Unpack all six weights.
    # Order: kernel_node, bias_node, kernel_edge, bias_edge, projector_node, projector_edge
    kn_tf, bn_tf, ke_tf, be_tf, pn_tf, pe_tf = tf_weights

    # Build weights on first pass
    if pt_layer.node_kernel is None:
        # Move parameters to the same device as input tensors
        pt_layer._build(kn_tf.T.shape, bn_tf.T.shape)
        #pt_layer.to(kn_tf.device)

    # 1. & 2. Transfer Node Kernel and Bias
    # Keras Dense kernel is (in_features, out_features)
    pt_layer.node_kernel.data = torch.from_numpy(kn_tf)
    pt_layer.edge_kernel.data = torch.from_numpy(bn_tf)

    # 3. & 4. Transfer Edge Kernel and Bias
    # Same transposition logic applies.
    pt_layer.node_weights.data = torch.from_numpy(ke_tf)
    pt_layer.edge_weights.data = torch.from_numpy(be_tf)

    # 5. Transfer Node Projector Weights (P_n)
    # These are [in_features, 1], which matches, so no transpose needed.
    pt_layer.node_bias.data = torch.from_numpy(pn_tf)

    # 6. Transfer Edge Projector Weights (P_e)
    # These are [in_features, 1], which matches, so no transpose needed.
    pt_layer.edge_bias.data = torch.from_numpy(pe_tf)
    

def transfer_recurrent_encoder_weights(tf_model, pt_model):
    """
    Transfers weights for the full recurrent encoder, finding layers
    by their default names and types to avoid modifying original code.
    """
    # The final dense layer is consistently the last one in the model's layer list.
    final_dense_tf = tf_model.layers[-1]
    final_dense_pt = pt_model.final_dense
    w, b = final_dense_tf.get_weights()
    final_dense_pt.weight.data = torch.from_numpy(w.T)
    final_dense_pt.bias.data = torch.from_numpy(b)

    if pt_model.use_gnn:
        # Keras automatically names nested models 'model', 'model_1', etc., by order of creation.
        # Node recurrent block is created first.
        node_recurrent_model = tf_model.get_layer("model")
        # Edge recurrent block is created second.
        edge_recurrent_model = tf_model.get_layer("model_1")
        # Find the CensNetConv layer by its class type.
        gnn_layer = next(l for l in tf_model.layers if isinstance(l, CensNetConv))

        transfer_recurrent_block_weights(node_recurrent_model, pt_model.node_recurrent_block)
        transfer_recurrent_block_weights(edge_recurrent_model, pt_model.edge_recurrent_block)
        transfer_censnet_weights(gnn_layer, pt_model.spatial_gnn_block)
    else: # Not using GNN
        # There is only one nested model, which Keras names 'model'.
        recurrent_model = tf_model.get_layer("model")
        transfer_recurrent_block_weights(recurrent_model, pt_model.recurrent_block)

In [25]:
def transfer_gmm_weights(tf_model, pt_model: GaussianMixtureLatentPT):
    """
    Transfers weights from the final TF model to the refactored PT model,
    using the updated attribute names.
    """
    # --- Transfer GMM component parameters ---
    # OLD: pt_model.c_mu
    pt_model.gmm_means.data = torch.from_numpy(tf_model.c_mu.numpy())
    # OLD: pt_model.log_c_sigma
    pt_model.gmm_log_vars.data = torch.from_numpy(tf_model.log_c_sigma.numpy())

    # --- Transfer Encoder layer parameters ---
    tf_mean_weights = tf_model.z_gauss_mean.get_weights()
    # OLD: pt_model.z_gauss_mean
    pt_model.encoder_mean.weight.data = torch.from_numpy(tf_mean_weights[0].T)
    pt_model.encoder_mean.bias.data = torch.from_numpy(tf_mean_weights[1])
    
    tf_var_weights = tf_model.z_gauss_var.get_weights()
    # OLD: pt_model.z_gauss_var
    pt_model.encoder_log_var.weight.data = torch.from_numpy(tf_var_weights[0].T)
    pt_model.encoder_log_var.bias.data = torch.from_numpy(tf_var_weights[1])

In [26]:
# Helper function from the provided example to handle gate order differences
def permute_gru_weights(keras_weights):
    """Permutes GRU weights from Keras (z, r, n) to PyTorch (r, z, n) format."""
    W_ih, W_hh, B = keras_weights
    # Keras gate order: z, r, n (update, reset, new/candidate)
    W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
    W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)

    # PyTorch gate order: r, z, n (reset, update, new/candidate)
    W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
    W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)

    # Keras has two bias vectors (input-hidden and recurrent), which are concatenated in B
    B_ih, B_hh = B
    B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
    B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)

    B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
    B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])

    return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt
    
def transfer_recurrent_decoder_weights(tf_model, pt_model):
    """
    Transfers weights for the full recurrent decoder model.
    """
    # Find layers by type to avoid index issues
    bidi_layers = [l for l in tf_model.layers if isinstance(l, Bidirectional)]
    norm_layers = [l for l in tf_model.layers if isinstance(l, LayerNormalization)]
    conv_layers = [l for l in tf_model.layers if isinstance(l, tf.keras.layers.Conv1D)]
    prob_dec_layer = next(l for l in tf_model.layers if isinstance(l, deepof.model_utils.ProbabilisticDecoder))

    # --- GRU 1 and Norm 1 ---
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(bidi_layers[0].forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1)
    pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(bidi_layers[0].backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1)
    pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)
    pt_model.norm1.weight.data = torch.from_numpy(norm_layers[0].get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm_layers[0].get_weights()[1])

    # --- GRU 2 and Norm 2 ---
    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(bidi_layers[1].forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2)
    pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(bidi_layers[1].backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2)
    pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    pt_model.norm2.weight.data = torch.from_numpy(norm_layers[1].get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm_layers[1].get_weights()[1])

    # --- Conv1D and Norm 3 ---
    # TF Conv1D weights: (kernel_w, kernel_h, in_c, out_c) -> (5, 1, 4*ld, 2*ld)
    # PT Conv1d weights: (out_c, in_c, kernel_w)
    conv_weights_tf = conv_layers[0].get_weights()[0]
    pt_model.conv1d.weight.data = torch.from_numpy(conv_weights_tf).squeeze(1).permute(2, 1, 0)
    pt_model.norm3.weight.data = torch.from_numpy(norm_layers[2].get_weights()[0]); pt_model.norm3.bias.data = torch.from_numpy(norm_layers[2].get_weights()[1])

    # --- Probabilistic Decoder ---
    # TF Dense weights: (in_features, out_features)
    # PT Linear weights: (out_features, in_features)
    prob_dec_weights, prob_dec_bias = prob_dec_layer.time_distributer.get_weights()
    pt_model.prob_decoder.loc_projection.weight.data = torch.from_numpy(prob_dec_weights.T)
    pt_model.prob_decoder.loc_projection.bias.data = torch.from_numpy(prob_dec_bias)

In [27]:
# Imports and Mocks from the previous response are assumed to be present
import unittest
import tensorflow as tf
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
import time
import deepof.clustering.models_new
# End of Mocks


def transfer_vade_class_weights(tf_vade_model, tf_decoder_model, pt_vade_model: VaDEPT):
    """
    Transfers weights from a full TensorFlow VaDE model to the self-contained PyTorch VaDEPT class.
    """
    print("Transferring weights for all VaDE components...")
    
    # 1. Get the inner Keras models/layers by name from the complete TF model
    tf_encoder_inner = tf_vade_model.get_layer("recurrent_encoder")
    tf_latent_layer = tf_vade_model.get_layer("gaussian_mixture_latent")
    
    # 2. Use the specialized weight transfer functions, passing the PT sub-modules
    print("  -> Transferring Encoder weights...")
    transfer_recurrent_encoder_weights(tf_encoder_inner, pt_vade_model.encoder)
    print("  -> Transferring GMM Latent weights...")
    transfer_gmm_weights(tf_latent_layer, pt_vade_model.latent_space)
    print("  -> Transferring Decoder weights...")
    transfer_recurrent_decoder_weights(tf_decoder_model, pt_vade_model.decoder)
    
    print("Weight transfer complete.")


class TestVaDETranslation(unittest.TestCase):
    def setUp(self):
        """Set up parameters, models, and data for testing."""
        tf.keras.backend.clear_session()
        tf.keras.backend.set_epsilon(1e-3)

        # --- 1. Define Fundamental Dimensions ---
        self.batch_size = 128
        self.window_length = 25
        self.num_nodes = 11
        # In your example, total features (n=33) / num_nodes (11) = 3
        self.features_per_node = 33
        self.num_edges = 11
        self.features_per_edge = 111 # Assuming 1 feature per edge

        # --- 2. Define Model Parameters ---
        self.latent_dim = 6
        self.n_components = 10
        self.kmeans_loss = 1.0
        self.use_gnn = False

        # --- 3. Create Adjacency Matrix ---
        m = np.zeros((self.num_nodes, self.num_nodes))
        ui = np.triu_indices(self.num_nodes)
        num_possible_edges = len(ui[0])
        c = np.random.choice(num_possible_edges, min(self.num_edges, num_possible_edges), replace=False)
        m[ui[0][c], ui[1][c]] = 1
        m += m.T # Make symmetric
        self.adj_matrix = m

        # --- 4. Create Framework-Specific Shapes for Model Instantiation ---
        
        # TensorFlow expects (batch, time, total_features)
        self.input_shape_tf = (self.batch_size, self.window_length, self.num_nodes * self.features_per_node)
        self.edge_feature_shape_tf = (self.batch_size, self.window_length, self.num_edges * self.features_per_edge)
        
        # PyTorch VaDEPT expects (time, nodes, features_per_node) for a SINGLE sample
        self.input_shape_pt = (self.window_length, self.num_nodes, self.features_per_node)
        self.edge_feature_shape_pt = (self.window_length, self.num_edges, self.features_per_edge)

        # --- 5. Instantiate Models ---
        self.tf_embedding, self.tf_decoder, self.tf_grouper, self.tf_vade = get_vade(
            input_shape=self.input_shape_tf,
            edge_feature_shape=self.edge_feature_shape_tf,
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=self.use_gnn,
            n_components=self.n_components,
            batch_size=self.batch_size,
            kmeans_loss=self.kmeans_loss
        )
        
        self.pt_vade = VaDEPT(
            input_shape=self.input_shape_pt,
            edge_feature_shape=self.edge_feature_shape_pt,
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            n_components=self.n_components,
            use_gnn=self.use_gnn,
            kmeans_loss=self.kmeans_loss
        )
        self.pt_vade.eval()

        # --- 6. Prepare Data Tensors for Each Framework ---
        np.random.seed(42)
        # The "canonical" data is 4D, as expected by the new PyTorch models
        self.x_np_4d = np.random.rand(
            self.batch_size, self.window_length, self.num_nodes, self.features_per_node
        ).astype(np.float32)
        self.a_np_4d = np.random.rand(
            self.batch_size, self.window_length, self.num_edges, self.features_per_edge
        ).astype(np.float32)

        # Create the 3D version for the legacy TensorFlow model by reshaping
        self.x_np_tf = self.x_np_4d.reshape(self.input_shape_tf)
        self.a_np_tf = self.a_np_4d.reshape(self.edge_feature_shape_tf)
        
        # --- 7. Transfer Weights ---
        transfer_vade_class_weights(self.tf_vade, self.tf_decoder, self.pt_vade)

    def test_full_model_and_parts(self):
        """Test the forward pass and helper methods of the VaDEPT class."""
        print("\n--- Testing Self-Contained VaDEPT Class Translation ---")
        
        # --- TensorFlow Execution (with its required 3D input) ---
        tf_start = time.time()
        tf_rec_dist = self.tf_vade([self.x_np_tf, self.a_np_tf], training=False)
        tf_rec_mean = tf_rec_dist.mean().numpy()
        tf_lat_out = self.tf_embedding([self.x_np_tf, self.a_np_tf], training=False).numpy()
        tf_cat_out = self.tf_grouper([self.x_np_tf, self.a_np_tf], training=False).numpy()
        tf_end = time.time()
        
        # --- PyTorch Execution (with its required 4D input) ---
        x_pt = torch.from_numpy(self.x_np_4d)
        a_pt = torch.from_numpy(self.a_np_4d)
        
        pt_start = time.time()
        with torch.no_grad():
            pt_rec_dist, _, _, _ = self.pt_vade(x_pt, a_pt)
            pt_rec_mean = pt_rec_dist.mean.numpy() 
            pt_lat_out = self.pt_vade.embed(x_pt, a_pt).numpy()
            pt_cat_out = self.pt_vade.group(x_pt, a_pt).numpy()
        pt_end = time.time()

        print(f"TensorFlow execution time: {tf_end - tf_start:.6f}s")
        print(f"PyTorch execution time: {pt_end - pt_start:.6f}s")
        
        # --- Assertions ---
        print("\nComparing latent space embeddings (from .embed() vs 'embedding' model)...")
        # Both outputs should be (batch_size, latent_dim), so (128, 6)
        np.testing.assert_allclose(tf_lat_out, pt_lat_out, rtol=1e-5, atol=1e-4)
        print("✅ Latent embeddings match.")

        print("Comparing categorical probabilities (from .group() vs 'grouper' model)...")
        # Both outputs should be (batch_size, n_components), so (128, 10)
        np.testing.assert_allclose(tf_cat_out, pt_cat_out, rtol=1e-5, atol=1e-5)
        print("✅ Categorical probabilities match.")
        
        print("Comparing final reconstruction means (from forward() vs 'vade' model)...")
        # Both outputs should be (batch_size, time_steps, total_features), so (128, 25, 33)
        np.testing.assert_allclose(tf_rec_mean, pt_rec_mean, rtol=1e-5, atol=1e-4)
        print("✅ Reconstructions match.")

        print("\n✅ Self-contained VaDEPT class translation test PASSED!")

# To run the test
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestVaDETranslation)
runner.run(suite)

test_full_model_and_parts (__main__.TestVaDETranslation)
Test the forward pass and helper methods of the VaDEPT class. ... 

Transferring weights for all VaDE components...
  -> Transferring Encoder weights...
  -> Transferring GMM Latent weights...
  -> Transferring Decoder weights...
Weight transfer complete.

--- Testing Self-Contained VaDEPT Class Translation ---


`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
ERROR

ERROR: test_full_model_and_parts (__main__.TestVaDETranslation)
Test the forward pass and helper methods of the VaDEPT class.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_16880\2925459415.py", line 130, in test_full_model_and_parts
    pt_rec_dist, _, _, _ = self.pt_vade(x_pt, a_pt)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\torch\nn\modules\module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\torch\nn\modules\module.py", line 1784, in _call_impl
    return forward_call(*args, **kwargs)
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_16880\3492692028.py", line 98, in forward
    latent, categorical, _, _, kmeans

<unittest.runner.TextTestResult run=1 errors=1 failures=0>

# Full Model

In [28]:
import unittest
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense
import tensorflow_probability as tfp
from tensorflow.keras.optimizers import Nadam
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from typing import List, Tuple, Dict, Callable
import time
import deepof.model_utils
from spektral.layers import CensNetConv
from deepof.model_utils import ClusterControl, compute_kmeans_loss, ProbabilisticDecoder
import deepof.models
from deepof.models import get_recurrent_encoder, get_recurrent_decoder, GaussianMixtureLatent, get_TCN_encoder, get_TCN_decoder, get_transformer_encoder, get_transformer_decoder
from deepof.clustering.models_new import RecurrentEncoderPT, RecurrentDecoderPT, GaussianMixtureLatentPT
from tensorflow.keras.layers import (
    GRU,
    Bidirectional,
    Dense,
    LayerNormalization,
    RepeatVector,
    TimeDistributed,
)

from deepof.data_loading import get_dt

tfd = tfp.distributions

In [29]:
class VaDE(tf.keras.models.Model):
    """Gaussian Mixture Variational Autoencoder for pose motif elucidation."""

    def __init__(
        self,
        input_shape: tuple,
        edge_feature_shape: tuple,
        adjacency_matrix: np.ndarray = None,
        latent_dim: int = 8,
        use_gnn: bool = True,
        n_components: int = 15,
        batch_size: int = 64,
        kl_annealing_mode: str = "linear",
        kl_warmup_epochs: int = 15,
        montecarlo_kl: int = 100,
        kmeans_loss: float = 1.0,
        reg_cat_clusters: float = 1.0,
        reg_cluster_variance: bool = False,
        encoder_type: str = "recurrent",
        interaction_regularization: float = 0.0,
        **kwargs,
    ):
        """Init a VaDE model.

        Args:
            input_shape (tuple): Shape of the input to the full model.
            edge_feature_shape (tuple): shape of the edge feature matrix used for graph representations.
            adjacency_matrix (np.ndarray): adjacency matrix of the connectivity graph to use.
            batch_size (int): Batch size for training.
            latent_dim (int): Dimensionality of the latent space.
            use_gnn (bool): If True, the encoder uses a graph representation of the input, with coordinates and speeds as node attributes, and distances as edge attributes. If False, a regular 3D tensor is used as input.
            kl_annealing_mode (str): Annealing mode for KL annealing. Can be one of 'linear' and 'sigmoid'.
            kl_warmup_epochs (int): Number of epochs to warmup KL annealing.
            montecarlo_kl (int): Number of Monte Carlo samples for KL divergence.
            n_components (int): Number of mixture components in the latent space.
            kmeans_loss (float): weight of the gram matrix regularization loss.
            reg_cat_clusters (bool): whether to use the penalized uneven cluster membership in the latent space, by minimizing the KL divergence between cluster membership and a uniform categorical distribution.
            reg_cluster_variance (bool): whether to penalize uneven cluster variances in the latent space.
            encoder_type (str): type of encoder to use. Can be set to "recurrent" (default), "TCN", or "transformer".
            interaction_regularization (float): Regularization parameter for the interaction features.
            **kwargs: Additional keyword arguments.

        """
        super(VaDE, self).__init__(**kwargs)
        self.seq_shape = input_shape
        self.edge_feature_shape = edge_feature_shape
        self.adjacency_matrix = adjacency_matrix
        self.batch_size = batch_size
        self.latent_dim = latent_dim
        self.use_gnn = use_gnn
        self.kl_annealing_mode = kl_annealing_mode
        self.kl_warmup = kl_warmup_epochs
        self.mc_kl = montecarlo_kl
        self.n_components = n_components
        self.optimizer = Nadam(learning_rate=1e-3, clipvalue=0.75)
        self.kmeans = kmeans_loss
        self.reg_cat_clusters = reg_cat_clusters
        self.reg_cluster_variance = reg_cluster_variance
        self.encoder_type = encoder_type
        self.interaction_regularization = interaction_regularization

        # Define VaDE model
        self.encoder, self.decoder, self.grouper, self.vade = deepof.models.get_vade(
            input_shape=self.seq_shape,
            edge_feature_shape=self.edge_feature_shape,
            adjacency_matrix=self.adjacency_matrix,
            n_components=self.n_components,
            latent_dim=self.latent_dim,
            use_gnn=use_gnn,
            batch_size=self.batch_size,
            kl_warmup=self.kl_warmup,
            kl_annealing_mode=self.kl_annealing_mode,
            mc_kl=self.mc_kl,
            kmeans_loss=self.kmeans,
            reg_cluster_variance=self.reg_cluster_variance,
            encoder_type=self.encoder_type,
            interaction_regularization=self.interaction_regularization,
        )

        # Propagate the optimizer to all relevant sub-models, to enable metric annealing
        self.vade.optimizer = self.optimizer
        self.vade.get_layer("gaussian_mixture_latent").optimizer = self.optimizer

        # Define metrics to track

        # Track all loss function components
        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.val_total_loss_tracker = tf.keras.metrics.Mean(name="val_total_loss")

        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(
            name="reconstruction_loss"
        )
        self.val_reconstruction_loss_tracker = tf.keras.metrics.Mean(
            name="val_reconstruction_loss"
        )

        if self.reg_cat_clusters:
            self.cat_cluster_loss_tracker = tf.keras.metrics.Mean(
                name="cat_cluster_loss"
            )
            self.val_cat_cluster_loss_tracker = tf.keras.metrics.Mean(
                name="val_cat_cluster_loss"
            )

    @property
    def metrics(self):  # pragma: no cover
        """Initializes tracked metrics of VaDE model."""
        metrics = [
            self.total_loss_tracker,
            self.val_total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.val_reconstruction_loss_tracker,
        ]

        if self.reg_cat_clusters:
            metrics += [
                self.cat_cluster_loss_tracker,
                self.val_cat_cluster_loss_tracker,
            ]

        return metrics

    @property
    def get_gmm_params(self):
        """Return the GMM parameters of the model."""
        # Get GMM parameters
        return {
            "means": self.grouper.get_layer("gaussian_mixture_latent").c_mu,
            "sigmas": tf.math.exp(
                self.grouper.get_layer("gaussian_mixture_latent").log_c_sigma
            ),
            "weights": tf.math.softmax(
                self.grouper.get_layer("gaussian_mixture_latent").prior
            ),
        }

    def set_pretrain_mode(self, switch):
        """Set the pretrain mode of the model."""
        self.grouper.get_layer("gaussian_mixture_latent").pretrain.assign(switch)

    def pretrain(
        self,
        data,
        embed_x,
        embed_a,
        epochs=10,
        samples=10000,
        gmm_initialize=True,
        **kwargs,
    ):
        """Run a GMM directed pretraining of the encoder, to minimize the likelihood of getting stuck in a local minimum."""
        # Turn on pretrain mode
        self.set_pretrain_mode(1.0)

        # pre-train
        self.fit(
            data,
            epochs=epochs,
            **kwargs,
        )


        # Turn off pretrain mode
        self.set_pretrain_mode(0.0)

        if gmm_initialize:

            with tf.device("CPU"):
                # Get embedding samples
                em_x=get_dt(embed_x, 'embed_x')
                em_a=get_dt(embed_a, 'embed_a')

                emb_idx = np.random.choice(range(em_x.shape[0]), samples)

                # map to latent
                z = self.encoder([em_x[emb_idx], em_a[emb_idx]])
                
                del em_x
                del em_a
                del emb_idx

                # fit GMM
                gmm = deepof.models.GaussianMixture(
                    n_components=self.n_components,
                    covariance_type="diag",
                    reg_covar=1e-04,
                    **kwargs,
                ).fit(z)
                # get GMM parameters
                mu = gmm.means_
                sigma2 = gmm.covariances_

            # initialize mixture components
            self.grouper.get_layer("gaussian_mixture_latent").c_mu.assign(
                tf.convert_to_tensor(value=mu, dtype=tf.float32)
            )
            self.grouper.get_layer("gaussian_mixture_latent").log_c_sigma.assign(
                tf.math.log(
                    tf.math.sqrt(tf.convert_to_tensor(value=sigma2, dtype=tf.float32))
                )
            )

    @tf.function
    def call(self, inputs, **kwargs):
        """Call the VaDE model."""
        return self.vade(inputs, **kwargs)

    def train_step(self, data):  # pragma: no cover
        """Perform a training step."""
        # Unpack data, repacking labels into a generator
        x, a, y = data
        if not isinstance(y, tuple):
            y = [y]
        y = (labels for labels in y)

        with tf.GradientTape() as tape:

            # Get outputs from the full model
            outputs = self.vade([x, a], training=True)

            # Get rid of the attention scores that the transformer decoder outputs
            if self.encoder_type == "transformer":
                outputs = outputs[0]

            if isinstance(outputs, list):
                reconstructions = outputs[0]
            else:
                reconstructions = outputs

            # Regularize embeddings
            # groups = self.grouper(x, training=True)

            # Compute losses
            seq_inputs = next(y)
            total_loss = sum(self.vade.losses)

            # Add a regularization term to the soft_counts, to prevent the embedding layer from
            # collapsing into a few clusters.
            if self.reg_cat_clusters:

                soft_counts = self.grouper([x, a], training=True)
                soft_counts_regulrization = (
                    self.reg_cat_clusters
                    * deepof.model_utils.cluster_frequencies_regularizer(
                        soft_counts=soft_counts, k=self.n_components
                    )
                )
                total_loss += soft_counts_regulrization

            # Compute reconstruction loss
            reconstruction_loss = -tf.reduce_mean(reconstructions.log_prob(seq_inputs))
            total_loss += reconstruction_loss

        # Backpropagation
        grads = tape.gradient(total_loss, self.vade.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.vade.trainable_variables))

        # Track losses
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)

        # Log results (coupled with TensorBoard)
        log_dict = {
            "total_loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
        }

        if self.reg_cat_clusters:
            self.cat_cluster_loss_tracker.update_state(soft_counts_regulrization)
            log_dict["cat_cluster_loss"] = self.cat_cluster_loss_tracker.result()

        # Log to TensorBoard, both explicitly and implicitly (within model) tracked metrics
        return {**log_dict, **{met.name: met.result() for met in self.vade.metrics}}

    # noinspection PyUnboundLocalVariable
    @tf.function
    def test_step(self, data):  # pragma: no cover
        """Performs a test step."""
        # Unpack data, repacking labels into a generator
        x, a, y = data
        if not isinstance(y, tuple):
            y = [y]
        y = (labels for labels in y)

        # Get outputs from the full model
        outputs = self.vade([x, a], training=False)

        # Get rid of the attention scores that the transformer decoder outputs
        if self.encoder_type == "transformer":
            outputs = outputs[0]

        if isinstance(outputs, list):
            reconstructions = outputs[0]
        else:
            reconstructions = outputs

        # Compute losses
        seq_inputs = next(y)
        total_loss = sum(self.vade.losses)

        # Add a regularization term to the soft_counts, to prevent the embedding layer from
        # collapsing into a few clusters.
        if self.reg_cat_clusters:
            soft_counts = self.grouper([x, a], training=False)
            soft_counts_regulrization = (
                self.reg_cat_clusters
                * deepof.model_utils.cluster_frequencies_regularizer(
                    soft_counts=soft_counts, k=self.n_components
                )
            )
            total_loss += soft_counts_regulrization

        # Compute reconstruction loss
        reconstruction_loss = -tf.reduce_mean(reconstructions.log_prob(seq_inputs))
        total_loss += reconstruction_loss

        # Track losses
        self.val_total_loss_tracker.update_state(total_loss)
        self.val_reconstruction_loss_tracker.update_state(reconstruction_loss)

        # Log results (coupled with TensorBoard)
        log_dict = {
            "total_loss": self.val_total_loss_tracker.result(),
            "reconstruction_loss": self.val_reconstruction_loss_tracker.result(),
        }

        if self.reg_cat_clusters:
            self.val_cat_cluster_loss_tracker.update_state(soft_counts_regulrization)
            log_dict["cat_cluster_loss"] = self.val_cat_cluster_loss_tracker.result()

        return {**log_dict, **{met.name: met.result() for met in self.vade.metrics}}


In [30]:
class VaDEPT(nn.Module):
    """
    A self-contained PyTorch implementation of the VaDE model.

    This class encapsulates the entire VaDE architecture, including the encoder,
    the Gaussian mixture latent space, and the decoder. It is instantiated with
    all necessary configuration parameters, building its sub-modules internally.
    This provides a clean, single-object interface for the model.
    """
    def __init__(
        self,
        input_shape: tuple,
        edge_feature_shape: tuple,
        adjacency_matrix: np.ndarray,
        latent_dim: int,
        n_components: int,
        use_gnn: bool = True,
        kmeans_loss: float = 1.0,
        interaction_regularization: float = 0.0,
    ):
        """
        Initializes and builds the VaDE model and its components.

        Args:
            input_shape (tuple): Shape of the input node features (Time, Nodes, Features_per_node).
            edge_feature_shape (tuple): Shape of the edge features (Time, Edges, Features_per_edge).
            adjacency_matrix (np.ndarray): Adjacency matrix of the connectivity graph.
            latent_dim (int): Dimensionality of the latent space.
            n_components (int): Number of components in the Gaussian mixture.
            use_gnn (bool): If True, use the GNN-based encoder.
            kmeans_loss (float): Weight of the k-means style loss in the latent space.
            interaction_regularization (float): Regularization for GNN interaction features.
        """
        super().__init__()
        
        # Store key dimensions for internal use (e.g., reshaping in forward pass)
        time_steps, n_nodes, n_features_per_node = input_shape
        self.input_n_nodes = n_nodes
        self.input_n_features_per_node = n_features_per_node

        # 1. Instantiate Encoder
        self.encoder = RecurrentEncoderPT(
            input_shape=input_shape,
            edge_feature_shape=edge_feature_shape,
            adjacency_matrix=adjacency_matrix,
            latent_dim=latent_dim,
            use_gnn=use_gnn,
            interaction_regularization=interaction_regularization,
        )

        # 2. Instantiate Latent Space
        self.latent_space = GaussianMixtureLatentPT(
            input_dim=latent_dim,
            n_components=n_components,
            latent_dim=latent_dim,
            kmeans=kmeans_loss,
        )

        # 3. Instantiate Decoder
        decoder_output_features = n_nodes * n_features_per_node
        self.decoder = RecurrentDecoderPT(
            output_shape=(time_steps, decoder_output_features),
            latent_dim=latent_dim,
        )

    def forward(
        self, x: torch.Tensor, a: torch.Tensor
    ) -> Tuple[torch.distributions.Distribution, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Defines the full forward pass for the VaDE model (training and evaluation).

        Args:
            x (torch.Tensor): Input node features tensor (B, T, N, F_node).
            a (torch.Tensor): Input edge features tensor (B, T, E, F_edge).

        Returns:
            A tuple containing:
            - reconstruction_dist (torch.distributions.Distribution): The output distribution from the decoder.
            - latent (torch.Tensor): The sampled latent representation from the GMM space.
            - categorical (torch.Tensor): The cluster probabilities (soft assignments).
            - kmeans_loss (torch.Tensor): The k-means regularization loss from the latent space.
        """
        # 1. Encode the input to get the pre-latent representation
        encoder_output = self.encoder(x, a)
        
        # 2. Pass through GMM latent space
        latent, categorical, _, _, kmeans_loss, _ = self.latent_space(encoder_output)
        
        # 3. Decode the latent sample back to the original data space
        # Reshape x to (B, T, N*F) for the decoder's masking logic
        B, T, _, _ = x.shape
        x_for_decoder = x.view(B, T, self.input_n_nodes * self.input_n_features_per_node)
        
        reconstruction_dist = self.decoder(latent, x_for_decoder)
        
        return reconstruction_dist, latent, categorical, kmeans_loss
    

    def get_gmm_params(self) -> dict:
        """Returns the GMM parameters from the latent space."""
        # This is the PyTorch equivalent of the TF property
        with torch.no_grad():
            means = self.latent_space.gmm_means
            # The latent space stores log-variances, convert to std-dev
            stds = torch.exp(0.5 * self.latent_space.gmm_log_vars)
            # Prior is already softmaxed if needed, or just probabilities
            weights = self.latent_space.prior
        return {"means": means, "stds": stds, "weights": weights}


    def set_pretrain_mode(self, pretrain_on: bool):
        """Sets the pretrain flag in the latent space."""
        # In TF it was a float (0.0/1.0), here a boolean is cleaner
        self.latent_space.pretrain.fill_(1.0 if pretrain_on else 0.0)


    def initialize_gmm_from_data(self, data_loader, n_samples=10000):
        """
        Runs the autoencoder part of the model over the data to get embeddings,
        then fits a scikit-learn GMM to initialize the latent space.
        """
        print("Initializing GMM from data embeddings...")
        self.eval() # Set model to evaluation mode
        
        # 1. Gather embeddings from the autoencoder
        all_embeddings = []
        samples_gathered = 0
        with torch.no_grad():
            for x, a in data_loader:
                # Assuming x,a are on the correct device
                embeddings = self.encoder(x, a)
                all_embeddings.append(embeddings.cpu())
                samples_gathered += embeddings.size(0)
                if samples_gathered >= n_samples:
                    break
        
        all_embeddings = torch.cat(all_embeddings, dim=0).numpy()
        if all_embeddings.shape[0] > n_samples:
            all_embeddings = all_embeddings[:n_samples]

        # 2. Fit a scikit-learn GMM
        from sklearn.mixture import GaussianMixture
        print(f"Fitting scikit-learn GMM on {all_embeddings.shape[0]} samples...")
        gmm = GaussianMixture(
            n_components=self.latent_space.n_components,
            covariance_type="diag",
            reg_covar=1e-04,
        ).fit(all_embeddings)

        # 3. Assign the learned parameters to the model's latent space
        print("Assigning learned GMM parameters to the model.")
        self.latent_space.gmm_means.data = torch.from_numpy(gmm.means_).float()
        # Convert covariance (variance) to log-variance for the model
        self.latent_space.gmm_log_vars.data = torch.from_numpy(np.log(gmm.covariances_)).float()

In [31]:
def transfer_recurrent_block_weights(tf_model, pt_model):
    """Transfers weights for the full recurrent block with GRU gate permutation."""
    conv_td, _, gru1_td, norm1, gru2_td, norm2 = tf_model.layers[1:]


    def permute_gru_weights(keras_weights):
        W_ih, W_hh, B = keras_weights
        W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
        W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)
        W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
        W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)
        B_ih, B_hh = B
        B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
        B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)
        B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
        B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])
        return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt

    pt_model.conv1d.weight.data = torch.from_numpy(conv_td.layer.get_weights()[0]).permute(2, 1, 0)
    
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(gru1_td.layer.forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1); pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(gru1_td.layer.backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1); pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)

    pt_model.norm1.weight.data = torch.from_numpy(norm1.get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm1.get_weights()[1])

    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(gru2_td.layer.forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2); pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(gru2_td.layer.backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2); pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    
    pt_model.norm2.weight.data = torch.from_numpy(norm2.get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm2.get_weights()[1])

    
def transfer_censnet_weights(tf_layer, pt_layer):
    """
    Transfers all six weights from a Spektral CensNetConv layer to the
    corresponding CensNetConvPT layer.
    """
    # Get all weights from the TensorFlow layer. The order is determined by
    # the layer's build order in Spektral's source code.
    tf_weights = tf_layer.get_weights()

    # Unpack all six weights.
    # Order: kernel_node, bias_node, kernel_edge, bias_edge, projector_node, projector_edge
    kn_tf, bn_tf, ke_tf, be_tf, pn_tf, pe_tf = tf_weights

    # Build weights on first pass
    if pt_layer.node_kernel is None:
        # Move parameters to the same device as input tensors
        pt_layer._build(kn_tf.T.shape, bn_tf.T.shape)
        #pt_layer.to(kn_tf.device)

    # 1. & 2. Transfer Node Kernel and Bias
    # Keras Dense kernel is (in_features, out_features)
    pt_layer.node_kernel.data = torch.from_numpy(kn_tf)
    pt_layer.edge_kernel.data = torch.from_numpy(bn_tf)

    # 3. & 4. Transfer Edge Kernel and Bias
    # Same transposition logic applies.
    pt_layer.node_weights.data = torch.from_numpy(ke_tf)
    pt_layer.edge_weights.data = torch.from_numpy(be_tf)

    # 5. Transfer Node Projector Weights (P_n)
    # These are [in_features, 1], which matches, so no transpose needed.
    pt_layer.node_bias.data = torch.from_numpy(pn_tf)

    # 6. Transfer Edge Projector Weights (P_e)
    # These are [in_features, 1], which matches, so no transpose needed.
    pt_layer.edge_bias.data = torch.from_numpy(pe_tf)
    

def transfer_recurrent_encoder_weights(tf_model, pt_model):
    """
    Transfers weights for the full recurrent encoder, finding layers
    by their default names and types to avoid modifying original code.
    """
    # The final dense layer is consistently the last one in the model's layer list.
    final_dense_tf = tf_model.layers[-1]
    final_dense_pt = pt_model.final_dense
    w, b = final_dense_tf.get_weights()
    final_dense_pt.weight.data = torch.from_numpy(w.T)
    final_dense_pt.bias.data = torch.from_numpy(b)

    if pt_model.use_gnn:
        # Keras automatically names nested models 'model', 'model_1', etc., by order of creation.
        # Node recurrent block is created first.
        node_recurrent_model = tf_model.get_layer("model")
        # Edge recurrent block is created second.
        edge_recurrent_model = tf_model.get_layer("model_1")
        # Find the CensNetConv layer by its class type.
        gnn_layer = next(l for l in tf_model.layers if isinstance(l, CensNetConv))

        transfer_recurrent_block_weights(node_recurrent_model, pt_model.node_recurrent_block)
        transfer_recurrent_block_weights(edge_recurrent_model, pt_model.edge_recurrent_block)
        transfer_censnet_weights(gnn_layer, pt_model.spatial_gnn_block)
    else: # Not using GNN
        # There is only one nested model, which Keras names 'model'.
        recurrent_model = tf_model.get_layer("model")
        transfer_recurrent_block_weights(recurrent_model, pt_model.recurrent_block)

In [32]:
def transfer_gmm_weights(tf_model, pt_model: GaussianMixtureLatentPT):
    """
    Transfers weights from the final TF model to the refactored PT model,
    using the updated attribute names.
    """
    # --- Transfer GMM component parameters ---
    # OLD: pt_model.c_mu
    pt_model.gmm_means.data = torch.from_numpy(tf_model.c_mu.numpy())
    # OLD: pt_model.log_c_sigma
    pt_model.gmm_log_vars.data = torch.from_numpy(tf_model.log_c_sigma.numpy())

    # --- Transfer Encoder layer parameters ---
    tf_mean_weights = tf_model.z_gauss_mean.get_weights()
    # OLD: pt_model.z_gauss_mean
    pt_model.encoder_mean.weight.data = torch.from_numpy(tf_mean_weights[0].T)
    pt_model.encoder_mean.bias.data = torch.from_numpy(tf_mean_weights[1])
    
    tf_var_weights = tf_model.z_gauss_var.get_weights()
    # OLD: pt_model.z_gauss_var
    pt_model.encoder_log_var.weight.data = torch.from_numpy(tf_var_weights[0].T)
    pt_model.encoder_log_var.bias.data = torch.from_numpy(tf_var_weights[1])

In [33]:
# Helper function from the provided example to handle gate order differences
def permute_gru_weights(keras_weights):
    """Permutes GRU weights from Keras (z, r, n) to PyTorch (r, z, n) format."""
    W_ih, W_hh, B = keras_weights
    # Keras gate order: z, r, n (update, reset, new/candidate)
    W_ih_z, W_ih_r, W_ih_n = np.split(W_ih, 3, axis=1)
    W_hh_z, W_hh_r, W_hh_n = np.split(W_hh, 3, axis=1)

    # PyTorch gate order: r, z, n (reset, update, new/candidate)
    W_ih_pt = np.concatenate([W_ih_r, W_ih_z, W_ih_n], axis=1)
    W_hh_pt = np.concatenate([W_hh_r, W_hh_z, W_hh_n], axis=1)

    # Keras has two bias vectors (input-hidden and recurrent), which are concatenated in B
    B_ih, B_hh = B
    B_ih_z, B_ih_r, B_ih_n = np.split(B_ih, 3)
    B_hh_z, B_hh_r, B_hh_n = np.split(B_hh, 3)

    B_ih_pt = np.concatenate([B_ih_r, B_ih_z, B_ih_n])
    B_hh_pt = np.concatenate([B_hh_r, B_hh_z, B_hh_n])

    return W_ih_pt.T, W_hh_pt.T, B_ih_pt, B_hh_pt
    
def transfer_recurrent_decoder_weights(tf_model, pt_model):
    """
    Transfers weights for the full recurrent decoder model.
    """
    # Find layers by type to avoid index issues
    bidi_layers = [l for l in tf_model.layers if isinstance(l, Bidirectional)]
    norm_layers = [l for l in tf_model.layers if isinstance(l, LayerNormalization)]
    conv_layers = [l for l in tf_model.layers if isinstance(l, tf.keras.layers.Conv1D)]
    prob_dec_layer = next(l for l in tf_model.layers if isinstance(l, deepof.model_utils.ProbabilisticDecoder))

    # --- GRU 1 and Norm 1 ---
    W_ih_f1, W_hh_f1, B_ih_f1, B_hh_f1 = permute_gru_weights(bidi_layers[0].forward_layer.get_weights())
    pt_model.gru1.weight_ih_l0.data = torch.from_numpy(W_ih_f1); pt_model.gru1.weight_hh_l0.data = torch.from_numpy(W_hh_f1)
    pt_model.gru1.bias_ih_l0.data = torch.from_numpy(B_ih_f1); pt_model.gru1.bias_hh_l0.data = torch.from_numpy(B_hh_f1)
    W_ih_b1, W_hh_b1, B_ih_b1, B_hh_b1 = permute_gru_weights(bidi_layers[0].backward_layer.get_weights())
    pt_model.gru1.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b1); pt_model.gru1.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b1)
    pt_model.gru1.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b1); pt_model.gru1.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b1)
    pt_model.norm1.weight.data = torch.from_numpy(norm_layers[0].get_weights()[0]); pt_model.norm1.bias.data = torch.from_numpy(norm_layers[0].get_weights()[1])

    # --- GRU 2 and Norm 2 ---
    W_ih_f2, W_hh_f2, B_ih_f2, B_hh_f2 = permute_gru_weights(bidi_layers[1].forward_layer.get_weights())
    pt_model.gru2.weight_ih_l0.data = torch.from_numpy(W_ih_f2); pt_model.gru2.weight_hh_l0.data = torch.from_numpy(W_hh_f2)
    pt_model.gru2.bias_ih_l0.data = torch.from_numpy(B_ih_f2); pt_model.gru2.bias_hh_l0.data = torch.from_numpy(B_hh_f2)
    W_ih_b2, W_hh_b2, B_ih_b2, B_hh_b2 = permute_gru_weights(bidi_layers[1].backward_layer.get_weights())
    pt_model.gru2.weight_ih_l0_reverse.data = torch.from_numpy(W_ih_b2); pt_model.gru2.weight_hh_l0_reverse.data = torch.from_numpy(W_hh_b2)
    pt_model.gru2.bias_ih_l0_reverse.data = torch.from_numpy(B_ih_b2); pt_model.gru2.bias_hh_l0_reverse.data = torch.from_numpy(B_hh_b2)
    pt_model.norm2.weight.data = torch.from_numpy(norm_layers[1].get_weights()[0]); pt_model.norm2.bias.data = torch.from_numpy(norm_layers[1].get_weights()[1])

    # --- Conv1D and Norm 3 ---
    # TF Conv1D weights: (kernel_w, kernel_h, in_c, out_c) -> (5, 1, 4*ld, 2*ld)
    # PT Conv1d weights: (out_c, in_c, kernel_w)
    conv_weights_tf = conv_layers[0].get_weights()[0]
    pt_model.conv1d.weight.data = torch.from_numpy(conv_weights_tf).squeeze(1).permute(2, 1, 0)
    pt_model.norm3.weight.data = torch.from_numpy(norm_layers[2].get_weights()[0]); pt_model.norm3.bias.data = torch.from_numpy(norm_layers[2].get_weights()[1])

    # --- Probabilistic Decoder ---
    # TF Dense weights: (in_features, out_features)
    # PT Linear weights: (out_features, in_features)
    prob_dec_weights, prob_dec_bias = prob_dec_layer.time_distributer.get_weights()
    pt_model.prob_decoder.loc_projection.weight.data = torch.from_numpy(prob_dec_weights.T)
    pt_model.prob_decoder.loc_projection.bias.data = torch.from_numpy(prob_dec_bias)

In [34]:
def transfer_vade_class_weights(tf_vade_model, tf_decoder_model, pt_vade_model: VaDEPT):
    """
    Transfers weights from a full TensorFlow VaDE model to the self-contained PyTorch VaDEPT class.
    """
    print("Transferring weights for all VaDE components...")
    
    # 1. Get the inner Keras models/layers by name from the complete TF model
    tf_encoder_inner = tf_vade_model.get_layer("recurrent_encoder")
    tf_latent_layer = tf_vade_model.get_layer("gaussian_mixture_latent")
    
    # 2. Use the specialized weight transfer functions, passing the PT sub-modules
    print("  -> Transferring Encoder weights...")
    transfer_recurrent_encoder_weights(tf_encoder_inner, pt_vade_model.encoder)
    print("  -> Transferring GMM Latent weights...")
    transfer_gmm_weights(tf_latent_layer, pt_vade_model.latent_space)
    print("  -> Transferring Decoder weights...")
    transfer_recurrent_decoder_weights(tf_decoder_model, pt_vade_model.decoder)
    
    print("Weight transfer complete.")


class TestVaDETranslation(unittest.TestCase):
    def setUp(self):
        """Set up parameters, models, and data for testing."""
        tf.keras.backend.clear_session()
        tf.keras.backend.set_epsilon(1e-3)

        # --- 1. Define Fundamental Dimensions ---
        self.batch_size = 128
        self.window_length = 25
        self.num_nodes = 11
        # In your example, total features (n=33) / num_nodes (11) = 3
        self.features_per_node = 3
        self.num_edges = 11
        self.features_per_edge = 1 # Assuming 1 feature per edge

        # --- 2. Define Model Parameters ---
        self.latent_dim = 6
        self.n_components = 10
        self.kmeans_loss = 1.0
        self.use_gnn = False

        # --- 3. Create Adjacency Matrix ---
        m = np.zeros((self.num_nodes, self.num_nodes))
        ui = np.triu_indices(self.num_nodes)
        num_possible_edges = len(ui[0])
        c = np.random.choice(num_possible_edges, min(self.num_edges, num_possible_edges), replace=False)
        m[ui[0][c], ui[1][c]] = 1
        m += m.T # Make symmetric
        self.adj_matrix = m

        # --- 4. Create Framework-Specific Shapes for Model Instantiation ---
        
        # TensorFlow expects (batch, time, total_features)
        self.input_shape_tf = (self.batch_size, self.window_length, self.num_nodes * self.features_per_node)
        self.edge_feature_shape_tf = (self.batch_size, self.window_length, self.num_edges * self.features_per_edge)
        
        # PyTorch VaDEPT expects (time, nodes, features_per_node) for a SINGLE sample
        self.input_shape_pt = (self.window_length, self.num_nodes, self.features_per_node)
        self.edge_feature_shape_pt = (self.window_length, self.num_edges, self.features_per_edge)

        # --- 5. Instantiate Models ---
        tf_model = VaDE(
            input_shape=self.input_shape_tf,
            edge_feature_shape=self.edge_feature_shape_tf,
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=self.use_gnn,
            n_components=self.n_components,
            batch_size=self.batch_size,
            kmeans_loss=self.kmeans_loss
        )
        self.tf_decoder = tf_model.decoder
        self.tf_vade = tf_model.vade
        self.tf_embedding = tf_model.encoder
        self.tf_grouper = tf_model.grouper
        
        self.pt_vade = VaDEPT(
            input_shape=self.input_shape_pt,
            edge_feature_shape=self.edge_feature_shape_pt,
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            n_components=self.n_components,
            use_gnn=self.use_gnn,
            kmeans_loss=self.kmeans_loss
        )
        self.pt_vade.eval()

        # --- 6. Prepare Data Tensors for Each Framework ---
        np.random.seed(42)
        # The "canonical" data is 4D, as expected by the new PyTorch models
        self.x_np_4d = np.random.rand(
            self.batch_size, self.window_length, self.num_nodes, self.features_per_node
        ).astype(np.float32)
        self.a_np_4d = np.random.rand(
            self.batch_size, self.window_length, self.num_edges, self.features_per_edge
        ).astype(np.float32)

        # Create the 3D version for the legacy TensorFlow model by reshaping
        self.x_np_tf = self.x_np_4d.reshape(self.input_shape_tf)
        self.a_np_tf = self.a_np_4d.reshape(self.edge_feature_shape_tf)
        
        # --- 7. Transfer Weights ---
        transfer_vade_class_weights(self.tf_vade, self.tf_decoder, self.pt_vade)

    def test_full_model_and_parts(self):
        """Test the forward pass and helper methods of the VaDEPT class."""
        print("\n--- Testing Self-Contained VaDEPT Class Translation ---")
        
        # --- TensorFlow Execution (with its required 3D input) ---
        tf_start = time.time()
        tf_rec_dist = self.tf_vade([self.x_np_tf, self.a_np_tf], training=False)
        tf_rec_mean = tf_rec_dist.mean().numpy()
        tf_lat_out = self.tf_embedding([self.x_np_tf, self.a_np_tf], training=False).numpy()
        tf_cat_out = self.tf_grouper([self.x_np_tf, self.a_np_tf], training=False).numpy()
        tf_end = time.time()
        
        # --- PyTorch Execution (with its required 4D input) ---
        x_pt = torch.from_numpy(self.x_np_4d)
        a_pt = torch.from_numpy(self.a_np_4d)
        
        pt_start = time.time()
        with torch.no_grad():
            pt_rec_dist, _, _, _ = self.pt_vade(x_pt, a_pt)
            pt_rec_mean = pt_rec_dist.mean.numpy() 
            pt_lat_out = self.pt_vade.embed(x_pt, a_pt).numpy()
            pt_cat_out = self.pt_vade.group(x_pt, a_pt).numpy()
        pt_end = time.time()

        print(f"TensorFlow execution time: {tf_end - tf_start:.6f}s")
        print(f"PyTorch execution time: {pt_end - pt_start:.6f}s")
        
        # --- Assertions ---
        print("\nComparing latent space embeddings (from .embed() vs 'embedding' model)...")
        # Both outputs should be (batch_size, latent_dim), so (128, 6)
        np.testing.assert_allclose(tf_lat_out, pt_lat_out, rtol=1e-5, atol=1e-4)
        print("✅ Latent embeddings match.")

        print("Comparing categorical probabilities (from .group() vs 'grouper' model)...")
        # Both outputs should be (batch_size, n_components), so (128, 10)
        np.testing.assert_allclose(tf_cat_out, pt_cat_out, rtol=1e-5, atol=1e-5)
        print("✅ Categorical probabilities match.")
        
        print("Comparing final reconstruction means (from forward() vs 'vade' model)...")
        # Both outputs should be (batch_size, time_steps, total_features), so (128, 25, 33)
        np.testing.assert_allclose(tf_rec_mean, pt_rec_mean, rtol=1e-5, atol=1e-4)
        print("✅ Reconstructions match.")

        print("\n✅ Self-contained VaDEPT class translation test PASSED!")

# To run the test
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestVaDETranslation)
runner.run(suite)

test_full_model_and_parts (__main__.TestVaDETranslation)
Test the forward pass and helper methods of the VaDEPT class. ... 

Transferring weights for all VaDE components...
  -> Transferring Encoder weights...
  -> Transferring GMM Latent weights...
  -> Transferring Decoder weights...
Weight transfer complete.

--- Testing Self-Contained VaDEPT Class Translation ---


`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
ERROR

ERROR: test_full_model_and_parts (__main__.TestVaDETranslation)
Test the forward pass and helper methods of the VaDEPT class.
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_16880\1105766262.py", line 124, in test_full_model_and_parts
    pt_rec_dist, _, _, _ = self.pt_vade(x_pt, a_pt)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\torch\nn\modules\module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "c:\Users\Petron\Desktop\Python_Projects\Deepof\dof\lib\site-packages\torch\nn\modules\module.py", line 1784, in _call_impl
    return forward_call(*args, **kwargs)
  File "C:\Users\Petron\AppData\Local\Temp\ipykernel_16880\2086274447.py", line 87, in forward
    latent, categorical, _, _, kmeans

<unittest.runner.TextTestResult run=1 errors=1 failures=0>

# TCN encoder test

In [35]:
from typing import Iterable, Tuple, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from spektral.layers import CensNetConv
from deepof.clustering.censNetConv_pt import CensNetConvPT
from tensorflow.keras.layers import Input, TimeDistributed, Bidirectional, GRU, LayerNormalization, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    GRU,
    Bidirectional,
    Dense,
    LayerNormalization,
    RepeatVector,
    TimeDistributed,
)
import numpy as np
import torch
import torch.nn as nn
import tensorflow as tf
import tcn


In [36]:
def get_TCN_encoder(
    input_shape: tuple,
    edge_feature_shape: tuple,
    adjacency_matrix: np.ndarray,
    latent_dim: int,
    use_gnn: bool = True,
    conv_filters: int = 32,
    kernel_size: int = 4,
    conv_stacks: int = 2,
    conv_dilations: tuple = (1, 2, 4, 8),
    padding: str = "causal",
    use_skip_connections: bool = True,
    dropout_rate: int = 0,
    activation: str = "relu",
    interaction_regularization: float = 0.0,
):
    """Return a Temporal Convolutional Network (TCN) encoder.

    Builds a neural network that can be used to encode motion tracking instances into a
    vector. Each layer contains a residual block with a convolutional layer and a skip connection. See the following
    paper for more details: https://arxiv.org/pdf/1803.01271.pdf

    Args:
        input_shape: shape of the input data
        edge_feature_shape (tuple): shape of the adjacency matrix to use in the graph attention layers. Should be time x edges x features.
        adjacency_matrix (np.ndarray): adjacency matrix for the mice connectivity graph. Shape should be nodes x nodes.
        latent_dim: dimensionality of the latent space
        use_gnn (bool): If True, the encoder uses a graph representation of the input, with coordinates and speeds as node attributes, and distances as edge attributes. If False, a regular 3D tensor is used as input.
        conv_filters: number of filters in the TCN layers
        kernel_size: size of the convolutional kernels
        conv_stacks: number of TCN layers
        conv_dilations: list of dilation factors for each TCN layer
        padding: padding mode for the TCN layers
        use_skip_connections: whether to use skip connections between TCN layers
        dropout_rate: dropout rate for the TCN layers
        activation: activation function for the TCN layers
        interaction_regularization (float): Regularization parameter for the interaction features

    Returns:
        keras.Model: a keras model that can be trained to encode a sequence of motion tracking instances into a latent
        space using temporal convolutional networks.

    """
    # Define feature and adjacency inputs
    x = Input(shape=input_shape)
    a = Input(shape=edge_feature_shape)

    if use_gnn:
        x_reshaped = tf.transpose(
            tf.reshape(
                tf.transpose(x),
                [
                    -1,
                    adjacency_matrix.shape[-1],
                    x.shape[1],
                    input_shape[-1] // adjacency_matrix.shape[-1],
                ][::-1],
            )
        )
        a_reshaped = tf.transpose(
            tf.reshape(
                tf.transpose(a),
                [
                    -1,
                    edge_feature_shape[-1],
                    a.shape[1],
                    1,
                ][::-1],
            )
        )

    else:
        x_reshaped = tf.expand_dims(x, axis=1)

    encoder = TimeDistributed(
        tcn.TCN(
            conv_filters,
            kernel_size,
            conv_stacks,
            conv_dilations,
            padding,
            use_skip_connections,
            dropout_rate,
            return_sequences=False,
            activation=activation,
            kernel_initializer="random_normal",
            use_batch_norm=True,
        )
    )(x_reshaped)

    # Instantiate spatial graph block
    if use_gnn:

        # Embed edge features too
        a_encoder = TimeDistributed(
            tcn.TCN(
                conv_filters,
                kernel_size,
                conv_stacks,
                conv_dilations,
                padding,
                use_skip_connections,
                dropout_rate,
                return_sequences=False,
                activation=activation,
                kernel_initializer="random_normal",
                use_batch_norm=True,
            )
        )(a_reshaped)

        spatial_block = CensNetConv(
            node_channels=latent_dim,
            edge_channels=latent_dim,
            activation="relu",
            node_regularizer=tf.keras.regularizers.l1(interaction_regularization),
        )

        # Process adjacency matrix
        laplacian, edge_laplacian, incidence = spatial_block.preprocess(
            adjacency_matrix
        )

        # Get and concatenate node and edge embeddings
        x_nodes, x_edges = spatial_block(
            [encoder, (laplacian, edge_laplacian, incidence), a_encoder], mask=None
        )

        x_nodes = tf.reshape(
            x_nodes,
            [-1, adjacency_matrix.shape[-1] * latent_dim],
        )

        x_edges = tf.reshape(
            x_edges,
            [-1, edge_feature_shape[-1] * latent_dim],
        )

        encoder = tf.concat([x_nodes, x_edges], axis=-1)

    else:
        encoder = tf.squeeze(encoder, axis=1)

    encoder = tf.keras.layers.Dense(2 * latent_dim, activation="relu")(encoder)
    encoder = tf.keras.layers.BatchNormalization()(encoder)
    encoder = Dense(latent_dim, activation="relu")(encoder)
    encoder = tf.keras.layers.BatchNormalization()(encoder)
    encoder = tf.keras.layers.Dense(latent_dim)(encoder)

    return Model([x, a], encoder, name="TCN_encoder")

In [37]:
from typing import Iterable, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from deepof.clustering.censNetConv_pt import CensNetConvPT


def _act(name: str) -> nn.Module:
    name = (name or "relu").lower()
    if name == "relu":
        return nn.ReLU()
    if name == "gelu":
        return nn.GELU()
    if name == "tanh":
        return nn.Tanh()
    if name == "leaky_relu":
        return nn.LeakyReLU(0.2)
    if name in {"linear", "identity", "none"}:
        return nn.Identity()
    raise ValueError(f"Unsupported activation: {name}")


class TemporalBlockPT(nn.Module):
    """
    Residual TCN block compatible with keras-tcn:
      - Conv1d -> BN(eps=1e-3) -> Act -> Drop
      - Conv1d -> BN(eps=1e-3) -> Act -> Drop
      - Residual add (with 1x1 projection if channels differ) -> Act
    Returns:
      out: post-residual activation
      skip: post-second-conv activation (summed across blocks when skip connections are used)
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        dilation: int,
        padding: str = "causal",
        dropout_rate: float = 0.0,
        activation: str = "relu",
        use_batch_norm: bool = True,
        conv_init_std: float = 0.05,
    ):
        super().__init__()
        assert padding in {"causal", "same"}
        self.dilation = int(dilation)
        self.kernel_size = int(kernel_size)
        self.padding_mode = padding
        self.act = _act(activation)
        self.use_batch_norm = use_batch_norm

        pad = lambda: ((self.kernel_size - 1) * self.dilation) // 2 if padding == "same" else 0

        self.conv1 = nn.Conv1d(in_channels, out_channels, self.kernel_size, dilation=self.dilation, padding=pad(), bias=True)
        self.bn1 = nn.BatchNorm1d(out_channels, eps=1e-3) if use_batch_norm else nn.Identity()
        self.drop1 = nn.Dropout(float(dropout_rate)) if dropout_rate else nn.Identity()

        self.conv2 = nn.Conv1d(out_channels, out_channels, self.kernel_size, dilation=self.dilation, padding=pad(), bias=True)
        self.bn2 = nn.BatchNorm1d(out_channels, eps=1e-3) if use_batch_norm else nn.Identity()
        self.drop2 = nn.Dropout(float(dropout_rate)) if dropout_rate else nn.Identity()

        # 1x1 residual projection if channels differ
        self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=True) if in_channels != out_channels else None

        # Init similar to keras random_normal
        nn.init.normal_(self.conv1.weight, mean=0.0, std=conv_init_std); nn.init.zeros_(self.conv1.bias)
        nn.init.normal_(self.conv2.weight, mean=0.0, std=conv_init_std); nn.init.zeros_(self.conv2.bias)
        if self.downsample is not None:
            nn.init.normal_(self.downsample.weight, mean=0.0, std=conv_init_std); nn.init.zeros_(self.downsample.bias)

    def _causal_pad(self, x: torch.Tensor) -> torch.Tensor:
        pad = (self.kernel_size - 1) * self.dilation
        return F.pad(x, (pad, 0))

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # x: (B, C_in, T)
        y = self._causal_pad(x) if self.padding_mode == "causal" else x
        y = self.drop1(self.act(self.bn1(self.conv1(y))))

        y = self._causal_pad(y) if self.padding_mode == "causal" else y
        y = self.drop2(self.act(self.bn2(self.conv2(y))))

        skip = y  # per-block skip is the post-second-activation output

        res = x if self.downsample is None else self.downsample(x)
        out = self.act(y + res)
        return out, skip  # both (B, C_out, T)


class TCN1DPT(nn.Module):
    """
    Temporal Convolutional Network over sequences (B, T, C_in).
    - When use_skip_connections=True: sum per-block skip outputs, then apply a final activation.
    - Otherwise: use the last block’s residual output.
    - return_sequences=False: returns last timestep features (B, C_out).
    """
    def __init__(
        self,
        in_channels: int,
        conv_filters: int = 32,
        kernel_size: int = 4,
        conv_stacks: int = 2,
        conv_dilations: Iterable[int] = (1, 2, 4, 8),
        padding: str = "causal",
        use_skip_connections: bool = True,
        dropout_rate: float = 0.0,
        activation: str = "relu",
        use_batch_norm: bool = True,
        return_sequences: bool = False,
    ):
        super().__init__()
        self.use_skip_connections = use_skip_connections
        self.return_sequences = return_sequences
        self.final_act = _act(activation)

        blocks = []
        c_in = in_channels
        for _ in range(int(conv_stacks)):
            for d in tuple(conv_dilations):
                blocks.append(
                    TemporalBlockPT(
                        in_channels=c_in,
                        out_channels=conv_filters,
                        kernel_size=kernel_size,
                        dilation=int(d),
                        padding=padding,
                        dropout_rate=dropout_rate,
                        activation=activation,
                        use_batch_norm=use_batch_norm,
                    )
                )
                c_in = conv_filters
        self.blocks = nn.ModuleList(blocks)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, C_in) -> Conv1d expects (B, C_in, T)
        y = x.transpose(1, 2)
        skip_sum, last_out = None, None

        for blk in self.blocks:
            y, skip = blk(y)
            last_out = y
            if self.use_skip_connections:
                skip_sum = skip if skip_sum is None else (skip_sum + skip)

        out = skip_sum if self.use_skip_connections else last_out  # (B, C, T)
        out = self.final_act(out)
        return out.transpose(1, 2) if self.return_sequences else out[:, :, -1]


class TCNEncoderPT(nn.Module):
    """
    PyTorch port of the TF get_TCN_encoder with matching behavior:
      - Inputs:
          x: (B, W, N, NF)   node features
          a: (B, W, E, EF)   edge features
      - use_gnn=True:
          TimeDistributed(TCN) over nodes/edges -> (B, N, C) and (B, E, C)
          CensNetConvPT([node, (lap, edge_lap, inc), edge]) -> (B, N, latent), (B, E, latent)
          Flatten and MLP head
      - use_gnn=False:
          Flatten nodes+features -> TCN -> MLP head

      Parity details:
        - keras-tcn-compatible skip semantics and activation placement
        - BN eps=1e-3 everywhere
        - 'causal' and 'same' paddings supported
    """
    def __init__(
        self,
        input_shape: Tuple[int, int, int],        # (W, N, NF)
        edge_feature_shape: Tuple[int, int, int], # (W, E, EF)
        adjacency_matrix: np.ndarray,
        latent_dim: int,
        use_gnn: bool = True,
        conv_filters: int = 32,
        kernel_size: int = 4,
        conv_stacks: int = 2,
        conv_dilations: Iterable[int] = (1, 2, 4, 8),
        padding: str = "causal",
        use_skip_connections: bool = True,
        dropout_rate: float = 0.0,
        activation: str = "relu",
        interaction_regularization: float = 0.0,  # not used explicitly in PT
        use_batch_norm: bool = True,
    ):
        super().__init__()
        self.use_gnn = use_gnn
        self.latent_dim = int(latent_dim)
        self.conv_filters = int(conv_filters)

        W, N, F_node = input_shape
        _, E, F_edge = edge_feature_shape
        assert adjacency_matrix.shape[0] == N == adjacency_matrix.shape[1], "Adjacency must be NxN and match input nodes."

        tcn_cfg = dict(
            conv_filters=conv_filters,
            kernel_size=kernel_size,
            conv_stacks=conv_stacks,
            conv_dilations=tuple(conv_dilations),
            padding=padding,
            use_skip_connections=use_skip_connections,
            dropout_rate=float(dropout_rate),
            activation=activation,
            use_batch_norm=use_batch_norm,
            return_sequences=False,
        )

        if use_gnn:
            # Per-node and per-edge TCNs
            self.node_tcn = TCN1DPT(in_channels=F_node, **tcn_cfg)
            self.edge_tcn = TCN1DPT(in_channels=F_edge, **tcn_cfg)

            # Graph block and buffers
            self.spatial_gnn_block = CensNetConvPT(node_channels=latent_dim, edge_channels=latent_dim, activation="relu")
            lap, edge_lap, inc = self.spatial_gnn_block.preprocess(torch.tensor(adjacency_matrix))
            self.register_buffer("laplacian", lap.float())
            self.register_buffer("edge_laplacian", edge_lap.float())
            self.register_buffer("incidence", inc.float())

            final_in = (N * latent_dim) + (E * latent_dim)
        else:
            # Single TCN over flattened node features
            self.flat_tcn = TCN1DPT(in_channels=N * F_node, **tcn_cfg)
            final_in = conv_filters

        # Head MLP: Dense(2*latent) -> BN -> Dense(latent) -> BN -> Dense(latent)
        self.head = nn.Sequential(
            nn.Linear(final_in, 2 * latent_dim),
            nn.ReLU(),
            nn.BatchNorm1d(2 * latent_dim, eps=1e-3),
            nn.Linear(2 * latent_dim, latent_dim),
            nn.ReLU(),
            nn.BatchNorm1d(latent_dim, eps=1e-3),
            nn.Linear(latent_dim, latent_dim),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """
        x: (B, W, N, NF)  a: (B, W, E, EF)  -> returns (B, latent_dim)
        """
        B, W, N, F_node = x.shape
        _, _, E, F_edge = a.shape

        if self.use_gnn:
            # Nodes: TF-style reshape pipeline to match memory layout exactly
            x_3d = x.view(B, W, N * F_node)          # (B, W, N*F)
            x_t = x_3d.permute(2, 1, 0)              # (N*F, W, B)
            x_reshaped_t = x_t.reshape(F_node, W, N, B)
            x_nodes = x_reshaped_t.permute(3, 2, 1, 0)  # (B, N, W, F)

            node_in = x_nodes.reshape(B * N, W, F_node)
            node_out = self.node_tcn(node_in).view(B, N, self.conv_filters)  # (B, N, C)

            # Edges: TF-style reshape pipeline to match memory layout exactly
            a_3d = a.view(B, W, E * F_edge)          # (B, W, E*F_edge)
            a_t = a_3d.permute(2, 1, 0)              # (E*F_edge, W, B)
            a_reshaped_t = a_t.reshape(F_edge, W, E, B)
            a_edges = a_reshaped_t.permute(3, 2, 1, 0)  # (B, E, W, F_edge)

            edge_in = a_edges.reshape(B * E, W, F_edge)
            edge_out = self.edge_tcn(edge_in).view(B, E, self.conv_filters)  # (B, E, C)

            # Graph block
            adj_tuple = (self.laplacian, self.edge_laplacian, self.incidence)
            x_nodes_g, x_edges_g = self.spatial_gnn_block([node_out, adj_tuple, edge_out])
            x_nodes_g = F.relu(x_nodes_g)
            x_edges_g = F.relu(x_edges_g)

            enc = torch.cat([x_nodes_g.reshape(B, -1), x_edges_g.reshape(B, -1)], dim=-1)
        else:
            # Non-GNN unchanged
            x_flat = x.view(B, W, N * F_node)        # (B, W, N*NF)
            enc = self.flat_tcn(x_flat)              # (B, C)

        return self.head(enc)

In [38]:
import numpy as np
import torch
import torch.nn as nn
import tensorflow as tf
import tcn as tcn_pkg

def _tf_conv1d_to_torch(w_keras: np.ndarray) -> torch.Tensor:
    # TF Conv1D [K, Cin, Cout] -> PT Conv1d [Cout, Cin, K]
    return torch.from_numpy(np.transpose(w_keras, (2, 1, 0)))

def _load_bn_tf_to_pt(pt_bn: nn.BatchNorm1d, tf_bn: tf.keras.layers.BatchNormalization):
    gamma, beta, moving_mean, moving_var = tf_bn.get_weights()
    pt_bn.weight.data = torch.from_numpy(gamma)
    pt_bn.bias.data = torch.from_numpy(beta)
    pt_bn.running_mean.data = torch.from_numpy(moving_mean)
    pt_bn.running_var.data = torch.from_numpy(moving_var)

def _kernel_size_1(conv: tf.keras.layers.Conv1D) -> bool:
    ks = conv.kernel_size
    ks = ks[0] if isinstance(ks, tuple) else ks
    return ks == 1

def _collect_tcn_sublayers(tf_tcn_layer: tf.keras.layers.Layer):
    # Use submodules as before; we’ll still verify counts and assign conservatively
    convs = [m for m in tf_tcn_layer.submodules if isinstance(m, tf.keras.layers.Conv1D)]
    bns = [m for m in tf_tcn_layer.submodules if isinstance(m, tf.keras.layers.BatchNormalization)]
    return convs, bns

def transfer_td_tcn_weights(tf_td_tcn: tf.keras.layers.TimeDistributed, pt_tcn) -> None:
    """
    Transfer weights from TF TimeDistributed(tcn.TCN) into PT TCN1DPT:
      - Map per-block [conv1, conv2] (in order) and their BN layers
      - Map the single residual 1x1 projection (matching_conv1D), if present
      - No skip 1x1 convs (your model has none)
    """
    assert isinstance(tf_td_tcn, tf.keras.layers.TimeDistributed)
    assert isinstance(tf_td_tcn.layer, tcn_pkg.TCN)
    tf_tcn = tf_td_tcn.layer

    convs, bns = _collect_tcn_sublayers(tf_tcn)
    block_convs = [c for c in convs if not _kernel_size_1(c)]   # conv1D_0 / conv1D_1 pairs per block
    proj_1x1 = [c for c in convs if _kernel_size_1(c)]          # matching_conv1D (0 or 1 in your build)

    num_blocks = len(pt_tcn.blocks)
    assert len(block_convs) == 2 * num_blocks, f"Conv count mismatch: TF block convs={len(block_convs)}, PT blocks={num_blocks}"

    use_bn = isinstance(pt_tcn.blocks[0].bn1, nn.BatchNorm1d)
    if use_bn:
        assert len(bns) >= 2 * num_blocks, f"BN count mismatch: TF BNs={len(bns)}, expected >= {2 * num_blocks}"

    # Load per-block convs and BN stats
    for i, blk in enumerate(pt_tcn.blocks):
        k1, b1 = block_convs[2 * i].get_weights()
        blk.conv1.weight.data = _tf_conv1d_to_torch(k1)
        blk.conv1.bias.data = torch.from_numpy(b1)

        k2, b2 = block_convs[2 * i + 1].get_weights()
        blk.conv2.weight.data = _tf_conv1d_to_torch(k2)
        blk.conv2.bias.data = torch.from_numpy(b2)

        if use_bn:
            _load_bn_tf_to_pt(blk.bn1, bns[2 * i])
            _load_bn_tf_to_pt(blk.bn2, bns[2 * i + 1])

    # Residual projection for the first block if needed
    proj_idx = 0
    for blk in pt_tcn.blocks:
        if isinstance(getattr(blk, "downsample", None), nn.Conv1d):
            rk, rb = proj_1x1[proj_idx].get_weights()
            blk.downsample.weight.data = _tf_conv1d_to_torch(rk)
            blk.downsample.bias.data = torch.from_numpy(rb)
            proj_idx += 1


# ---------- MLP head transfer ----------

def transfer_head_mlp(tf_model, pt_model_head: nn.Sequential):
    """
    Transfer the final MLP head:
      Dense(2*latent, relu) -> BN -> Dense(latent, relu) -> BN -> Dense(latent)
    from TF model to PT head (Linear, BN, Linear, BN, Linear).
    """
    # Extract the final [Dense, BN, Dense, BN, Dense] from TF model
    tail = [l for l in tf_model.layers if isinstance(l, (tf.keras.layers.Dense, tf.keras.layers.BatchNormalization))]
    d1, bn1, d2, bn2, d3 = tail[-5:]

    # PT head layout: [Linear, ReLU, BN, Linear, ReLU, BN, Linear]
    lin1: nn.Linear = pt_model_head[0]
    bn1_pt: nn.BatchNorm1d = pt_model_head[2]
    lin2: nn.Linear = pt_model_head[3]
    bn2_pt: nn.BatchNorm1d = pt_model_head[5]
    lin3: nn.Linear = pt_model_head[6]

    # Dense 1
    w, b = d1.get_weights()
    lin1.weight.data = torch.from_numpy(w.T)
    lin1.bias.data = torch.from_numpy(b)
    # BN 1
    _load_bn_tf_to_pt(bn1_pt, bn1)
    # Dense 2
    w, b = d2.get_weights()
    lin2.weight.data = torch.from_numpy(w.T)
    lin2.bias.data = torch.from_numpy(b)
    # BN 2
    _load_bn_tf_to_pt(bn2_pt, bn2)
    # Dense 3
    w, b = d3.get_weights()
    lin3.weight.data = torch.from_numpy(w.T)
    lin3.bias.data = torch.from_numpy(b)


# ---------- High-level: TCN encoder transfer ----------

def transfer_tcn_encoder_weights(tf_model, pt_model, use_gnn: bool):
    """
    Transfers weights for the full TCN encoder.
      - Node and edge TimeDistributed(TCN) blocks
      - CensNetConv (if use_gnn)
      - Final MLP head
    """
    # 1) Final head
    transfer_head_mlp(tf_model, pt_model.head)

    # 2) TimeDistributed(TCN) blocks
    td_layers = [l for l in tf_model.layers if isinstance(l, tf.keras.layers.TimeDistributed) and isinstance(l.layer, tcn.TCN)]
    if use_gnn:
        assert len(td_layers) >= 2, "Expected two TimeDistributed(TCN) layers (node and edge) for use_gnn=True"
        # Heuristically: first TD is nodes, second is edges (matches build order)
        node_td = td_layers[0]
        edge_td = td_layers[1]
        transfer_td_tcn_weights(node_td, pt_model.node_tcn)
        transfer_td_tcn_weights(edge_td, pt_model.edge_tcn)

        # 3) CensNetConv
        gnn_layer = next(l for l in tf_model.layers if isinstance(l, CensNetConv))
        transfer_censnet_weights(gnn_layer, pt_model.spatial_gnn_block)

    else:
        # Non-GNN: single TD(TCN); TF input_shape should be (T, N*F_node)
        assert len(td_layers) >= 1, "Expected one TimeDistributed(TCN) layer for use_gnn=False"
        transfer_td_tcn_weights(td_layers[0], pt_model.flat_tcn)

def transfer_censnet_weights(tf_layer, pt_layer):
    """
    Transfers all six weights from a Spektral CensNetConv layer to the
    corresponding CensNetConvPT layer.
    """
    # Get all weights from the TensorFlow layer. The order is determined by
    # the layer's build order in Spektral's source code.
    tf_weights = tf_layer.get_weights()

    # Unpack all six weights.
    # Order: kernel_node, bias_node, kernel_edge, bias_edge, projector_node, projector_edge
    kn_tf, bn_tf, ke_tf, be_tf, pn_tf, pe_tf = tf_weights

    # Build weights on first pass
    if pt_layer.node_kernel is None:
        # Move parameters to the same device as input tensors
        pt_layer._build(kn_tf.T.shape, bn_tf.T.shape)
        #pt_layer.to(kn_tf.device)

    # 1. & 2. Transfer Node Kernel and Bias
    # Keras Dense kernel is (in_features, out_features)
    pt_layer.node_kernel.data = torch.from_numpy(kn_tf)
    pt_layer.edge_kernel.data = torch.from_numpy(bn_tf)

    # 3. & 4. Transfer Edge Kernel and Bias
    # Same transposition logic applies.
    pt_layer.node_weights.data = torch.from_numpy(ke_tf)
    pt_layer.edge_weights.data = torch.from_numpy(be_tf)

    # 5. Transfer Node Projector Weights (P_n)
    # These are [in_features, 1], which matches, so no transpose needed.
    pt_layer.node_bias.data = torch.from_numpy(pn_tf)

    # 6. Transfer Edge Projector Weights (P_e)
    # These are [in_features, 1], which matches, so no transpose needed.
    pt_layer.edge_bias.data = torch.from_numpy(pe_tf)

In [39]:
import unittest, time
import numpy as np
import torch

def count_undirected_edges(adj: np.ndarray) -> int:
    # Count upper-triangular non-zero entries (undirected edges)
    return int(np.count_nonzero(np.triu(adj, 1)))

class TestTCNEncoderTranslation(unittest.TestCase):
    def setUp(self):
        tf.keras.backend.clear_session()

        # Fundamental dims (use your conventions)
        self.R = 2048                 # number of rows (not used for model build)
        self.W = 25                   # window length
        self.N = 11                   # nodes
        self.NF = 3                   # features per node
        self.EF = 1                   # features per edge (TF expects 1 for the reshape quirk)
        self.latent_dim = 6
        self.use_gnn = True

        # Batch used for parity test
        self.B = 128

        # Make an adjacency whose undirected edge count E matches the edge axis we'll use
        # Example: pick a sparse symmetric adjacency with E edges
        # If you already have an adjacency, just set self.adj_matrix = your_matrix and let E = count_undirected_edges(A)
        rng = np.random.default_rng(0)
        A = np.zeros((self.N, self.N), dtype=np.float32)
        # randomly pick E edges; here we choose E = 11 (as in your typical config)
        target_E = 11
        iu = np.triu_indices(self.N, 1)
        idx = rng.choice(len(iu[0]), size=target_E, replace=False)
        A[iu[0][idx], iu[1][idx]] = 1.0
        A = A + A.T
        self.adj_matrix = A
        self.E = count_undirected_edges(self.adj_matrix)  # should be target_E

        # TF input shapes (flattened)
        self.tf_input_shape_gnn = (self.W, self.N * self.NF)  # (W, NNF)
        self.tf_edge_shape      = (self.W, self.E * self.EF)  # (W, EEF) -> with EF=1, equals (W, E)

        # PT input shapes (split)
        self.pt_input_shape = (self.W, self.N, self.NF)       # (W, N, NF)
        self.pt_edge_shape  = (self.W, self.E, self.EF)       # (W, E, EF)

        # Random inputs
        self.x_pt = rng.normal(size=(self.B, self.W, self.N, self.NF)).astype(np.float32)
        self.a_pt = rng.normal(size=(self.B, self.W, self.E, self.EF)).astype(np.float32)
        # Flatten for TF model
        self.x_tf = self.x_pt.reshape(self.B, self.W, self.N * self.NF)
        self.a_tf = self.a_pt.reshape(self.B, self.W, self.E * self.EF)  # with EF=1, (B, W, E)

        # Common TCN params
        self.conv_filters = 32
        self.kernel_size = 3
        self.conv_stacks = 2
        self.conv_dilations = (1, 2)
        self.padding = "causal"
        self.use_skip = True
        self.dropout = 0.0
        self.activation = "relu"

    def test_forward_pass_gnn(self):
        # Build TF and PT models
        tf_model = get_TCN_encoder(
            input_shape=self.tf_input_shape_gnn,   # (W, NNF)
            edge_feature_shape=self.tf_edge_shape, # (W, EEF) -> E when EF=1
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=True,
            conv_filters=self.conv_filters,
            kernel_size=self.kernel_size,
            conv_stacks=self.conv_stacks,
            conv_dilations=self.conv_dilations,
            padding=self.padding,
            use_skip_connections=self.use_skip,
            dropout_rate=self.dropout,
            activation=self.activation,
        )
        pt_model = TCNEncoderPT(
            input_shape=self.pt_input_shape,       # (W, N, NF)
            edge_feature_shape=self.pt_edge_shape, # (W, E, EF)
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=True,
            conv_filters=self.conv_filters,
            kernel_size=self.kernel_size,
            conv_stacks=self.conv_stacks,
            conv_dilations=self.conv_dilations,
            padding=self.padding,
            use_skip_connections=self.use_skip,
            dropout_rate=self.dropout,
            activation=self.activation,
        )
        pt_model.eval()

        # Warm-up PT graph (optional)
        with torch.no_grad():
            _ = pt_model(torch.from_numpy(self.x_pt), torch.from_numpy(self.a_pt))

        # Transfer weights TF -> PT
        transfer_tcn_encoder_weights(tf_model, pt_model, use_gnn=True)

        # Compare outputs (TF expects flattened a)
        t0 = time.time()
        y_tf = tf_model([self.x_tf, self.a_tf], training=False).numpy()
        t1 = time.time()
        with torch.no_grad():
            y_pt = pt_model(torch.from_numpy(self.x_pt), torch.from_numpy(self.a_pt)).cpu().numpy()
        t2 = time.time()
        print("GNN TF time:", t1 - t0, "PT time:", t2 - t1)
        np.testing.assert_allclose(y_tf, y_pt, rtol=1e-5, atol=2e-4)
        print("✅ TCNEncoderPT (GNN path) parity PASSED")

    def test_forward_pass_no_gnn(self):
        # Build TF and PT models (TF expects flattened x, a still provided but unused)
        tf_model = get_TCN_encoder(
            input_shape=self.tf_input_shape_gnn,   # (W, NNF) in your pipeline
            edge_feature_shape=self.tf_edge_shape, # (W, EEF)
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=False,
            conv_filters=self.conv_filters,
            kernel_size=self.kernel_size,
            conv_stacks=self.conv_stacks,
            conv_dilations=self.conv_dilations,
            padding=self.padding,
            use_skip_connections=self.use_skip,
            dropout_rate=self.dropout,
            activation=self.activation,
        )
        pt_model = TCNEncoderPT(
            input_shape=self.pt_input_shape,       # (W, N, NF)
            edge_feature_shape=self.pt_edge_shape, # (W, E, EF)
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=False,
            conv_filters=self.conv_filters,
            kernel_size=self.kernel_size,
            conv_stacks=self.conv_stacks,
            conv_dilations=self.conv_dilations,
            padding=self.padding,
            use_skip_connections=self.use_skip,
            dropout_rate=self.dropout,
            activation=self.activation,
        )
        pt_model.eval()

        # Transfer weights TF -> PT
        transfer_tcn_encoder_weights(tf_model, pt_model, use_gnn=False)

        # Compare outputs (TF expects flattened x, a flattened to EEF)
        y_tf = tf_model([self.x_tf, self.a_tf], training=False).numpy()
        with torch.no_grad():
            y_pt = pt_model(torch.from_numpy(self.x_pt), torch.from_numpy(self.a_pt)).cpu().numpy()

        np.testing.assert_allclose(y_tf, y_pt, rtol=1e-5, atol=2e-4)
        print("✅ TCNEncoderPT (non-GNN path) parity PASSED")

# Run
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestTCNEncoderTranslation)
runner.run(suite)

test_forward_pass_gnn (__main__.TestTCNEncoderTranslation) ... The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
ok
test_forward_pass_no_gnn (__main__.TestTCNEncoderTranslation) ... 

GNN TF time: 0.5147807598114014 PT time: 0.0169522762298584
✅ TCNEncoderPT (GNN path) parity PASSED


ok

----------------------------------------------------------------------
Ran 2 tests in 1.261s

OK


✅ TCNEncoderPT (non-GNN path) parity PASSED


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

In [40]:
# TCN-only parity diagnostic (non-GNN), fully executable with your provided dims

import numpy as np
import torch
import torch.nn as nn
import tensorflow as tf
import tcn as tcn_pkg

# Assumes get_TCN_encoder and TCNEncoderPT are already defined/imported in your notebook.

# -------------------------
# 1) Your provided settings
# -------------------------
batch_size = 128
window_length = 25
num_nodes = 11
features_per_node = 3
num_edges = 11
features_per_edge = 1

latent_dim = 6
use_gnn = False

# Adjacency matrix
m = np.zeros((num_nodes, num_nodes), dtype=np.float32)
ui = np.triu_indices(num_nodes)
num_possible_edges = len(ui[0])
c = np.random.choice(num_possible_edges, min(num_edges, num_possible_edges), replace=False)
m[ui[0][c], ui[1][c]] = 1
m = (m + m.T).astype(np.float32)  # symmetric
adj_matrix = m

# Input shapes
tf_input_shape = (window_length, num_nodes * features_per_node)     # TF non-GNN expects flattened features
pt_input_shape = (window_length, num_nodes, features_per_node)      # PT expects (T, N, F_node)
edge_shape = (window_length, num_edges, features_per_edge)

# Random inputs
rng = np.random.default_rng(0)
x_np = rng.normal(size=(batch_size, window_length, num_nodes, features_per_node)).astype(np.float32)
a_np = rng.normal(size=(batch_size, window_length, num_edges, features_per_edge)).astype(np.float32)

# ------------------------------------
# 2) Build TF and PT (non-GNN) models
# ------------------------------------
tf_model = get_TCN_encoder(
    input_shape=tf_input_shape,
    edge_feature_shape=edge_shape,
    adjacency_matrix=adj_matrix,
    latent_dim=latent_dim,
    use_gnn=False,
    conv_filters=32,
    kernel_size=3,
    conv_stacks=2,
    conv_dilations=(1, 2),
    padding="causal",
    use_skip_connections=True,
    dropout_rate=0.0,
    activation="relu",
)

pt_model = TCNEncoderPT(
    input_shape=pt_input_shape,
    edge_feature_shape=edge_shape,
    adjacency_matrix=adj_matrix,
    latent_dim=latent_dim,
    use_gnn=False,
    conv_filters=32,
    kernel_size=3,
    conv_stacks=2,
    conv_dilations=(1, 2),
    padding="causal",
    use_skip_connections=True,
    dropout_rate=0.0,
    activation="relu",
)

pt_model.eval()

# Ensure PT BN eps=1e-3 inside TCN (BN eps mismatch is a common source of diffs)
for blk in pt_model.flat_tcn.blocks:
    if isinstance(blk.bn1, nn.BatchNorm1d):
        blk.bn1.eps = 1e-3
    if isinstance(blk.bn2, nn.BatchNorm1d):
        blk.bn2.eps = 1e-3

# ------------------------------------------------
# 3) Helpers: extract TD(TCN) and transfer weights
# ------------------------------------------------
def get_first_td_tcn(tf_model):
    for l in tf_model.layers:
        if isinstance(l, tf.keras.layers.TimeDistributed) and isinstance(l.layer, tcn_pkg.TCN):
            return l
    raise RuntimeError("No TimeDistributed(TCN) found in TF model.")

def _tf_conv1d_to_torch(w_keras: np.ndarray) -> torch.Tensor:
    # TF/Keras Conv1D: [kernel, in, out] -> PT Conv1d: [out, in, kernel]
    return torch.from_numpy(np.transpose(w_keras, (2, 1, 0)))

def _load_bn_tf_to_pt(pt_bn: nn.BatchNorm1d, tf_bn: tf.keras.layers.BatchNormalization):
    gamma, beta, moving_mean, moving_var = tf_bn.get_weights()
    pt_bn.weight.data = torch.from_numpy(gamma)
    pt_bn.bias.data = torch.from_numpy(beta)
    pt_bn.running_mean.data = torch.from_numpy(moving_mean)
    pt_bn.running_var.data = torch.from_numpy(moving_var)

def _kernel_size_1(conv: tf.keras.layers.Conv1D) -> bool:
    ks = conv.kernel_size
    ks = ks[0] if isinstance(ks, tuple) else ks
    return ks == 1

def _collect_tcn_sublayers(tf_tcn_layer: tf.keras.layers.Layer):
    convs = [m for m in tf_tcn_layer.submodules if isinstance(m, tf.keras.layers.Conv1D)]
    bns = [m for m in tf_tcn_layer.submodules if isinstance(m, tf.keras.layers.BatchNormalization)]
    return convs, bns

def transfer_td_tcn_weights(tf_td_tcn: tf.keras.layers.TimeDistributed, pt_tcn) -> None:
    assert isinstance(tf_td_tcn, tf.keras.layers.TimeDistributed)
    assert isinstance(tf_td_tcn.layer, tcn_pkg.TCN)
    tf_tcn = tf_td_tcn.layer

    convs, bns = _collect_tcn_sublayers(tf_tcn)
    block_convs = [c for c in convs if not _kernel_size_1(c)]
    resid_convs = [c for c in convs if _kernel_size_1(c)]  # includes residual 1x1 (and possibly skip 1x1s)

    num_blocks = len(pt_tcn.blocks)
    assert len(block_convs) == 2 * num_blocks, f"Conv count mismatch: TF block convs={len(block_convs)}, PT blocks={num_blocks}"

    # Map conv1/conv2 + BN1/BN2
    use_bn = isinstance(pt_tcn.blocks[0].bn1, nn.BatchNorm1d)
    if use_bn:
        assert len(bns) >= 2 * num_blocks, f"BN count mismatch: TF BNs={len(bns)}, expected >= {2 * num_blocks}"

    for i, blk in enumerate(pt_tcn.blocks):
        k1, b1 = block_convs[2 * i].get_weights()
        blk.conv1.weight.data = _tf_conv1d_to_torch(k1)
        blk.conv1.bias.data = torch.from_numpy(b1)

        k2, b2 = block_convs[2 * i + 1].get_weights()
        blk.conv2.weight.data = _tf_conv1d_to_torch(k2)
        blk.conv2.bias.data = torch.from_numpy(b2)

        if use_bn:
            _load_bn_tf_to_pt(blk.bn1, bns[2 * i])
            _load_bn_tf_to_pt(blk.bn2, bns[2 * i + 1])

    # Residual 1x1 projection: only if PT block has it (assumes attribute name 'downsample' as you set)
    resid_idx = 0
    for blk in pt_tcn.blocks:
        if isinstance(getattr(blk, "downsample", None), nn.Conv1d):
            rk, rb = resid_convs[resid_idx].get_weights()
            blk.downsample.weight.data = _tf_conv1d_to_torch(rk)
            blk.downsample.bias.data = torch.from_numpy(rb)
            resid_idx += 1

# ------------------------------------------------------
# 4) Compare TCN-only outputs (TF TD(TCN) vs PT flat_tcn)
# ------------------------------------------------------
# Build a TF submodel that outputs the TimeDistributed(TCN) output only
td = get_first_td_tcn(tf_model)
tf_tcn_sub = tf.keras.Model(tf_model.inputs, td.output)  # -> (B, 1, conv_filters)

# Transfer only the TCN weights TF -> PT (no head)
transfer_td_tcn_weights(td, pt_model.flat_tcn)

# Prepare inputs
x_tf = x_np.reshape(batch_size, window_length, num_nodes * features_per_node)  # flattened for TF
x_pt = torch.from_numpy(x_np)

# Run
tf_out = tf_tcn_sub([x_tf, a_np], training=False).numpy()   # (B, 1, C)
tf_out = np.squeeze(tf_out, axis=1)                         # (B, C)

with torch.no_grad():
    pt_out = pt_model.flat_tcn(x_pt.view(batch_size, window_length, num_nodes * features_per_node)).cpu().numpy()  # (B, C)

# Report basic stats
abs_diff = np.abs(tf_out - pt_out)
print("TCN-only shapes -> TF:", tf_out.shape, "PT:", pt_out.shape)
print("TCN-only mean abs diff:", abs_diff.mean())
print("TCN-only max abs diff:", abs_diff.max())

TCN-only shapes -> TF: (128, 32) PT: (128, 32)
TCN-only mean abs diff: 4.136673e-08
TCN-only max abs diff: 2.9802322e-07


In [41]:
def inspect_tf_tcn(tf_model):
    import tcn as tcn_pkg
    from tensorflow.keras.layers import TimeDistributed, Conv1D, BatchNormalization

    td = None
    for l in tf_model.layers:
        if isinstance(l, TimeDistributed) and isinstance(l.layer, tcn_pkg.TCN):
            td = l; break
    assert td is not None, "No TimeDistributed(TCN) found."

    tf_tcn = td.layer
    convs = [m for m in tf_tcn.submodules if isinstance(m, Conv1D)]
    bns   = [m for m in tf_tcn.submodules if isinstance(m, BatchNormalization)]

    print("Convs:")
    for i, c in enumerate(convs):
        ks = c.kernel_size[0] if isinstance(c.kernel_size, tuple) else c.kernel_size
        print(f"  {i:2d}: name={c.name}, kernel_size={ks}, filters={c.filters}")
    print("BNs:")
    for i, b in enumerate(bns):
        print(f"  {i:2d}: name={b.name}, epsilon={b.epsilon}, momentum={b.momentum}")

inspect_tf_tcn(tf_model)

Convs:
   0: name=conv1D_0, kernel_size=3, filters=32
   1: name=conv1D_1, kernel_size=3, filters=32
   2: name=matching_conv1D, kernel_size=1, filters=32
   3: name=conv1D_0, kernel_size=3, filters=32
   4: name=conv1D_1, kernel_size=3, filters=32
   5: name=conv1D_0, kernel_size=3, filters=32
   6: name=conv1D_1, kernel_size=3, filters=32
   7: name=conv1D_0, kernel_size=3, filters=32
   8: name=conv1D_1, kernel_size=3, filters=32
BNs:
   0: name=batch_normalization, epsilon=0.001, momentum=0.99
   1: name=batch_normalization_1, epsilon=0.001, momentum=0.99
   2: name=batch_normalization_2, epsilon=0.001, momentum=0.99
   3: name=batch_normalization_3, epsilon=0.001, momentum=0.99
   4: name=batch_normalization_4, epsilon=0.001, momentum=0.99
   5: name=batch_normalization_5, epsilon=0.001, momentum=0.99
   6: name=batch_normalization_6, epsilon=0.001, momentum=0.99
   7: name=batch_normalization_7, epsilon=0.001, momentum=0.99


# TCN decoder test

In [1]:
from typing import Iterable, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# TensorFlow + keras-tcn for building the reference TF decoder and grabbing weights
import tensorflow as tf
import tcn as tcn_pkg

# The TF probabilistic decoder layer type (for weight transfer)
# If your TF class lives elsewhere, adjust this import accordingly.
import deepof
from deepof.model_utils import ProbabilisticDecoder as ProbabilisticDecoderTF

# The PT probabilistic decoder (you said this already exists)
# Adjust import path if needed.
from deepof.clustering.model_utils_new import ProbabilisticDecoderPT

from tensorflow.keras.layers import Input, TimeDistributed, Bidirectional, GRU, LayerNormalization, Masking
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    GRU,
    Bidirectional,
    Dense,
    LayerNormalization,
    RepeatVector,
    TimeDistributed,
)
import tensorflow as tf
import tcn


In [2]:
def get_TCN_decoder(
    input_shape: tuple,
    latent_dim: int,
    conv_filters: int = 64,
    kernel_size: int = 4,
    conv_stacks: int = 1,
    conv_dilations: tuple = (8, 4, 2, 1),
    padding: str = "causal",
    use_skip_connections: bool = True,
    dropout_rate: int = 0,
    activation: str = "relu",
):
    """Return a Temporal Convolutional Network (TCN) decoder.

    Builds a neural network that can be used to decode a latent space into a sequence of
    motion tracking instances. Each layer contains a residual block with a convolutional layer and a skip connection. See
    the following paper for more details: https://arxiv.org/pdf/1803.01271.pdf,

    Args:
        input_shape: shape of the input data
        latent_dim: dimensionality of the latent space
        conv_filters: number of filters in the TCN layers
        kernel_size: size of the convolutional kernels
        conv_stacks: number of TCN layers
        conv_dilations: list of dilation factors for each TCN layer
        padding: padding mode for the TCN layers
        use_skip_connections: whether to use skip connections between TCN layers
        dropout_rate: dropout rate for the TCN layers
        activation: activation function for the TCN layers

    Returns:
        keras.Model: a keras model that can be trained to decode a latent space into a sequence of motion tracking
        instances using temporal convolutional networks.

    """
    # Define and instantiate generator
    g = Input(shape=latent_dim)  # Decoder input, shaped as the latent space
    x = Input(shape=input_shape)  # Encoder input, used to generate an output mask
    validity_mask = tf.math.logical_not(tf.reduce_all(x == 0.0, axis=2))

    generator = tf.keras.layers.Dense(latent_dim)(g)
    generator = tf.keras.layers.BatchNormalization()(generator)
    generator = tf.keras.layers.Dense(2 * latent_dim, activation="relu")(generator)
    generator = tf.keras.layers.BatchNormalization()(generator)
    generator = tf.keras.layers.Dense(4 * latent_dim, activation="relu")(generator)
    generator = tf.keras.layers.BatchNormalization()(generator)
    generator = tf.keras.layers.RepeatVector(input_shape[0])(generator)

    generator = tcn.TCN(
        conv_filters,
        kernel_size,
        conv_stacks,
        conv_dilations,
        padding,
        use_skip_connections,
        dropout_rate,
        return_sequences=True,
        activation=activation,
        kernel_initializer="random_normal",
        use_batch_norm=True,
    )(generator)

    x_decoded = deepof.model_utils.ProbabilisticDecoder(input_shape)(
        [generator, validity_mask]
    )

    return Model([g, x], x_decoded, name="TCN_decoder")

In [3]:
from deepof.clustering.models_new import TCN1DPT, TemporalBlockPT, _act



class TCNDecoderPT(nn.Module):
    """
    PyTorch port of TF get_TCN_decoder:
      - g: (B, latent_dim)
      - x: (B, W, NNF) or (B, W, N, NF) for mask computation
      Pipeline:
        Dense(latent) -> BN ->
        Dense(2*latent, relu) -> BN ->
        Dense(4*latent, relu) -> BN ->
        RepeatVector(W) ->
        TCN(return_sequences=True) ->
        ProbabilisticDecoderPT(hidden_dim=conv_filters, data_dim=NNF)
      Returns: a distribution whose .mean is (B, W, NNF)
    """
    def __init__(
        self,
        input_shape: Tuple[int, int],   # (W, NNF)
        latent_dim: int,
        conv_filters: int = 64,
        kernel_size: int = 4,
        conv_stacks: int = 1,
        conv_dilations: Iterable[int] = (8, 4, 2, 1),
        padding: str = "causal",
        use_skip_connections: bool = True,
        dropout_rate: float = 0.0,
        activation: str = "relu",
        use_batch_norm: bool = True,
    ):
        super().__init__()
        self.W, self.data_dim = int(input_shape[0]), int(input_shape[1])
        self.latent_dim = int(latent_dim)

        # Front MLP: Dense -> BN -> Dense(relu) -> BN -> Dense(relu) -> BN
        self.fc0 = nn.Linear(latent_dim, latent_dim)
        self.bn0 = nn.BatchNorm1d(latent_dim, eps=1e-3)

        self.fc1 = nn.Linear(latent_dim, 2 * latent_dim)
        self.act1 = _act(activation)
        self.bn1 = nn.BatchNorm1d(2 * latent_dim, eps=1e-3)

        self.fc2 = nn.Linear(2 * latent_dim, 4 * latent_dim)
        self.act2 = _act(activation)
        self.bn2 = nn.BatchNorm1d(4 * latent_dim, eps=1e-3)

        # TCN over repeated latent sequence
        self.tcn = TCN1DPT(
            in_channels=4 * latent_dim,
            conv_filters=conv_filters,
            kernel_size=kernel_size,
            conv_stacks=conv_stacks,
            conv_dilations=conv_dilations,
            padding=padding,
            use_skip_connections=use_skip_connections,
            dropout_rate=float(dropout_rate),
            activation=activation,
            use_batch_norm=use_batch_norm,
            return_sequences=True,
        )

        # Probabilistic reconstruction head
        self.prob_decoder = ProbabilisticDecoderPT(hidden_dim=conv_filters, data_dim=self.data_dim)

        # Init linear layers (BN stats copied by transfer)
        for m in [self.fc0, self.fc1, self.fc2]:
            nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, g: torch.Tensor, x: torch.Tensor):
        """
        g: (B, latent_dim)
        x: (B, W, NNF) or (B, W, N, NF)  -> used only to compute validity mask
        returns: distribution with .mean of shape (B, W, NNF)
        """
        B = g.shape[0]
        # Build validity mask as in TF: logical_not(reduce_all(x == 0, axis=2))
        if x.dim() == 4:
            # (B, W, N, NF) -> (B, W, N*NF)
            x_flat = x.view(x.size(0), x.size(1), -1)
        else:
            x_flat = x
        validity_mask = ~torch.all(x_flat == 0.0, dim=-1)  # (B, W), bool

        # Generator MLP
        z = self.bn0(self.fc0(g))
        z = self.bn1(self.act1(self.fc1(z)))
        z = self.bn2(self.act2(self.fc2(z)))

        # Repeat across time (RepeatVector)
        z_rep = z.unsqueeze(1).repeat(1, self.W, 1)  # (B, W, 4*latent)

        # Temporal block
        hidden_seq = self.tcn(z_rep)  # (B, W, conv_filters)

        # Probabilistic reconstruction
        return self.prob_decoder(hidden_seq, validity_mask)

In [4]:
def _tf_conv1d_to_torch(w_keras: np.ndarray) -> torch.Tensor:
    # TF Conv1D: [K, Cin, Cout] -> PT Conv1d: [Cout, Cin, K]
    return torch.from_numpy(np.transpose(w_keras, (2, 1, 0)))


def _load_bn_tf_to_pt(pt_bn: nn.BatchNorm1d, tf_bn: tf.keras.layers.BatchNormalization):
    gamma, beta, moving_mean, moving_var = tf_bn.get_weights()
    pt_bn.weight.data = torch.from_numpy(gamma)
    pt_bn.bias.data = torch.from_numpy(beta)
    pt_bn.running_mean.data = torch.from_numpy(moving_mean)
    pt_bn.running_var.data = torch.from_numpy(moving_var)


def _collect_tcn_sublayers(tf_tcn_layer: tf.keras.layers.Layer):
    from tensorflow.keras.layers import Conv1D, BatchNormalization
    convs = [m for m in tf_tcn_layer.submodules if isinstance(m, Conv1D)]
    bns = [m for m in tf_tcn_layer.submodules if isinstance(m, BatchNormalization)]
    return convs, bns


def _is_1x1(conv: tf.keras.layers.Conv1D) -> bool:
    ks = conv.kernel_size
    ks = ks[0] if isinstance(ks, tuple) else ks
    return ks == 1


def transfer_decoder_front_mlp_weights(tf_model: tf.keras.Model, pt_model: TCNDecoderPT):
    """
    Map the three Dense + three BN layers before RepeatVector in order.
    TF sequence: Dense(latent) -> BN -> Dense(2*latent, relu) -> BN -> Dense(4*latent, relu) -> BN
    PT: fc0/bn0, fc1/bn1, fc2/bn2 (with relu applied before BN for fc1,fc2).
    """
    from tensorflow.keras.layers import Dense, BatchNormalization
    denses = [l for l in tf_model.layers if isinstance(l, Dense)]
    bns = [l for l in tf_model.layers if isinstance(l, BatchNormalization)]

    # Use the first three Dense and first three BN layers encountered (decoder-local)
    d0, d1, d2 = denses[0], denses[1], denses[2]
    bn0, bn1, bn2 = bns[0], bns[1], bns[2]

    # Dense 0
    w, b = d0.get_weights()
    pt_model.fc0.weight.data = torch.from_numpy(w.T)
    pt_model.fc0.bias.data = torch.from_numpy(b)
    _load_bn_tf_to_pt(pt_model.bn0, bn0)

    # Dense 1
    w, b = d1.get_weights()
    pt_model.fc1.weight.data = torch.from_numpy(w.T)
    pt_model.fc1.bias.data = torch.from_numpy(b)
    _load_bn_tf_to_pt(pt_model.bn1, bn1)

    # Dense 2
    w, b = d2.get_weights()
    pt_model.fc2.weight.data = torch.from_numpy(w.T)
    pt_model.fc2.bias.data = torch.from_numpy(b)
    _load_bn_tf_to_pt(pt_model.bn2, bn2)


def transfer_tcn_weights(tf_tcn_layer: tcn_pkg.TCN, pt_tcn: TCN1DPT):
    """
    Transfer weights from a Keras tcn.TCN layer to our TCN1DPT (no TimeDistributed).
    - Maps per-block conv1/conv2 + BN
    - Maps residual 1x1 projection(s) if present
    """
    convs, bns = _collect_tcn_sublayers(tf_tcn_layer)
    block_convs = [c for c in convs if not _is_1x1(c)]
    conv1x1 = [c for c in convs if _is_1x1(c)]
    num_blocks = len(pt_tcn.blocks)

    assert len(block_convs) == 2 * num_blocks, f"Conv count mismatch: TF block convs={len(block_convs)}, PT blocks={num_blocks}"

    use_bn = isinstance(pt_tcn.blocks[0].bn1, nn.BatchNorm1d)
    if use_bn:
        assert len(bns) >= 2 * num_blocks, f"BN count mismatch: TF BNs={len(bns)}, expected >= {2 * num_blocks}"

    # Load block convs and BN params
    for i, blk in enumerate(pt_tcn.blocks):
        k1, b1 = block_convs[2 * i].get_weights()
        blk.conv1.weight.data = _tf_conv1d_to_torch(k1)
        blk.conv1.bias.data = torch.from_numpy(b1)

        k2, b2 = block_convs[2 * i + 1].get_weights()
        blk.conv2.weight.data = _tf_conv1d_to_torch(k2)
        blk.conv2.bias.data = torch.from_numpy(b2)

        if use_bn:
            _load_bn_tf_to_pt(blk.bn1, bns[2 * i])
            _load_bn_tf_to_pt(blk.bn2, bns[2 * i + 1])

    # Residual projections if any (first blocks typically)
    proj_idx = 0
    for blk in pt_tcn.blocks:
        if isinstance(getattr(blk, "downsample", None), nn.Conv1d):
            rk, rb = conv1x1[proj_idx].get_weights()
            blk.downsample.weight.data = _tf_conv1d_to_torch(rk)
            blk.downsample.bias.data = torch.from_numpy(rb)
            proj_idx += 1


def transfer_prob_decoder_weights(tf_prob_layer: ProbabilisticDecoderTF, pt_prob: ProbabilisticDecoderPT):
    """
    Copy the final projection Dense from TF ProbabilisticDecoder to PT ProbabilisticDecoderPT.loc_projection.
    Assumes TF layer exposes a single Dense with [hidden_dim, data_dim] kernel and bias at get_weights()[:2].
    """
    w, b = tf_prob_layer.get_weights()[:2]
    pt_prob.loc_projection.weight.data = torch.from_numpy(w.T)
    pt_prob.loc_projection.bias.data = torch.from_numpy(b)


def transfer_tcn_decoder_weights(tf_model: tf.keras.Model, pt_model: TCNDecoderPT):
    """
    Orchestrates the full TF -> PT transfer for the decoder:
      - Front MLP (3x Dense + 3x BN)
      - TCN
      - Probabilistic head projection
    """
    # 1) Front MLP
    transfer_decoder_front_mlp_weights(tf_model, pt_model)

    # 2) TCN
    tf_tcn_layer = next(l for l in tf_model.layers if isinstance(l, tcn_pkg.TCN))
    transfer_tcn_weights(tf_tcn_layer, pt_model.tcn)

    # 3) Probabilistic head
    tf_prob_layer = next(l for l in tf_model.layers if isinstance(l, ProbabilisticDecoderTF))
    transfer_prob_decoder_weights(tf_prob_layer, pt_model.prob_decoder)

In [5]:
import unittest
import time

# Assume get_TCN_decoder is available in your environment (as per your note)


class TestTCNDecoderPT(unittest.TestCase):
    def setUp(self):
        tf.keras.backend.clear_session()

        # Shapes (use your conventions)
        self.B = 32
        self.W = 25
        self.N = 11
        self.NF = 3
        self.NNF = self.N * self.NF
        self.latent_dim = 6

        # Decoder params (keep small for speed; mirror TF defaults where relevant)
        self.conv_filters = 32
        self.kernel_size = 4
        self.conv_stacks = 1
        self.conv_dilations = (8, 4, 2, 1)
        self.padding = "causal"
        self.use_skip = True
        self.dropout = 0.0
        self.activation = "relu"

        # Random inputs
        rng = np.random.default_rng(0)
        self.g_np = rng.normal(size=(self.B, self.latent_dim)).astype(np.float32)
        self.x_np = rng.normal(size=(self.B, self.W, self.NNF)).astype(np.float32)

        # Inject some zero windows to exercise the mask path
        mask_rows = rng.integers(0, self.B, size=self.B // 8)
        mask_ts = rng.integers(0, self.W, size=self.B // 8)
        self.x_np[mask_rows, mask_ts, :] = 0.0

    def test_tcn_decoder_full_parity(self):
        # Build TF and PT models
        tf_model = get_TCN_decoder(
            input_shape=(self.W, self.NNF),
            latent_dim=self.latent_dim,
            conv_filters=self.conv_filters,
            kernel_size=self.kernel_size,
            conv_stacks=self.conv_stacks,
            conv_dilations=self.conv_dilations,
            padding=self.padding,
            use_skip_connections=self.use_skip,
            dropout_rate=self.dropout,
            activation=self.activation,
        )
        pt_model = TCNDecoderPT(
            input_shape=(self.W, self.NNF),
            latent_dim=self.latent_dim,
            conv_filters=self.conv_filters,
            kernel_size=self.kernel_size,
            conv_stacks=self.conv_stacks,
            conv_dilations=self.conv_dilations,
            padding=self.padding,
            use_skip_connections=self.use_skip,
            dropout_rate=self.dropout,
            activation=self.activation,
        )
        pt_model.eval()

        # Transfer weights
        transfer_tcn_decoder_weights(tf_model, pt_model)

        # Compare outputs
        t0 = time.time()
        y_tf = tf_model([self.g_np, self.x_np], training=False).mean().numpy() # (B, W, NNF)
        t1 = time.time()

        with torch.no_grad():
            dist_pt = pt_model(torch.from_numpy(self.g_np), torch.from_numpy(self.x_np))
            y_pt = dist_pt.mean.detach().cpu().numpy()  # (B, W, NNF)
        t2 = time.time()

        print("Decoder TF time:", t1 - t0, "PT time:", t2 - t1)
        np.testing.assert_allclose(y_tf, y_pt, rtol=1e-5, atol=2e-4)
        print("✅ TCNDecoderPT end-to-end parity PASSED")

    def test_tcn_only_parity(self):
        """
        Optional: compare TCN outputs (before probabilistic head).
        """
        # Build TF/PT decoders
        tf_model = get_TCN_decoder(
            input_shape=(self.W, self.NNF),
            latent_dim=self.latent_dim,
            conv_filters=self.conv_filters,
            kernel_size=self.kernel_size,
            conv_stacks=self.conv_stacks,
            conv_dilations=self.conv_dilations,
            padding=self.padding,
            use_skip_connections=self.use_skip,
            dropout_rate=self.dropout,
            activation=self.activation,
        )
        pt_model = TCNDecoderPT(
            input_shape=(self.W, self.NNF),
            latent_dim=self.latent_dim,
            conv_filters=self.conv_filters,
            kernel_size=self.kernel_size,
            conv_stacks=self.conv_stacks,
            conv_dilations=self.conv_dilations,
            padding=self.padding,
            use_skip_connections=self.use_skip,
            dropout_rate=self.dropout,
            activation=self.activation,
        )
        pt_model.eval()

        # Transfer weights (front MLP + TCN only; prob head not needed for this test)
        transfer_decoder_front_mlp_weights(tf_model, pt_model)
        tf_tcn_layer = next(l for l in tf_model.layers if isinstance(l, tcn_pkg.TCN))
        transfer_tcn_weights(tf_tcn_layer, pt_model.tcn)

        # Build TF submodel up to TCN output
        g_in, x_in = tf_model.inputs
        tf_tcn_sub = tf.keras.Model([g_in, x_in], tf_tcn_layer.output)  # (B, W, conv_filters)

        # Compute TF TCN output
        y_tf_tcn = tf_tcn_sub([self.g_np, self.x_np], training=False).numpy()

        # Compute PT TCN output
        with torch.no_grad():
            g_pt = torch.from_numpy(self.g_np)
            # Front MLP
            z = pt_model.bn0(pt_model.fc0(g_pt))
            z = pt_model.bn1(pt_model.act1(pt_model.fc1(z)))
            z = pt_model.bn2(pt_model.act2(pt_model.fc2(z)))
            # Repeat and TCN
            z_rep = z.unsqueeze(1).repeat(1, self.W, 1)
            y_pt_tcn = pt_model.tcn(z_rep).detach().cpu().numpy()

        np.testing.assert_allclose(y_tf_tcn, y_pt_tcn, rtol=1e-5, atol=2e-4)
        print("✅ TCN-only (pre-prob head) parity PASSED")


# Run tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestTCNDecoderPT)
runner.run(suite)

test_tcn_decoder_full_parity (__main__.TestTCNDecoderPT) ... ok
test_tcn_only_parity (__main__.TestTCNDecoderPT)
Optional: compare TCN outputs (before probabilistic head). ... 

Decoder TF time: 0.09670162200927734 PT time: 0.011658430099487305
✅ TCNDecoderPT end-to-end parity PASSED


ok

----------------------------------------------------------------------
Ran 2 tests in 0.956s

OK


✅ TCN-only (pre-prob head) parity PASSED


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

# Transformer encoder

In [1]:
# 1) IMPORTS

from typing import Iterable, Tuple, List
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import deepof
from spektral.layers import CensNetConv



# TensorFlow + keras for the reference model and weights
import tensorflow as tf
from tensorflow.keras.layers import Input, TimeDistributed, Bidirectional, GRU, LayerNormalization, Masking
from tensorflow.keras.initializers import he_uniform
from tensorflow.keras.models import Model

# Your TF entry-point we will test against (provided by you)
# from your_module import get_transformer_encoder  # assumed available in the notebook

# We will scan for these types inside the TF Transformer layer
from tensorflow.keras.layers import (
    TimeDistributed,
    Conv1D,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    BatchNormalization,
)

# Your PT graph block (same API as used for TCN encoder)
from deepof.clustering.censNetConv_pt import CensNetConvPT

In [2]:


def get_transformer_encoder(
    input_shape: tuple,
    edge_feature_shape: tuple,
    adjacency_matrix: np.ndarray,
    latent_dim: int,
    use_gnn: bool = True,
    num_layers: int = 4,
    num_heads: int = 64,
    dff: int = 128,
    dropout_rate: float = 0.1,
    interaction_regularization: float = 0.0,
):
    """Build a Transformer encoder.

    Based on https://www.tensorflow.org/text/tutorials/transformer.
    Adapted according to https://academic.oup.com/gigascience/article/8/11/giz134/5626377?login=true
    and https://arxiv.org/abs/1711.03905.

    Args:
        input_shape (tuple): shape of the input data
        edge_feature_shape (tuple): shape of the adjacency matrix to use in the graph attention layers. Should be time x edges x features.
        adjacency_matrix (np.ndarray): adjacency matrix for the mice connectivity graph. Shape should be nodes x nodes.
        latent_dim (int): dimensionality of the latent space
        use_gnn (bool): If True, the encoder uses a graph representation of the input, with coordinates and speeds as node attributes, and distances as edge attributes. If False, a regular 3D tensor is used as input.
        num_layers (int): number of transformer layers to include
        num_heads (int): number of heads of the multi-head-attention layers used on the transformer encoder
        dff (int): dimensionality of the token embeddings
        dropout_rate (float): dropout rate
        interaction_regularization (float): regularization parameter for the interaction features

    """
    # Define feature and adjacency inputs
    x = Input(shape=input_shape)
    a = Input(shape=edge_feature_shape)

    if use_gnn:
        x_reshaped = tf.transpose(
            tf.reshape(
                tf.transpose(x),
                [
                    -1,
                    adjacency_matrix.shape[-1],
                    x.shape[1],
                    input_shape[-1] // adjacency_matrix.shape[-1],
                ][::-1],
            )
        )
        a_reshaped = tf.transpose(
            tf.reshape(
                tf.transpose(a),
                [
                    -1,
                    edge_feature_shape[-1],
                    a.shape[1],
                    1,
                ][::-1],
            )
        )

    else:
        x_reshaped = tf.expand_dims(x, axis=1)

    transformer_embedding = TimeDistributed(
        deepof.clustering.model_utils_new.TransformerEncoder(
            num_layers=num_layers,
            seq_dim=input_shape[-1],
            key_dim=input_shape[-1],
            num_heads=num_heads,
            dff=dff,
            maximum_position_encoding=input_shape[0],
            rate=dropout_rate,
        )
    )(x_reshaped, training=False)
    transformer_embedding = tf.reshape(
        transformer_embedding,
        [
            -1,
            (adjacency_matrix.shape[0] if x_reshaped.shape[1] != 1 else 1),
            input_shape[0] * input_shape[1],
        ],
    )

    if use_gnn:

        # Embed edge features too
        transformer_a_embedding = TimeDistributed(
            deepof.clustering.model_utils_new.TransformerEncoder(
                num_layers=num_layers,
                seq_dim=input_shape[-1],
                key_dim=input_shape[-1],
                num_heads=num_heads,
                dff=dff,
                maximum_position_encoding=input_shape[0],
                rate=dropout_rate,
            )
        )(a_reshaped, training=False)

        transformer_a_embedding = tf.reshape(
            transformer_a_embedding,
            [-1, adjacency_matrix.shape[0], input_shape[0] * input_shape[1]],
        )

        spatial_block = CensNetConv(
            node_channels=latent_dim,
            edge_channels=latent_dim,
            activation="relu",
            node_regularizer=tf.keras.regularizers.l1(interaction_regularization),
        )

        # Process adjacency matrix
        laplacian, edge_laplacian, incidence = spatial_block.preprocess(
            adjacency_matrix
        )

        # Get and concatenate node and edge embeddings
        x_nodes, x_edges = spatial_block(
            [
                transformer_embedding,
                (laplacian, edge_laplacian, incidence),
                transformer_a_embedding,
            ],
            mask=None,
        )

        x_nodes = tf.reshape(
            x_nodes,
            [-1, adjacency_matrix.shape[-1] * latent_dim],
        )

        x_edges = tf.reshape(
            x_edges,
            [-1, edge_feature_shape[-1] * latent_dim],
        )

        transformer_embedding = tf.concat([x_nodes, x_edges], axis=-1)

    else:
        transformer_embedding = tf.squeeze(transformer_embedding, axis=1)

    encoder = tf.keras.layers.Dense(2 * latent_dim, activation="relu")(
        transformer_embedding
    )
    encoder = tf.keras.layers.BatchNormalization()(encoder)
    encoder = tf.keras.layers.Dense(latent_dim, activation="relu")(encoder)
    encoder = tf.keras.layers.BatchNormalization()(encoder)
    encoder = tf.keras.layers.Dense(latent_dim)(encoder)

    return tf.keras.models.Model([x, a], encoder, name="transformer_encoder")

In [3]:
def _act(name: str) -> nn.Module:
    name = (name or "relu").lower()
    if name == "relu": return nn.ReLU()
    if name == "gelu": return nn.GELU()
    if name == "tanh": return nn.Tanh()
    if name == "leaky_relu": return nn.LeakyReLU(0.2)
    if name in {"linear", "identity", "none"}: return nn.Identity()
    raise ValueError(f"Unsupported activation: {name}")


class BatchNorm1dKerasFP32(nn.BatchNorm1d):
    """Keras-like BatchNorm with eps=1e-3 and momentum=0.01 (Keras uses 0.99)."""
    def __init__(self, num_features, eps=1e-3, momentum=0.01, affine=True, track_running_stats=True):
        super().__init__(num_features, eps=eps, momentum=momentum, affine=affine, track_running_stats=track_running_stats)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = super().forward(x.float())
        return y.to(dtype=x.dtype)


def sinusoidal_positional_encoding(max_len: int, d_model: int, device=None, dtype=torch.float32) -> torch.Tensor:
    """Generate sinusoidal positional encodings."""
    pe = torch.zeros(max_len, d_model, dtype=dtype, device=device)
    position = torch.arange(0, max_len, dtype=dtype, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=dtype, device=device) * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    n_odd = pe[:, 1::2].shape[1]
    pe[:, 1::2] = torch.cos(position * div_term)[:, :n_odd]
    return pe.unsqueeze(0)  # (1, max_len, d_model)


class MultiHeadAttentionPT(nn.Module):
    """Multi-head attention layer compatible with Keras implementation."""
    def __init__(self, in_dim: int, num_heads: int, key_dim: int, dropout: float = 0.0):
        super().__init__()
        self.in_dim = int(in_dim)
        self.num_heads = int(num_heads)
        self.key_dim = int(key_dim)
        self.inner_dim = self.num_heads * self.key_dim

        self.q_proj = nn.Linear(self.in_dim, self.inner_dim, bias=True)
        self.k_proj = nn.Linear(self.in_dim, self.inner_dim, bias=True)
        self.v_proj = nn.Linear(self.in_dim, self.inner_dim, bias=True)
        self.out_proj = nn.Linear(self.inner_dim, self.in_dim, bias=True)
        self.dropout = nn.Dropout(dropout)

        for m in [self.q_proj, self.k_proj, self.v_proj, self.out_proj]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor = None) -> torch.Tensor:
        B, T, _ = x.shape

        def proj(linear: nn.Linear):
            y = linear(x)
            return y.reshape(B, T, self.num_heads, self.key_dim).permute(0, 2, 1, 3).contiguous()

        q = proj(self.q_proj)
        k = proj(self.k_proj)
        v = proj(self.v_proj)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.key_dim ** 0.5)
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask.unsqueeze(1).unsqueeze(2), float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        ctx = torch.matmul(attn, v)
        ctx = ctx.permute(0, 2, 1, 3).contiguous().reshape(B, T, self.inner_dim)
        out = self.out_proj(ctx)

        if attn_mask is not None:
            out = out.masked_fill(attn_mask.unsqueeze(-1), 0.0)
        return out


class TransformerEncoderLayerPT(nn.Module):
    """Transformer encoder layer with post-normalization."""
    def __init__(self, key_dim: int, num_heads: int, dff: int, rate: float = 0.1):
        super().__init__()
        self.mha = MultiHeadAttentionPT(in_dim=key_dim, num_heads=num_heads, key_dim=key_dim, dropout=rate)
        self.dropout1 = nn.Dropout(rate)
        self.norm1 = nn.LayerNorm(key_dim, eps=1e-6)

        self.ffn1 = nn.Linear(key_dim, dff)
        self.act = nn.ReLU()
        self.ffn2 = nn.Linear(dff, key_dim)
        self.dropout2 = nn.Dropout(rate)
        self.norm2 = nn.LayerNorm(key_dim, eps=1e-6)

        for m in [self.ffn1, self.ffn2]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor = None) -> torch.Tensor:
        attn_out = self.mha(x, attn_mask=attn_mask)
        x = self.norm1(x + self.dropout1(attn_out))
        ff = self.ffn2(self.act(self.ffn1(x)))
        x = self.norm2(x + self.dropout2(ff))
        return x


class TransformerCorePT(nn.Module):
    """Core transformer: Conv1D embedding -> positional encoding -> transformer layers."""
    def __init__(self, in_channels: int, key_dim: int, num_layers: int, num_heads: int, dff: int, max_pos: int, rate: float = 0.1):
        super().__init__()
        self.key_dim = int(key_dim)
        self.max_pos = int(max_pos)
        self.dropout = nn.Dropout(rate)

        self.embed = nn.Conv1d(in_channels, self.key_dim, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.embed.weight)
        nn.init.zeros_(self.embed.bias)

        self.layers = nn.ModuleList([
            TransformerEncoderLayerPT(key_dim=self.key_dim, num_heads=num_heads, dff=dff, rate=rate) 
            for _ in range(int(num_layers))
        ])

        pe = sinusoidal_positional_encoding(self.max_pos, self.key_dim)
        self.register_buffer("pos_encoding", pe, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        
        # Compute mask for all-zero timesteps
        with torch.no_grad():
            mask = torch.all(x == 0.0, dim=-1)

        # Embedding with Conv1D
        y = self.embed(x.transpose(1, 2)).transpose(1, 2)
        y = F.relu(y)
        y = y * (self.key_dim ** 0.5)

        # Add positional encoding
        if T > self.pos_encoding.size(1):
            self.pos_encoding = sinusoidal_positional_encoding(T, self.key_dim, device=x.device).to(self.pos_encoding.dtype)
        y = y + self.pos_encoding[:, :T, :].to(y.dtype)
        y = self.dropout(y)

        # Apply transformer layers
        for layer in self.layers:
            y = layer(y, attn_mask=mask)
        return y


class TFMEncoderPT(nn.Module):
    """PyTorch implementation of TensorFlow Transformer Encoder with optional GNN."""
    def __init__(
        self,
        input_shape: Tuple[int, int, int],        # (W, N, NF)
        edge_feature_shape: Tuple[int, int, int], # (W, E, EF)
        adjacency_matrix: np.ndarray,
        latent_dim: int,
        use_gnn: bool = True,
        num_layers: int = 4,
        num_heads: int = 8,
        dff: int = 128,
        dropout_rate: float = 0.1,
    ):
        super().__init__()
        self.use_gnn = use_gnn
        self.latent_dim = int(latent_dim)
        self.W, self.N, self.NF = input_shape
        _, self.E, self.EF = edge_feature_shape
        assert adjacency_matrix.shape[0] == self.N == adjacency_matrix.shape[1], "Adjacency must be NxN"

        key_dim = self.N * self.NF

        if use_gnn:
            # Node transformer
            self.node_tf = TransformerCorePT(
                in_channels=self.NF, key_dim=key_dim,
                num_layers=num_layers, num_heads=num_heads, dff=dff, max_pos=self.W, rate=dropout_rate
            )
            # Edge transformer
            self.edge_tf = TransformerCorePT(
                in_channels=1, key_dim=key_dim,
                num_layers=num_layers, num_heads=num_heads, dff=dff, max_pos=self.W, rate=dropout_rate
            )

            # Spatial GNN
            self.spatial_gnn = CensNetConvPT(node_channels=self.latent_dim, edge_channels=self.latent_dim, activation="relu")
            lap, edge_lap, inc = self.spatial_gnn.preprocess(torch.tensor(adjacency_matrix))
            self.register_buffer("laplacian", lap.float())
            self.register_buffer("edge_laplacian", edge_lap.float())
            self.register_buffer("incidence", inc.float())

            final_in = 2 * self.N * self.latent_dim
        else:
            # Single transformer for flattened input
            self.flat_tf = TransformerCorePT(
                in_channels=self.N * self.NF, key_dim=key_dim,
                num_layers=num_layers, num_heads=num_heads, dff=dff, max_pos=self.W, rate=dropout_rate
            )
            final_in = self.W * self.N * self.NF

        # MLP head
        self.head = nn.Sequential(
            nn.Linear(final_in, 2 * self.latent_dim),
            nn.ReLU(),
            BatchNorm1dKerasFP32(2 * self.latent_dim, eps=1e-3),
            nn.Linear(2 * self.latent_dim, self.latent_dim),
            nn.ReLU(),
            BatchNorm1dKerasFP32(self.latent_dim, eps=1e-3),
            nn.Linear(self.latent_dim, self.latent_dim),
        )
        
        # Initialize head weights
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        B, W, N, NF = x.shape
        B, W, E, EF = a.shape
        assert (W, N, NF) == (self.W, self.N, self.NF)

        if self.use_gnn:
            # Process nodes using TF's transpose-reshape-transpose pattern
            x_flat = x.view(B, W, N * NF)
            x_transposed = x_flat.permute(2, 1, 0)  # (N*NF, W, B)
            x_reshaped = x_transposed.reshape(NF, W, N, B)  # (NF, W, N, B)
            x_nodes = x_reshaped.permute(3, 2, 1, 0)  # (B, N, W, NF)
            
            node_in = x_nodes.reshape(B * N, W, NF)
            node_out = self.node_tf(node_in).view(B, N, W, -1)
            nodes_flat = node_out.reshape(B, N, W * (self.N * self.NF))

            # Process edges using TF's transpose-reshape-transpose pattern
            EEF = E * EF
            a_flat = a.view(B, W, EEF)
            a_transposed = a_flat.permute(2, 1, 0)  # (EEF, W, B)
            a_reshaped = a_transposed.reshape(1, W, EEF, B)  # (1, W, EEF, B)
            a_edges = a_reshaped.permute(3, 2, 1, 0)  # (B, EEF, W, 1)
            
            edge_in = a_edges.reshape(B * EEF, W, 1)
            edge_out = self.edge_tf(edge_in).view(B, EEF, W, -1)
            edges_flat = edge_out.reshape(B, self.N, W * (self.N * self.NF))
                    
            # Apply spatial GNN
            x_nodes_g, x_edges_g = self.spatial_gnn([
                nodes_flat, (self.laplacian, self.edge_laplacian, self.incidence), edges_flat
            ])

            # Concatenate node and edge features
            enc = torch.cat([x_nodes_g, x_edges_g], dim=1).reshape(B, -1)
            
        else:
            # Non-GNN path: simple transformer on flattened input
            x_flat = x.view(B, W, N * NF)
            seq_out = self.flat_tf(x_flat)
            enc = seq_out.reshape(B, -1)

        # Apply MLP head
        out = self.head(enc.float()).to(enc.dtype)
        return out

In [4]:
# 3) WEIGHT TRANSFER UTILITIES (TF -> PT)

def _tf_conv1d_to_torch(w_keras: np.ndarray) -> torch.Tensor:
    # TF Conv1D [K, Cin, Cout] -> PT Conv1d [Cout, Cin, K]
    return torch.from_numpy(np.transpose(w_keras, (2, 1, 0)))


def _transfer_layernorm(tf_ln: LayerNormalization, pt_ln: nn.LayerNorm):
    gamma, beta = tf_ln.get_weights()
    pt_ln.weight.data = torch.from_numpy(gamma)
    pt_ln.bias.data = torch.from_numpy(beta)


def _flatten_qkv_kernel_bias(w: np.ndarray, b: np.ndarray, in_dim: int, num_heads: int, key_dim: int):
    """
    Returns w2: (in_dim, H*K), b2: (H*K,)
    Accepts Keras MHA kernels that may be 2D or 3D.
    """
    # Kernel
    if w.ndim == 2:
        # (in_dim, H*K) already
        w2 = w
    elif w.ndim == 3:
        # Common layouts:
        # 1) (in_dim, num_heads, key_dim)
        # 2) (num_heads, in_dim, key_dim)
        # 3) (in_dim, key_dim, num_heads)
        if w.shape == (in_dim, num_heads, key_dim):
            w2 = w.reshape(in_dim, num_heads * key_dim)
        elif w.shape == (num_heads, in_dim, key_dim):
            w2 = np.transpose(w, (1, 0, 2)).reshape(in_dim, num_heads * key_dim)
        elif w.shape == (in_dim, key_dim, num_heads):
            w2 = np.transpose(w, (0, 2, 1)).reshape(in_dim, num_heads * key_dim)
        else:
            raise ValueError(f"Unexpected qkv kernel shape: {w.shape}")
    else:
        raise ValueError(f"Unexpected qkv kernel rank: {w.ndim}")

    # Bias
    if b is None:
        b2 = None
    elif b.ndim == 1:
        # (H*K,)
        b2 = b
    elif b.ndim == 2:
        # Common: (num_heads, key_dim)
        if b.shape == (num_heads, key_dim):
            b2 = b.reshape(num_heads * key_dim)
        else:
            b2 = b.reshape(-1)
    else:
        b2 = b.reshape(-1)

    return w2, b2


def _flatten_out_kernel_bias(w: np.ndarray, b: np.ndarray, out_dim: int, num_heads: int, key_dim: int):
    """
    Returns w2: (H*K, out_dim), b2: (out_dim,)
    Accepts Keras MHA output kernels that may be 2D or 3D.
    """
    inner = num_heads * key_dim
    if w.ndim == 2:
        # (H*K, out_dim)
        w2 = w
    elif w.ndim == 3:
        # Common layouts:
        # 1) (num_heads, key_dim, out_dim)
        # 2) (key_dim, num_heads, out_dim)
        if w.shape == (num_heads, key_dim, out_dim):
            w2 = w.reshape(inner, out_dim)
        elif w.shape == (key_dim, num_heads, out_dim):
            w2 = np.transpose(w, (1, 0, 2)).reshape(inner, out_dim)
        else:
            raise ValueError(f"Unexpected out kernel shape: {w.shape}")
    else:
        raise ValueError(f"Unexpected out kernel rank: {w.ndim}")

    # Bias
    if b is None:
        b2 = None
    elif b.ndim == 1:
        # (out_dim,)
        b2 = b
    else:
        b2 = b.reshape(-1)

    return w2, b2


def _transfer_mha_keras_to_pt(tf_mha: MultiHeadAttention, pt_mha: MultiHeadAttentionPT):
    """
    Robustly map Keras MHA Dense kernels/biases to our PT MHA, flattening 3D tensors if needed.
    """
    def get_dense(obj, primary, fallback):
        d = getattr(obj, primary, None)
        if d is None:
            d = getattr(obj, fallback, None)
        return d

    qd = get_dense(tf_mha, "query_dense", "_query_dense")
    kd = get_dense(tf_mha, "key_dense", "_key_dense")
    vd = get_dense(tf_mha, "value_dense", "_value_dense")
    od = get_dense(tf_mha, "output_dense", "_output_dense")
    assert all([qd, kd, vd, od]), "Could not find Keras MHA Dense sublayers (query/key/value/output)."

    in_dim = pt_mha.in_dim
    H = pt_mha.num_heads
    K = pt_mha.key_dim
    inner = H * K

    # q
    Wq, bq = qd.get_weights()
    Wq2, bq2 = _flatten_qkv_kernel_bias(Wq, bq, in_dim, H, K)
    pt_mha.q_proj.weight.data = torch.from_numpy(Wq2.T)  # (inner, in_dim)
    pt_mha.q_proj.bias.data = torch.from_numpy(bq2) if bq2 is not None else torch.zeros(inner)

    # k
    Wk, bk = kd.get_weights()
    Wk2, bk2 = _flatten_qkv_kernel_bias(Wk, bk, in_dim, H, K)
    pt_mha.k_proj.weight.data = torch.from_numpy(Wk2.T)
    pt_mha.k_proj.bias.data = torch.from_numpy(bk2) if bk2 is not None else torch.zeros(inner)

    # v
    Wv, bv = vd.get_weights()
    Wv2, bv2 = _flatten_qkv_kernel_bias(Wv, bv, in_dim, H, K)
    pt_mha.v_proj.weight.data = torch.from_numpy(Wv2.T)
    pt_mha.v_proj.bias.data = torch.from_numpy(bv2) if bv2 is not None else torch.zeros(inner)

    # out
    Wo, bo = od.get_weights()
    Wo2, bo2 = _flatten_out_kernel_bias(Wo, bo, in_dim, H, K)  # (inner, in_dim_out==in_dim)
    pt_mha.out_proj.weight.data = torch.from_numpy(Wo2.T)      # (in_dim, inner)
    pt_mha.out_proj.bias.data = torch.from_numpy(bo2) if bo2 is not None else torch.zeros(in_dim)


def _collect_tf_te(tf_te_layer):
    """Collect TF sublayers from TransformerEncoder (inner of TimeDistributed)."""
    convs = [m for m in tf_te_layer.submodules if isinstance(m, Conv1D)]
    enc_layers = list(getattr(tf_te_layer, "enc_layers"))
    return convs, enc_layers


def transfer_td_transformer_weights(tf_td: TimeDistributed, pt_core: TransformerCorePT):
    assert isinstance(tf_td, TimeDistributed), "Expected a TimeDistributed layer"
    tf_te = tf_td.layer  # inner TransformerEncoder

    convs, enc_layers = _collect_tf_te(tf_te)
    assert len(convs) >= 1, "No Conv1D embedding found in TF transformer"
    k, b = convs[0].get_weights()
    pt_core.embed.weight.data = _tf_conv1d_to_torch(k)
    pt_core.embed.bias.data = torch.from_numpy(b)

    assert len(enc_layers) == len(pt_core.layers), "Transformer layer count mismatch"
    for i, (tf_el, pt_el) in enumerate(zip(enc_layers, pt_core.layers)):
        _transfer_mha_keras_to_pt(tf_el.mha, pt_el.mha)
        # FFN Dense
        d1, d2 = tf_el.ffn.layers  # Dense(dff, relu), Dense(key_dim)
        W1, B1 = d1.get_weights(); W2, B2 = d2.get_weights()
        pt_el.ffn1.weight.data = torch.from_numpy(W1.T); pt_el.ffn1.bias.data = torch.from_numpy(B1)
        pt_el.ffn2.weight.data = torch.from_numpy(W2.T); pt_el.ffn2.bias.data = torch.from_numpy(B2)
        # LayerNorms
        _transfer_layernorm(tf_el.layernorm1, pt_el.norm1)
        _transfer_layernorm(tf_el.layernorm2, pt_el.norm2)


def transfer_censnet_weights(tf_layer, pt_layer: CensNetConvPT):
    """
    Transfer CensNetConv weights from TF to PyTorch.
    The TF layer returns weights in this actual order (despite misleading names):
    [node_kernel, edge_kernel, node_weights, edge_weights, node_bias, edge_bias]
    """
    weights = tf_layer.get_weights()
    
    # Map based on actual shapes, not the misleading variable names
    pt_layer.node_kernel.data = torch.from_numpy(weights[0])   # (825, 6)
    pt_layer.edge_kernel.data = torch.from_numpy(weights[1])   # (825, 6) 
    pt_layer.node_weights.data = torch.from_numpy(weights[2])  # (825, 1)
    pt_layer.edge_weights.data = torch.from_numpy(weights[3])  # (825, 1)
    pt_layer.node_bias.data = torch.from_numpy(weights[4])     # (6,)
    pt_layer.edge_bias.data = torch.from_numpy(weights[5])     # (6,)


def transfer_head_mlp(tf_model: tf.keras.Model, pt_head: nn.Sequential):
    # Dense(2*latent)->BN->Dense(latent)->BN->Dense(latent)
    tail = [l for l in tf_model.layers if isinstance(l, (Dense, BatchNormalization))]
    d1, bn1, d2, bn2, d3 = tail[-5:]

    lin1: nn.Linear = pt_head[0]; bn1_pt: BatchNorm1dKerasFP32 = pt_head[2]
    lin2: nn.Linear = pt_head[3]; bn2_pt: BatchNorm1dKerasFP32 = pt_head[5]
    lin3: nn.Linear = pt_head[6]

    # Dense 1
    w, b = d1.get_weights(); lin1.weight.data = torch.from_numpy(w.T); lin1.bias.data = torch.from_numpy(b)
    # BN 1
    gamma, beta, mmean, mvar = bn1.get_weights()
    bn1_pt.weight.data = torch.from_numpy(gamma); bn1_pt.bias.data = torch.from_numpy(beta)
    bn1_pt.running_mean.data = torch.from_numpy(mmean); bn1_pt.running_var.data = torch.from_numpy(mvar)
    # Dense 2
    w, b = d2.get_weights(); lin2.weight.data = torch.from_numpy(w.T); lin2.bias.data = torch.from_numpy(b)
    # BN 2
    gamma, beta, mmean, mvar = bn2.get_weights()
    bn2_pt.weight.data = torch.from_numpy(gamma); bn2_pt.bias.data = torch.from_numpy(beta)
    bn2_pt.running_mean.data = torch.from_numpy(mmean); bn2_pt.running_var.data = torch.from_numpy(mvar)
    # Dense 3
    w, b = d3.get_weights(); lin3.weight.data = torch.from_numpy(w.T); lin3.bias.data = torch.from_numpy(b)


def transfer_transformer_encoder_weights(tf_model: tf.keras.Model, pt_model: TFMEncoderPT, use_gnn: bool):
    # Head first
    transfer_head_mlp(tf_model, pt_model.head)

    # Collect all TimeDistributed(TransformerEncoder) layers
    td_layers = [
        l for l in tf_model.layers
        if isinstance(l, TimeDistributed) and hasattr(l.layer, "embedding") and hasattr(l.layer, "enc_layers")
    ]

    if use_gnn:
        assert len(td_layers) >= 2, "Expected node and edge TimeDistributed(TransformerEncoder) for GNN=True"

        # Identify which TD is nodes vs edges by inspecting the Conv1D embedding in_channels
        def td_in_channels(td):
            tf_te = td.layer
            convs, enc_layers = _collect_tf_te(tf_te)  # reuse your helper
            assert len(convs) >= 1, "No Conv1D embedding found in TF transformer"
            k, _ = convs[0].get_weights()  # k shape: (kernel_size=1, in_channels, out_channels)
            return int(k.shape[1])

        # Keras TD order can vary; determine by in_channels
        td_info = [(td, td_in_channels(td)) for td in td_layers]
        # Nodes TD: in_channels == NF; Edges TD: in_channels == 1 (since TF reshapes edges to last dim 1)
        nodes_td = next(td for td, in_ch in td_info if in_ch == pt_model.NF)
        edges_td = next(td for td, in_ch in td_info if in_ch == 1)

        # Transfer weights into the correct PT cores
        transfer_td_transformer_weights(nodes_td, pt_model.node_tf)
        transfer_td_transformer_weights(edges_td, pt_model.edge_tf)

        # Ensure CensNetConvPT is built before weight copy (warm-up if needed)
        needs_build = any(
            getattr(pt_model.spatial_gnn, name, None) is None
            for name in ("node_kernel", "edge_kernel", "node_weights", "edge_weights", "node_bias", "edge_bias")
        )
        if needs_build:
            with torch.no_grad():
                B = 2
                W, N, NF = pt_model.W, pt_model.N, pt_model.NF
                E, EF = pt_model.E, pt_model.EF
                x_dummy = torch.zeros(B, W, N, NF)
                a_dummy = torch.zeros(B, W, E, EF)
                _ = pt_model(x_dummy, a_dummy)

        # Copy CensNetConv weights
        from deepof.clustering.model_utils_new import CensNetConv as CensNetConvTF
        gnn_layer = next(l for l in tf_model.layers if isinstance(l, CensNetConvTF))
        transfer_censnet_weights(gnn_layer, pt_model.spatial_gnn)

    else:
        assert len(td_layers) >= 1, "Expected one TimeDistributed(TransformerEncoder) for non-GNN"
        transfer_td_transformer_weights(td_layers[0], pt_model.flat_tf)

In [ ]:
import numpy as np
import torch
import tensorflow as tf
from tensorflow.keras.layers import Dense

try:
    from scipy.optimize import linear_sum_assignment
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False


def _extract_tf_prehead(tf_model, x_np, a_np):
    """
    Get the exact TF tensor fed into the first Dense of the head (d1.input).
    x_np: (B, W, N*NF), a_np: (B, W, E*EF)
    """
    denses = [l for l in tf_model.layers if isinstance(l, Dense)]
    d1 = None
    for l in denses:
        if getattr(l, "name", "") == "dense_4":  # based on your model print
            d1 = l
            break
    if d1 is None:
        assert len(denses) >= 3, "Could not find three Dense layers for the head"
        d1 = denses[-3]
    sub = tf.keras.Model(tf_model.inputs, d1.input)
    return sub([x_np, a_np], training=False).numpy()  # (B, D)


def _capture_pt_prehead(pt_model, x_np, a_np):
    """
    Run PT forward once and capture the tensor fed into self.head via a pre-hook.
    Accepts either split dims (B,W,N,NF)/(B,W,E,EF) or flattened (B,W,N*NF)/(B,W,E*EF).
    Returns: (B, D) numpy array
    """
    # reshape flattened -> split if needed
    if x_np.ndim == 3:
        B, W, D = x_np.shape
        N, NF = pt_model.N, pt_model.NF
        assert D == N * NF, f"x_np last dim {D} != N*NF {N*NF}"
        x_np = x_np.reshape(B, W, N, NF)
    if a_np.ndim == 3:
        B, W, D2 = a_np.shape
        E, EF = pt_model.E, pt_model.EF
        assert D2 == E * EF, f"a_np last dim {D2} != E*EF {E*EF}"
        a_np = a_np.reshape(B, W, E, EF)

    captured = {}
    def hook(_mod, inputs):
        captured["enc"] = inputs[0].detach().cpu().numpy()

    h = pt_model.head.register_forward_pre_hook(hook)
    with torch.no_grad():
        _ = pt_model(torch.from_numpy(x_np), torch.from_numpy(a_np))
    h.remove()
    return captured["enc"]  # (B, D)


def _standardize_columns(X):
    """Zero-mean, unit-variance per column."""
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True) + 1e-8
    return (X - mu) / sd


def _learn_perm_from_batch(tf_model, pt_model, x_tf_flat, a_tf_flat, use_cosine=True):
    """
    Find perm such that PT_prehead[:, perm] ~= TF_prehead on one batch.
    x_tf_flat: (B, W, N*NF), a_tf_flat: (B, W, E*EF)
    Returns: torch.LongTensor of shape (D,)
    """
    tf_pre = _extract_tf_prehead(tf_model, x_tf_flat, a_tf_flat)  # (B, D)
    pt_pre = _capture_pt_prehead(pt_model, x_tf_flat, a_tf_flat)  # (B, D)
    B, D = tf_pre.shape

    tf_std = _standardize_columns(tf_pre)
    pt_std = _standardize_columns(pt_pre)

    if use_cosine:
        tf_norm = tf_std / (np.linalg.norm(tf_std, axis=0, keepdims=True) + 1e-8)
        pt_norm = pt_std / (np.linalg.norm(pt_std, axis=0, keepdims=True) + 1e-8)
        sim = tf_norm.T @ pt_norm  # (D, D)
        cost = 1.0 - sim
    else:
        diff = np.abs(tf_pre[:, :, None] - pt_pre[:, None, :])  # (B, D, D)
        cost = diff.mean(axis=0)                                # (D, D)

    if _HAS_SCIPY:
        row_ind, col_ind = linear_sum_assignment(cost)
        perm = np.full(D, -1, dtype=np.int64)
        perm[row_ind] = col_ind
    else:
        # Greedy fallback
        perm = np.full(D, -1, dtype=np.int64)
        used_pt = np.zeros(D, dtype=bool)
        pairs = [(cost[i, j], i, j) for i in range(D) for j in range(D)]
        pairs.sort(key=lambda t: t[0])
        assigned = 0
        for _, i, j in pairs:
            if perm[i] == -1 and not used_pt[j]:
                perm[i] = j
                used_pt[j] = True
                assigned += 1
                if assigned == D:
                    break
        assert (perm >= 0).all(), "Failed to assign a unique PT column to each TF column"

    # Quality check
    pt_matched = pt_pre[:, perm]
    mean_abs = np.mean(np.abs(tf_pre - pt_matched))
    max_abs = np.max(np.abs(tf_pre - pt_matched))
    print(f"Permutation quality -> mean abs diff: {mean_abs:.6g}, max abs diff: {max_abs:.6g}")

    return torch.from_numpy(perm)

def apply_perm_into_head_first_linear(pt_model, perm_idx: torch.LongTensor):
    """
    Compose the learned input permutation into the first Linear of the head:
      new_weight = old_weight[:, perm]
    Bias unchanged. Removes the need to permute activations at runtime.
    """
    lin1 = pt_model.head[0]
    with torch.no_grad():
        lin1.weight.copy_(lin1.weight[:, perm_idx])
    # If you had set prehead_perm for runtime, clear it:
    if hasattr(pt_model, "prehead_perm"):
        pt_model.prehead_perm = None

In [6]:
import unittest, time
import tensorflow as tf
import torch
import numpy as np
from tensorflow.keras.layers import TimeDistributed, Conv1D, Dense, BatchNormalization
from spektral.layers import CensNetConv as CensNetConvTF



class TestTransformerEncoderPT(unittest.TestCase):
    def setUp(self):
        tf.keras.backend.clear_session()
        # Your TCN-style init
        self.batch_size = 128
        self.W = 25
        self.N = 11
        self.NF = 3
        self.E = 11
        self.EF = 1
        self.latent_dim = 6

        # Adjacency
        m = np.zeros((self.N, self.N), dtype=np.float32)
        ui = np.triu_indices(self.N)
        num_possible = len(ui[0])
        c = np.random.choice(num_possible, min(self.E, num_possible), replace=False)
        m[ui[0][c], ui[1][c]] = 1
        self.adj_matrix = (m + m.T).astype(np.float32)

        # Data
        rng = np.random.default_rng(0)
        self.x_np = rng.normal(size=(self.batch_size, self.W, self.N, self.NF)).astype(np.float32)
        self.a_np = rng.normal(size=(self.batch_size, self.W, self.E, self.EF)).astype(np.float32)

        # Transformer params (keep small to avoid OOM)
        self.num_layers = 1
        self.num_heads = 4
        self.dff = 64
        self.dropout = 0.0

    def test_non_gnn_parity(self):
        B, W, N, NF, E, EF = self.batch_size, self.W, self.N, self.NF, self.E, self.EF

        # TF model: IMPORTANT — 2D shapes (W, N*NF) and (W, E*EF)
        tf_model = get_transformer_encoder(
            input_shape=(W, N * NF),
            edge_feature_shape=(W, E * EF),
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=False,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            dff=self.dff,
            dropout_rate=self.dropout,
        )

        # PT model: split dims
        pt_model = TFMEncoderPT(
            input_shape=(W, N, NF),
            edge_feature_shape=(W, E, EF),
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=False,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            dff=self.dff,
            dropout_rate=self.dropout,
        )
        pt_model.eval()

        # Transfer weights
        transfer_transformer_encoder_weights(tf_model, pt_model, use_gnn=False)

        # Inputs to TF are flattened 2D per time step
        x_tf = self.x_np.reshape(B, W, N * NF)
        a_tf = self.a_np.reshape(B, W, E * EF)

        y_tf = tf_model([x_tf, a_tf], training=False).numpy()
        with torch.no_grad():
            y_pt = pt_model(torch.from_numpy(self.x_np), torch.from_numpy(self.a_np)).cpu().numpy()

        np.testing.assert_allclose(y_tf, y_pt, rtol=1e-5, atol=2e-4)
        print("✅ TransformerEncoderPT non-GNN parity PASSED")


    def test_gnn_parity(self):
        B, W, N, NF, E, EF = self.batch_size, self.W, self.N, self.NF, self.E, self.EF

        # TF model: IMPORTANT — 2D shapes (W, N*NF) and (W, E*EF)
        tf.keras.backend.clear_session()
        tf_model = get_transformer_encoder(
            input_shape=(self.W, self.N * self.NF),
            edge_feature_shape=(self.W, self.E * self.EF),
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=True,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            dff=self.dff,
            dropout_rate=self.dropout,
        )
        pt_model = TFMEncoderPT(
            input_shape=(self.W, self.N, self.NF),
            edge_feature_shape=(self.W, self.E, self.EF),
            adjacency_matrix=self.adj_matrix,
            latent_dim=self.latent_dim,
            use_gnn=True,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            dff=self.dff,
            dropout_rate=self.dropout,
        )
        pt_model.eval()

        # Warm up PT model to build parameters
        with torch.no_grad():
            _ = pt_model(torch.from_numpy(self.x_np[:2]), torch.from_numpy(self.a_np[:2]))
        transfer_transformer_encoder_weights(tf_model, pt_model, use_gnn=True)

        # Inputs to TF are flattened 2D per time step
        x_tf = self.x_np.reshape(B, W, N * NF)
        a_tf = self.a_np.reshape(B, W, E * EF)

        y_tf = tf_model([x_tf, a_tf], training=False).numpy()
        with torch.no_grad():
            y_pt = pt_model(torch.from_numpy(self.x_np), torch.from_numpy(self.a_np)).cpu().numpy()

        np.testing.assert_allclose(y_tf, y_pt, rtol=1e-5, atol=2e-4)
        print("✅ TransformerEncoderPT GNN parity PASSED")


# Run tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestTransformerEncoderPT)
runner.run(suite)

test_gnn_parity (__main__.TestTransformerEncoderPT) ... The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
ok
test_non_gnn_parity (__main__.TestTransformerEncoderPT) ... ok

----------------------------------------------------------------------
Ran 2 tests in 1.724s

OK


✅ TransformerEncoderPT GNN parity PASSED
✅ TransformerEncoderPT non-GNN parity PASSED


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

# Transformer decoder

In [1]:
# 1) IMPORTS

from typing import Iterable, Tuple, List, Optional, Dict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import deepof
import deepof.clustering
from spektral.layers import CensNetConv
from deepof.clustering.model_utils_new import ProbabilisticDecoderPT
from deepof.clustering.models_new import sinusoidal_positional_encoding, MultiHeadAttentionPT
import unittest, time




# TensorFlow + keras for the reference model and weights
import tensorflow as tf
from tensorflow.keras.layers import Input, TimeDistributed, Bidirectional, GRU, LayerNormalization, Masking
from tensorflow.keras.initializers import he_uniform
from tensorflow.keras.models import Model

# Your TF entry-point we will test against (provided by you)
# from your_module import get_transformer_encoder  # assumed available in the notebook

# We will scan for these types inside the TF Transformer layer
from tensorflow.keras.layers import (
    TimeDistributed,
    Conv1D,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    BatchNormalization,
)

# Your PT graph block (same API as used for TCN encoder)
from deepof.clustering.censNetConv_pt import CensNetConvPT
from deepof.clustering.models_new import BatchNorm1dKerasFP32

In [2]:
import deepof.clustering.model_utils_new


def create_look_ahead_mask(size: int):
    """Create a triangular matrix containing an increasing amount of ones from left to right on each subsequent row.

    Useful for transformer decoder, which allows it to go through the data in a sequential manner, without taking
    the future into account.

    Args:
        size (int): number of time steps in the sequence

    """
    mask = tf.linalg.band_part(tf.ones((size, size)), -1, 0)
    return tf.cast(mask, tf.float32)


def create_padding_mask(seq: tf.Tensor):
    """Create a padding mask, with zeros where data is missing, and ones where data is available.

    Args:
        seq (tf.Tensor): Sequence to compute the mask on

    """
    seq = tf.cast(tf.math.equal(seq, 0), tf.float32)

    # add extra dimensions to add the padding to the attention logits.
    return tf.cast(1 - seq[:, tf.newaxis, tf.newaxis, :], tf.float32)


def create_masks(inp: tf.Tensor):
    """Given an input sequence, it creates all necessary masks to pass it through the transformer architecture.

    This includes encoder and decoder padding masks, and a look-ahead mask

    Args:
        inp (tf.Tensor): input sequence to create the masks for.

    """
    # Reduces the dimensionality of the mask to remove the feature dimension
    tar = inp[:, :, 0]
    inp = inp[:, :, 0]

    # Encoder padding mask
    enc_padding_mask = create_padding_mask(inp)

    # Used in the 2nd attention block in the decoder.
    # This padding mask is used to mask the encoder outputs.
    dec_padding_mask = create_padding_mask(inp)

    # Used in the 1st attention block in the decoder.
    # It is used to pad and mask future tokens in the input received by
    # the decoder.
    look_ahead_mask = create_look_ahead_mask(tf.shape(tar)[1])
    dec_target_padding_mask = create_padding_mask(tar)
    combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)

    return enc_padding_mask, combined_mask, dec_padding_mask


def get_transformer_decoder(
    input_shape, latent_dim, num_layers=2, num_heads=8, dff=128, dropout_rate=0.1
):
    """Build a Transformer decoder.

    Based on https://www.tensorflow.org/text/tutorials/transformer.
    Adapted according to https://academic.oup.com/gigascience/article/8/11/giz134/5626377?login=true
    and https://arxiv.org/abs/1711.03905.

    Args:
        input_shape (tuple): shape of the input data
        latent_dim (int): dimensionality of the latent space
        num_layers (int): number of transformer layers to include
        num_heads (int): number of heads of the multi-head-attention layers used on the transformer encoder
        dff (int): dimensionality of the token embeddings
        dropout_rate (float): dropout rate

    """
    x = tf.keras.layers.Input(input_shape)
    g = tf.keras.layers.Input([latent_dim])
    validity_mask = tf.math.logical_not(tf.reduce_all(x == 0.0, axis=2))

    generator = tf.keras.layers.Dense(latent_dim)(g)
    generator = tf.keras.layers.BatchNormalization()(generator)
    generator = tf.keras.layers.Dense(2 * latent_dim, activation="relu")(generator)
    generator = tf.keras.layers.BatchNormalization()(generator)
    generator = tf.keras.layers.Dense(4 * latent_dim, activation="relu")(generator)
    generator = tf.keras.layers.BatchNormalization()(generator)
    generator = tf.keras.layers.RepeatVector(input_shape[0])(generator)

    # Get masks for generated output
    _, look_ahead_mask, padding_mask = deepof.clustering.model_utils_new.create_masks(generator)

    (transformer_embedding, attention_weights,) = deepof.clustering.model_utils_new.TransformerDecoder(
        num_layers=num_layers,
        seq_dim=input_shape[-1],
        key_dim=input_shape[-1],
        num_heads=num_heads,
        dff=dff,
        maximum_position_encoding=input_shape[0],
        rate=dropout_rate,
    )(
        x,
        generator,
        training=False,
        look_ahead_mask=look_ahead_mask,
        padding_mask=padding_mask,
    )

    x_decoded = deepof.clustering.model_utils_new.ProbabilisticDecoder(input_shape)(
        [transformer_embedding, validity_mask]
    )

    return tf.keras.models.Model(
        [g, x], [x_decoded, attention_weights], name="transformer_decoder"
    )

In [3]:
# ================================
# PyTorch Transformer Decoder
# ================================

from typing import Tuple, Dict, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reuse your helpers
from deepof.clustering.models_new import sinusoidal_positional_encoding, BatchNorm1dKerasFP32
from deepof.clustering.model_utils_new import ProbabilisticDecoderPT


def create_look_ahead_mask_pt(size: int, device=None, dtype=torch.bool) -> torch.Tensor:
    """
    PyTorch replica of TF create_look_ahead_mask (KEEP mask).
    Returns lower-triangular True (keep), False above diagonal.
    Shape: (T, T) boolean.
    """
    # TF: mask = tf.linalg.band_part(tf.ones((size, size)), -1, 0)
    #     return tf.cast(mask, tf.float32)
    # We return boolean keep mask
    return torch.tril(torch.ones(size, size, dtype=torch.bool, device=device))


def create_masks_pt(inp_3d: torch.Tensor):
    """
    PyTorch replica of TF create_masks for the decoder (KEEP semantics).

    Given inp_3d (B, T, D):
      - tar = inp[:, :, 0]
      - dec_padding_keep = create_padding_mask(tar)  -> (B, 1, 1, T), True = keep (not padded)
      - look_ahead_keep = create_look_ahead_mask(T)  -> (T, T), True = keep (causal lower-tri)
      - combined_keep = maximum(dec_padding_keep, look_ahead_keep) -> (B, 1, T, T), True = keep

    Returns:
      combined_keep: (B, 1, T, T) boolean
      dec_padding_keep: (B, 1, 1, T) boolean
    """
    device = inp_3d.device
    B, T, _ = inp_3d.shape

    # TF: tar = inp[:, :, 0]
    tar = inp_3d[:, :, 0]  # (B, T)

    # TF create_padding_mask: 1 - (seq == 0) -> keep (1) for non-zero, 0 for zero
    # We return boolean keep mask
    dec_padding_keep = (tar != 0).unsqueeze(1).unsqueeze(1)  # (B, 1, 1, T), True = keep

    # TF create_look_ahead_mask: lower-tri ones (keep)
    la_keep = create_look_ahead_mask_pt(T, device=device, dtype=torch.bool)  # (T, T)

    # TF combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)
    # Both are keep masks -> maximum is logical OR of keeps
    combined_keep = (dec_padding_keep | la_keep.unsqueeze(0).unsqueeze(0))  # (B, 1, T, T)

    return combined_keep, dec_padding_keep


def create_padding_mask_pt(seq_1d: torch.Tensor) -> torch.Tensor:
    """
    Padding mask from a (B, T) tensor, True where seq == 0.0.
    Returns shape (B, T) boolean.
    """
    return (seq_1d == 0.0)


class MultiHeadAttentionGeneralPT(nn.Module):
    """
    Multi-head attention with separate in_dims for query and key/value,
    using boolean keep masks (True=keep, False=mask-out).

    Accepted attn_mask shapes:
      - (B, Tk)
      - (B, Tq, Tk)
      - (B, 1, Tq, Tk)
      - (B, H, Tq, Tk)
    """
    def __init__(self, q_in_dim: int, kv_in_dim: int, num_heads: int, key_dim: int, dropout: float = 0.0):
        super().__init__()
        self.q_in = int(q_in_dim)
        self.kv_in = int(kv_in_dim)
        self.num_heads = int(num_heads)
        self.key_dim = int(key_dim)
        self.inner_dim = self.num_heads * self.key_dim

        self.q_proj = nn.Linear(self.q_in, self.inner_dim, bias=True)
        self.k_proj = nn.Linear(self.kv_in, self.inner_dim, bias=True)
        self.v_proj = nn.Linear(self.kv_in, self.inner_dim, bias=True)
        self.out_proj = nn.Linear(self.inner_dim, self.q_in, bias=True)
        self.dropout = nn.Dropout(dropout)

        for m in [self.q_proj, self.k_proj, self.v_proj, self.out_proj]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(
        self,
        query: torch.Tensor,   # (B, Tq, q_in)
        key: torch.Tensor,     # (B, Tk, kv_in)
        value: torch.Tensor,   # (B, Tk, kv_in)
        attn_mask: torch.Tensor = None,   # keep mask
        return_attention_scores: bool = True,
    ):
        B, Tq, _ = query.shape
        Bk, Tk, _ = key.shape
        assert B == Bk, "Batch mismatch between query and key/value"

        def proj(x, linear: nn.Linear):
            y = linear(x)  # (B, T, H*K)
            return y.view(B, -1, self.num_heads, self.key_dim).permute(0, 2, 1, 3).contiguous()  # (B,H,T,K)

        q = proj(query, self.q_proj)   # (B, H, Tq, K)
        k = proj(key, self.k_proj)     # (B, H, Tk, K)
        v = proj(value, self.v_proj)   # (B, H, Tk, K)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.key_dim ** 0.5)  # (B, H, Tq, Tk)

        if attn_mask is not None:
            # Normalize keep mask to (B, H, Tq, Tk)
            if attn_mask.ndim == 2:        # (B, Tk)
                keep = attn_mask[:, None, None, :]                     # (B,1,1,Tk)
                keep = keep.expand(B, self.num_heads, Tq, Tk)          # (B,H,Tq,Tk)
            elif attn_mask.ndim == 3:      # (B, Tq, Tk)
                keep = attn_mask[:, None, :, :]                        # (B,1,Tq,Tk)
                keep = keep.expand(B, self.num_heads, Tq, Tk)          # (B,H,Tq,Tk)
            elif attn_mask.ndim == 4:      # (B, 1 or H, Tq, Tk)
                keep = attn_mask
                if keep.size(1) == 1:
                    keep = keep.expand(B, self.num_heads, Tq, Tk)
            else:
                raise ValueError(f"Unsupported attn_mask shape: {attn_mask.shape}")

            if keep.dtype != torch.bool:
                keep = keep > 0
            keep = keep.to(scores.device)

            # Mask logits where keep=False
            scores = scores.masked_fill(~keep, -1e9)

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        ctx = torch.matmul(attn, v)  # (B,H,Tq,K)
        ctx = ctx.permute(0, 2, 1, 3).contiguous().view(B, Tq, self.inner_dim)
        out = self.out_proj(ctx)  # (B,Tq,q_in)

        if return_attention_scores:
            return out, attn
        else:
            return out


class TransformerDecoderLayerPT(nn.Module):
    def __init__(self, model_dim: int, memory_dim: int, num_heads: int, dff: int, rate: float = 0.1):
        super().__init__()
        self.mha1 = MultiHeadAttentionGeneralPT(q_in_dim=model_dim, kv_in_dim=model_dim, num_heads=num_heads, key_dim=model_dim, dropout=rate)
        self.dropout1 = nn.Dropout(rate)
        self.norm1 = nn.LayerNorm(model_dim, eps=1e-6)

        self.mha2 = MultiHeadAttentionGeneralPT(q_in_dim=model_dim, kv_in_dim=memory_dim, num_heads=num_heads, key_dim=model_dim, dropout=rate)
        self.dropout2 = nn.Dropout(rate)
        self.norm2 = nn.LayerNorm(model_dim, eps=1e-6)

        self.ffn1 = nn.Linear(model_dim, dff)
        self.act = nn.ReLU()
        self.ffn2 = nn.Linear(dff, model_dim)
        self.dropout3 = nn.Dropout(rate)
        self.norm3 = nn.LayerNorm(model_dim, eps=1e-6)

        for m in [self.ffn1, self.ffn2]:
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(
        self,
        x: torch.Tensor,                # (B, T, model_dim)
        memory: torch.Tensor,           # (B, T, memory_dim)
        look_ahead_mask_3d: torch.Tensor,  # (B, Tq, Tk) True=masked-out
        padding_mask_2d: torch.Tensor,     # (B, Tk) True=masked-out
        training: bool = False,
    ):
        # Self-attention
        attn1, w1 = self.mha1(query=x, key=x, value=x, attn_mask=look_ahead_mask_3d, return_attention_scores=True)
        x = self.norm1(x + self.dropout1(attn1))

        # Cross-attention
        attn2, w2 = self.mha2(query=x, key=memory, value=memory, attn_mask=padding_mask_2d, return_attention_scores=True)
        x = self.norm2(x + self.dropout2(attn2))

        # FFN
        ffn_out = self.ffn2(self.act(self.ffn1(x)))
        x = self.norm3(x + self.dropout3(ffn_out))
        return x, w1, w2


class DecoderCorePT(nn.Module):
    def __init__(self, model_dim: int, memory_dim: int, num_layers: int, num_heads: int, dff: int, max_pos: int, rate: float = 0.1):
        super().__init__()
        self.model_dim = int(model_dim)
        self.memory_dim = int(memory_dim)
        self.max_pos = int(max_pos)
        self.dropout = nn.Dropout(rate)

        self.embed = nn.Conv1d(self.model_dim, self.model_dim, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.embed.weight)
        nn.init.zeros_(self.embed.bias)

        self.layers = nn.ModuleList([
            TransformerDecoderLayerPT(model_dim=self.model_dim, memory_dim=self.memory_dim, num_heads=num_heads, dff=dff, rate=rate)
            for _ in range(int(num_layers))
        ])

        pe = sinusoidal_positional_encoding(self.max_pos, self.model_dim)
        self.register_buffer("pos_encoding", pe, persistent=False)

    def forward(
        self,
        x: torch.Tensor,                # (B, T, model_dim)
        memory: torch.Tensor,           # (B, T, memory_dim)
        look_ahead_mask_3d: torch.Tensor,  # (B, T, T) True=masked-out
        padding_mask_2d: torch.Tensor,     # (B, T) True=masked-out
        training: bool = False,
    ):
        B, T, _ = x.shape

        y = self.embed(x.transpose(1, 2)).transpose(1, 2)
        y = torch.relu(y)
        y = y * (self.model_dim ** 0.5)

        if T > self.pos_encoding.size(1):
            self.pos_encoding = sinusoidal_positional_encoding(T, self.model_dim, device=x.device).to(self.pos_encoding.dtype)
        y = y + self.pos_encoding[:, :T, :].to(y.dtype)
        y = self.dropout(y)

        attention_weights = {}
        out = y
        for i, layer in enumerate(self.layers, start=1):
            out, w1, w2 = layer(out, memory, look_ahead_mask_3d, padding_mask_2d, training=training)
            attention_weights[f"decoder_layer{i}_block1"] = w1
            attention_weights[f"decoder_layer{i}_block2"] = w2

        return out, attention_weights


class TFMDecoderPT(nn.Module):
    """
    Full PyTorch translation of the TF Transformer decoder:
      - Generator MLP from z (latent) -> memory (RepeatVector)
      - Transformer decoder core over input x using memory
      - ProbabilisticDecoderPT to produce mean-output masked by validity
    """
    def __init__(
        self,
        input_shape: Tuple[int, int],  # (W, D_in) where D_in = N*NF (flattened like TF)
        latent_dim: int,
        num_layers: int = 2,
        num_heads: int = 8,
        dff: int = 128,
        dropout_rate: float = 0.1,
    ):
        super().__init__()
        self.W, self.D_in = input_shape
        self.latent_dim = int(latent_dim)
        self.model_dim = self.D_in
        self.memory_dim = 4 * self.latent_dim  # final Dense in TF generator

        # Generator MLP: Dense(latent) -> BN -> Dense(2*latent, relu) -> BN -> Dense(4*latent, relu) -> BN
        self.generator_mlp = nn.Sequential(
            nn.Linear(self.latent_dim, self.latent_dim),
            BatchNorm1dKerasFP32(self.latent_dim, eps=1e-3),
            nn.Linear(self.latent_dim, 2 * self.latent_dim),
            nn.ReLU(),
            BatchNorm1dKerasFP32(2 * self.latent_dim, eps=1e-3),
            nn.Linear(2 * self.latent_dim, 4 * self.latent_dim),
            nn.ReLU(),
            BatchNorm1dKerasFP32(4 * self.latent_dim, eps=1e-3),
        )
        for m in self.generator_mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

        # Decoder core (embedding + layers)
        self.decoder = DecoderCorePT(
            model_dim=self.model_dim,
            memory_dim=self.memory_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            dff=dff,
            max_pos=self.W,
            rate=dropout_rate,
        )

        # Probabilistic decoder to map to data space (D_in)
        self.prob_decoder = ProbabilisticDecoderPT(hidden_dim=self.model_dim, data_dim=self.D_in)

    def forward(self, g: torch.Tensor, x: torch.Tensor, training: bool = False):
        B, W, D = x.shape
        assert (W, D) == (self.W, self.D_in)

        # validity mask: True for valid steps (NOT all-zero features)
        with torch.no_grad():
            validity_mask = torch.logical_not(torch.all(x == 0.0, dim=-1))  # (B, W), bool

        # Generator MLP + repeat across time
        gen = self.generator_mlp(g.float()).to(x.dtype)                    # (B, 4*latent)
        memory = gen.unsqueeze(1).expand(-1, self.W, -1).contiguous()      # (B, W, memory_dim)

        # TF-style mask-out masks (3D/2D)
        combined_mask_3d, pad_mask_2d = create_masks_pt(memory)            # (B,T,T), (B,T), True=masked-out

        # Decoder pass
        hidden, attention_weights = self.decoder(x, memory, combined_mask_3d, pad_mask_2d, training=training)

        # Probabilistic output (mean)
        dist = self.prob_decoder(hidden, validity_mask)
        x_decoded = dist.mean
        return x_decoded, attention_weights

In [4]:
# =======================================
# Transfer weights TF -> PT (Decoder)
# =======================================

import tensorflow as tf
from tensorflow.keras.layers import Conv1D, Dense, BatchNormalization, LayerNormalization, RepeatVector
from tensorflow.keras.layers import MultiHeadAttention as KerasMHA

# Reuse helpers from your encoder utilities if already defined/imported:
# _tf_conv1d_to_torch, _transfer_layernorm, _flatten_qkv_kernel_bias, _flatten_out_kernel_bias

def _tf_conv1d_to_torch(w_keras: np.ndarray) -> torch.Tensor:
    # TF Conv1D [K, Cin, Cout] -> PT Conv1d [Cout, Cin, K]
    return torch.from_numpy(np.transpose(w_keras, (2, 1, 0)))


def _transfer_layernorm(tf_ln: LayerNormalization, pt_ln: nn.LayerNorm):
    gamma, beta = tf_ln.get_weights()
    pt_ln.weight.data = torch.from_numpy(gamma)
    pt_ln.bias.data = torch.from_numpy(beta)

def _flatten_qkv_kernel_bias(w: np.ndarray, b: np.ndarray, in_dim: int, num_heads: int, key_dim: int):
    """
    Returns w2: (in_dim, H*K), b2: (H*K,)
    Accepts Keras MHA kernels that may be 2D or 3D.
    """
    # Kernel
    if w.ndim == 2:
        # (in_dim, H*K) already
        w2 = w
    elif w.ndim == 3:
        # Common layouts:
        # 1) (in_dim, num_heads, key_dim)
        # 2) (num_heads, in_dim, key_dim)
        # 3) (in_dim, key_dim, num_heads)
        if w.shape == (in_dim, num_heads, key_dim):
            w2 = w.reshape(in_dim, num_heads * key_dim)
        elif w.shape == (num_heads, in_dim, key_dim):
            w2 = np.transpose(w, (1, 0, 2)).reshape(in_dim, num_heads * key_dim)
        elif w.shape == (in_dim, key_dim, num_heads):
            w2 = np.transpose(w, (0, 2, 1)).reshape(in_dim, num_heads * key_dim)
        else:
            raise ValueError(f"Unexpected qkv kernel shape: {w.shape}")
    else:
        raise ValueError(f"Unexpected qkv kernel rank: {w.ndim}")

    # Bias
    if b is None:
        b2 = None
    elif b.ndim == 1:
        # (H*K,)
        b2 = b
    elif b.ndim == 2:
        # Common: (num_heads, key_dim)
        if b.shape == (num_heads, key_dim):
            b2 = b.reshape(num_heads * key_dim)
        else:
            b2 = b.reshape(-1)
    else:
        b2 = b.reshape(-1)

    return w2, b2


def _flatten_out_kernel_bias(w: np.ndarray, b: np.ndarray, out_dim: int, num_heads: int, key_dim: int):
    """
    Returns w2: (H*K, out_dim), b2: (out_dim,)
    Accepts Keras MHA output kernels that may be 2D or 3D.
    """
    inner = num_heads * key_dim
    if w.ndim == 2:
        # (H*K, out_dim)
        w2 = w
    elif w.ndim == 3:
        # Common layouts:
        # 1) (num_heads, key_dim, out_dim)
        # 2) (key_dim, num_heads, out_dim)
        if w.shape == (num_heads, key_dim, out_dim):
            w2 = w.reshape(inner, out_dim)
        elif w.shape == (key_dim, num_heads, out_dim):
            w2 = np.transpose(w, (1, 0, 2)).reshape(inner, out_dim)
        else:
            raise ValueError(f"Unexpected out kernel shape: {w.shape}")
    else:
        raise ValueError(f"Unexpected out kernel rank: {w.ndim}")

    # Bias
    if b is None:
        b2 = None
    elif b.ndim == 1:
        # (out_dim,)
        b2 = b
    else:
        b2 = b.reshape(-1)

    return w2, b2


def _tf_conv1d_to_torch(w_keras: np.ndarray) -> torch.Tensor:
    # TF Conv1D [K, Cin, Cout] -> PT Conv1d [Cout, Cin, K]
    return torch.from_numpy(np.transpose(w_keras, (2, 1, 0)))


def _transfer_mha_keras_to_pt_general(tf_mha: KerasMHA, pt_mha: MultiHeadAttentionGeneralPT):
    """
    Transfer query/key/value/output dense weights from Keras MHA (which may have different in_dims for q vs k/v)
    into our PT MultiHeadAttentionGeneralPT that has explicit q_in and kv_in projections.
    """
    def get_dense(obj, primary, fallback):
        d = getattr(obj, primary, None)
        if d is None:
            d = getattr(obj, fallback, None)
        return d

    qd = get_dense(tf_mha, "query_dense", "_query_dense")
    kd = get_dense(tf_mha, "key_dense", "_key_dense")
    vd = get_dense(tf_mha, "value_dense", "_value_dense")
    od = get_dense(tf_mha, "output_dense", "_output_dense")
    assert all([qd, kd, vd, od]), "Could not find Keras MHA Dense sublayers (query/key/value/output)."

    H = pt_mha.num_heads
    K = pt_mha.key_dim
    inner = H * K

    # q
    Wq, bq = qd.get_weights()  # (q_in, inner), (inner,)
    Wq2, bq2 = _flatten_qkv_kernel_bias(Wq, bq, pt_mha.q_in, H, K)
    pt_mha.q_proj.weight.data = torch.from_numpy(Wq2.T)
    pt_mha.q_proj.bias.data = torch.from_numpy(bq2) if bq2 is not None else torch.zeros(inner)

    # k
    Wk, bk = kd.get_weights()  # (kv_in, inner)
    Wk2, bk2 = _flatten_qkv_kernel_bias(Wk, bk, pt_mha.kv_in, H, K)
    pt_mha.k_proj.weight.data = torch.from_numpy(Wk2.T)
    pt_mha.k_proj.bias.data = torch.from_numpy(bk2) if bk2 is not None else torch.zeros(inner)

    # v
    Wv, bv = vd.get_weights()  # (kv_in, inner)
    Wv2, bv2 = _flatten_qkv_kernel_bias(Wv, bv, pt_mha.kv_in, H, K)
    pt_mha.v_proj.weight.data = torch.from_numpy(Wv2.T)
    pt_mha.v_proj.bias.data = torch.from_numpy(bv2) if bv2 is not None else torch.zeros(inner)

    # out
    Wo, bo = od.get_weights()  # (inner, out_dim==q_in)
    Wo2, bo2 = _flatten_out_kernel_bias(Wo, bo, pt_mha.q_in, H, K)
    pt_mha.out_proj.weight.data = torch.from_numpy(Wo2.T)
    pt_mha.out_proj.bias.data = torch.from_numpy(bo2) if bo2 is not None else torch.zeros(pt_mha.q_in)


def _collect_tf_td_decoder(tf_td_layer):
    """Collect TF sublayers from TransformerDecoder (custom layer)."""
    convs = [m for m in tf_td_layer.submodules if isinstance(m, Conv1D)]
    dec_layers = list(getattr(tf_td_layer, "dec_layers"))
    return convs, dec_layers


def transfer_generator_mlp_weights(tf_model: tf.keras.Model, pt_model: TFMDecoderPT):
    """
    Transfer Dense/BN weights from the generator MLP (g -> Dense -> BN -> Dense -> BN -> Dense -> BN -> RepeatVector)
    into pt_model.generator_mlp.
    """
    # Find the first RepeatVector layer; generator layers are before it.
    layers = tf_model.layers
    rv_idx = next(i for i, l in enumerate(layers) if isinstance(l, RepeatVector))
    prior = layers[:rv_idx]

    gens = [l for l in prior if isinstance(l, (Dense, BatchNormalization))]
    # Expected order: Dense(latent), BN, Dense(2*latent), BN, Dense(4*latent), BN
    assert len(gens) >= 6, "Could not find 3 Dense + 3 BatchNorm layers for generator MLP."
    d1, bn1, d2, bn2, d3, bn3 = gens[-6:]  # pick the last 6 just in case

    # PyTorch mapping: [0]=Linear, [1]=BN, [2]=Linear, [3]=ReLU, [4]=BN, [5]=Linear, [6]=ReLU, [7]=BN
    lin1: nn.Linear = pt_model.generator_mlp[0]; bn1_pt: BatchNorm1dKerasFP32 = pt_model.generator_mlp[1]
    lin2: nn.Linear = pt_model.generator_mlp[2]; bn2_pt: BatchNorm1dKerasFP32 = pt_model.generator_mlp[4]
    lin3: nn.Linear = pt_model.generator_mlp[5]; bn3_pt: BatchNorm1dKerasFP32 = pt_model.generator_mlp[7]

    # Dense 1
    w, b = d1.get_weights(); lin1.weight.data = torch.from_numpy(w.T); lin1.bias.data = torch.from_numpy(b)
    # BN 1
    gamma, beta, mmean, mvar = bn1.get_weights()
    bn1_pt.weight.data = torch.from_numpy(gamma); bn1_pt.bias.data = torch.from_numpy(beta)
    bn1_pt.running_mean.data = torch.from_numpy(mmean); bn1_pt.running_var.data = torch.from_numpy(mvar)

    # Dense 2
    w, b = d2.get_weights(); lin2.weight.data = torch.from_numpy(w.T); lin2.bias.data = torch.from_numpy(b)
    # BN 2
    gamma, beta, mmean, mvar = bn2.get_weights()
    bn2_pt.weight.data = torch.from_numpy(gamma); bn2_pt.bias.data = torch.from_numpy(beta)
    bn2_pt.running_mean.data = torch.from_numpy(mmean); bn2_pt.running_var.data = torch.from_numpy(mvar)

    # Dense 3
    w, b = d3.get_weights(); lin3.weight.data = torch.from_numpy(w.T); lin3.bias.data = torch.from_numpy(b)
    # BN 3
    gamma, beta, mmean, mvar = bn3.get_weights()
    bn3_pt.weight.data = torch.from_numpy(gamma); bn3_pt.bias.data = torch.from_numpy(beta)
    bn3_pt.running_mean.data = torch.from_numpy(mmean); bn3_pt.running_var.data = torch.from_numpy(mvar)


def transfer_probabilistic_decoder_weights(tf_model: tf.keras.Model, pt_model: TFMDecoderPT):
    """
    Transfer the Dense layer inside the TF ProbabilisticDecoder to PT ProbabilisticDecoderPT.loc_projection.
    """
    # Find the ProbabilisticDecoder custom layer (has attribute time_distributer)
    prob_layer = next(l for l in tf_model.layers if hasattr(l, "time_distributer"))
    d: Dense = prob_layer.time_distributer
    w, b = d.get_weights()
    pt_model.prob_decoder.loc_projection.weight.data = torch.from_numpy(w.T)
    pt_model.prob_decoder.loc_projection.bias.data = torch.from_numpy(b)


def transfer_transformer_decoder_weights(tf_model: tf.keras.Model, pt_model: TFMDecoderPT):
    """
    Transfer all weights:
      - Generator MLP
      - Decoder core: Conv1D embedding, per-layer MHA1, MHA2, FFN, LayerNorms
      - ProbabilisticDecoder output projection
    """
    # Generator MLP
    transfer_generator_mlp_weights(tf_model, pt_model)

    # Find the TF TransformerDecoder custom layer
    tf_td = next(l for l in tf_model.layers if hasattr(l, "dec_layers") and hasattr(l, "embedding"))
    convs, dec_layers = _collect_tf_td_decoder(tf_td)

    # Embedding Conv1D
    assert len(convs) >= 1, "No Conv1D embedding found in TF TransformerDecoder"
    k, b = convs[0].get_weights()  # TF: (K=1, in_ch=model_dim, out_ch=model_dim)
    pt_model.decoder.embed.weight.data = _tf_conv1d_to_torch(k)
    pt_model.decoder.embed.bias.data = torch.from_numpy(b)

    # Per-layer transfer
    assert len(dec_layers) == len(pt_model.decoder.layers), "Decoder layer count mismatch"
    for tf_el, pt_el in zip(dec_layers, pt_model.decoder.layers):
        # MHA1 (self-attn)
        _transfer_mha_keras_to_pt_general(tf_el.mha1, pt_el.mha1)
        # MHA2 (cross-attn)
        _transfer_mha_keras_to_pt_general(tf_el.mha2, pt_el.mha2)

        # FFN Dense
        d1, d2 = tf_el.ffn.layers  # Dense(dff), Dense(model_dim)
        W1, B1 = d1.get_weights(); W2, B2 = d2.get_weights()
        pt_el.ffn1.weight.data = torch.from_numpy(W1.T); pt_el.ffn1.bias.data = torch.from_numpy(B1)
        pt_el.ffn2.weight.data = torch.from_numpy(W2.T); pt_el.ffn2.bias.data = torch.from_numpy(B2)

        # LayerNorms
        _transfer_layernorm(tf_el.layernorm1, pt_el.norm1)
        _transfer_layernorm(tf_el.layernorm2, pt_el.norm2)
        _transfer_layernorm(tf_el.layernorm3, pt_el.norm3)

    # Probabilistic decoder projection
    transfer_probabilistic_decoder_weights(tf_model, pt_model)


In [5]:
import numpy as np
import torch
import tensorflow as tf

def keep_to_keras_mask_bool(keep_mask_tf: tf.Tensor) -> tf.Tensor:
    """
    Convert keep mask (True=keep) to Keras boolean mask (False=masked).
    Keras MHA interprets boolean mask where False entries are masked out.
    """
    return tf.cast(keep_mask_tf, tf.bool)  # simply ensure boolean; Keras treats False as masked, True as keep

def dense_forward_np(x_np, W, b):
    y = np.tensordot(x_np, W, axes=([2], [0]))  # (B, T, out_dim)
    if b is not None:
        y = y + b
    return y

def to_numpy(t):
    return t.detach().cpu().numpy() if isinstance(t, torch.Tensor) else t.numpy()

def check_close(name, a, b, rtol=1e-5, atol=2e-4):
    np.testing.assert_allclose(a, b, rtol=rtol, atol=atol)
    print(f"✅ {name} parity")



def get_tf_masks_from_memory(memory_np):
    """
    Uses TF's own create_masks (from deepof) to get masks exactly as TF decoder uses.
    Returns TF tensors:
      - combined_mask_tf: (B, 1, T, T) mask-out (1=masked)
      - dec_pad_tf:       (B, 1, 1, T) mask-out (1=masked)
    """
    import deepof
    memory_tf = tf.convert_to_tensor(memory_np, dtype=tf.float32)
    enc_pad, combined_mask_tf, dec_pad_tf = deepof.model_utils.create_masks(memory_tf)
    return combined_mask_tf, dec_pad_tf


def masks_tf_to_pt(combined_mask_tf, dec_pad_tf):
    """
    Convert TF mask tensors to torch tensors (boolean mask-out).
    Returns (B,1,T,T), (B,1,1,T) in torch.bool, plus squeezed 3D/2D.
    """
    comb_np = combined_mask_tf.numpy()
    pad_np = dec_pad_tf.numpy()

    comb_4d_pt = torch.from_numpy(comb_np.astype(np.bool_))  # (B,1,T,T)
    pad_4d_pt = torch.from_numpy(pad_np.astype(np.bool_))    # (B,1,1,T)

    comb_3d_pt = comb_4d_pt.squeeze(1)                       # (B,T,T)
    pad_2d_pt = pad_4d_pt.squeeze(1).squeeze(1)              # (B,T)

    return comb_4d_pt, pad_4d_pt, comb_3d_pt, pad_2d_pt


In [7]:
# ===========================
# Parity Tests for Decoder
# ===========================

import unittest
import numpy as np
import torch
import tensorflow as tf
import os

# Assumes the following are already defined/imported in your notebook:
# - get_transformer_decoder (TF)
# - TFMDecoderPT, MultiHeadAttentionGeneralPT, create_masks_pt
# - transfer_transformer_decoder_weights
# - sinusoidal_positional_encoding
# - ProbabilisticDecoderPT

# Determinism (optional but recommended)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

np.random.seed(0)
tf.random.set_seed(0)
torch.manual_seed(0)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

class TestTransformerDecoderPTUnits(unittest.TestCase):
    def setUp(self):
        tf.keras.backend.clear_session()
        torch.manual_seed(0); np.random.seed(0)

        # Shapes
        self.B = 4
        self.W = 12
        self.N = 11
        self.NF = 3
        self.D_in = self.N * self.NF
        self.latent_dim = 6

        # Data
        rng = np.random.default_rng(0)
        self.x_np = rng.normal(size=(self.B, self.W, self.D_in)).astype(np.float32)
        self.g_np = rng.normal(size=(self.B, self.latent_dim)).astype(np.float32)

        # Some zeros to exercise masks
        self.x_np[0, 3, :] = 0.0
        self.x_np[2, 7, :] = 0.0

        # Config
        self.num_layers = 1
        self.num_heads = 4
        self.dff = 64
        self.dropout = 0.0

        # Build models
        self.tf_model = get_transformer_decoder(
            input_shape=(self.W, self.D_in),
            latent_dim=self.latent_dim,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            dff=self.dff,
            dropout_rate=self.dropout,
        )
        self.pt_model = TFMDecoderPT(
            input_shape=(self.W, self.D_in),
            latent_dim=self.latent_dim,
            num_layers=self.num_layers,
            num_heads=self.num_heads,
            dff=self.dff,
            dropout_rate=self.dropout,
        ).eval()

        # Warm-up PT (positional buffer sizing etc.)
        with torch.no_grad():
            _ = self.pt_model(torch.from_numpy(self.g_np[:2]), torch.from_numpy(self.x_np[:2]))

        # Transfer weights
        transfer_transformer_decoder_weights(self.tf_model, self.pt_model)

        # Shortcuts to TF inner decoder
        self.tf_td = next(l for l in self.tf_model.layers if hasattr(l, "dec_layers") and hasattr(l, "embedding"))

    def diag_mha2_heads(self, out1_tf: np.ndarray, memory_np: np.ndarray, pad_tf_4d_bool: tf.Tensor, tol: float = 2e-4):
        """
        Prints per-head mismatch counts for attention maps of MHA2 (TF vs PT).
        Helps identify if 1/2/3 heads diverge (25/50/75% patterns).
        """
        # TF attention maps
        out_tf2, attn_tf = self.tf_td.dec_layers[0].mha2(
            query=out1_tf,
            key=memory_np,
            value=memory_np,
            attention_mask=pad_tf_4d_bool,
            return_attention_scores=True,
            training=False,
        )
        attn_tf = attn_tf.numpy()  # expected shape (B, H, Tq, Tk)

        # PT attention maps
        out1_t = torch.from_numpy(out1_tf)
        mem_t = torch.from_numpy(memory_np)
        pad_pt = torch.from_numpy(np.array(pad_tf_4d_bool, dtype=bool))
        with torch.no_grad():
            out_pt2, attn_pt = self.pt_model.decoder.layers[0].mha2(
                query=out1_t,
                key=mem_t,
                value=mem_t,
                attn_mask=pad_pt,
                return_attention_scores=True,
            )
        attn_pt = attn_pt.detach().cpu().numpy()  # (B, H, Tq, Tk)

        # Compare per head
        B, H, Tq, Tk = attn_tf.shape
        total_per_head = B * Tq * Tk
        mism_per_head = []
        for h in range(H):
            diff = np.abs(attn_tf[:, h] - attn_pt[:, h])
            mism = int((diff > tol).sum())
            mism_per_head.append(mism)
        print(f"MHA2 per-head mismatches: {mism_per_head} of total {total_per_head} per head (tol={tol})")

    def _tf_generator_pre_repeat(self, g_np):
        # Locate pre-Repeat generator stack: Dense -> BN -> Dense -> BN -> Dense -> BN
        layers = self.tf_model.layers
        rv_idx = next(i for i, l in enumerate(layers) if isinstance(l, tf.keras.layers.RepeatVector))
        prior = layers[:rv_idx]
        gens = [l for l in prior if isinstance(l, (tf.keras.layers.Dense, tf.keras.layers.BatchNormalization))]
        d1, bn1, d2, bn2, d3, bn3 = gens[-6:]
        sub = tf.keras.Model(inputs=self.tf_model.inputs[0], outputs=bn3.output)
        y = sub(g_np, training=False)
        return y.numpy()  # (B, 4*latent)

    def _tf_embedding_pos(self, x_np):
        x_tf = tf.convert_to_tensor(x_np, dtype=tf.float32)  # (B, W, D_in)
        y = self.tf_td.embedding(x_tf)                       # Conv1D + relu inside
        scale = tf.math.sqrt(tf.cast(self.tf_td.key_dim, tf.float32))
        y = y * scale
        y = y + self.tf_td.pos_encoding[:, :self.W, :]
        return y.numpy()

    def _pt_embedding_pos(self, x_np):
        x_t = torch.from_numpy(x_np)  # (B, W, D_in)
        y = self.pt_model.decoder.embed(x_t.transpose(1, 2)).transpose(1, 2)
        y = torch.relu(y)
        y = y * (self.pt_model.decoder.model_dim ** 0.5)
        pe = self.pt_model.decoder.pos_encoding[:, :self.W, :].to(y.dtype)
        y = y + pe
        return y.detach().cpu().numpy()

    def _pt_masks_from_generator(self, gen_rep_np):
        gen_t = torch.from_numpy(gen_rep_np)
        comb_3d, pad_2d = create_masks_pt(gen_t)   # (B,T,T), (B,T), True=masked-out
        return comb_3d, pad_2d

    def test_positional_encoding_parity(self):
        pe_tf = self.tf_td.pos_encoding[:, :self.W, :].numpy()
        pe_pt = self.pt_model.decoder.pos_encoding[:, :self.W, :].cpu().numpy()
        # Relaxed tolerance for tiny float32 differences
        np.testing.assert_allclose(pe_tf, pe_pt, rtol=1e-6, atol=1e-6)
        print("✅ Positional encoding parity")

    def test_generator_mlp_parity(self):
        gen_tf = self._tf_generator_pre_repeat(self.g_np)  # (B, 4*latent)
        with torch.no_grad():
            gen_pt = self.pt_model.generator_mlp(torch.from_numpy(self.g_np)).cpu().numpy()
        np.testing.assert_allclose(gen_tf, gen_pt, rtol=1e-5, atol=2e-4)
        print("✅ Generator MLP (pre-Repeat) parity")

    def test_embedding_block_parity(self):
        y_tf = self._tf_embedding_pos(self.x_np)  # (B, W, D_in)
        y_pt = self._pt_embedding_pos(self.x_np)
        np.testing.assert_allclose(y_tf, y_pt, rtol=1e-5, atol=2e-4)
        print("✅ Embedding + pos parity")

    def test_mha1_self_attention_parity(self):
        x0 = self._tf_embedding_pos(self.x_np)
        gen_tf = self._tf_generator_pre_repeat(self.g_np)
        memory_np = np.repeat(gen_tf[:, None, :], self.W, axis=1).astype(np.float32)

        comb_tf_4d, _ = get_tf_masks_from_memory(memory_np)
        comb_tf_4d_bool = tf.cast(comb_tf_4d, tf.bool)
        comb_pt_4d_bool = torch.from_numpy(np.array(comb_tf_4d_bool, dtype=bool))

        attn1_tf, _ = self.tf_td.dec_layers[0].mha1(query=x0, key=x0, value=x0, attention_mask=comb_tf_4d_bool, return_attention_scores=True, training=False)
        attn1_tf = attn1_tf.numpy()

        x0_t = torch.from_numpy(x0)
        attn1_pt, _ = self.pt_model.decoder.layers[0].mha1(query=x0_t, key=x0_t, value=x0_t, attn_mask=comb_pt_4d_bool, return_attention_scores=True)
        attn1_pt = attn1_pt.detach().cpu().numpy()

        np.testing.assert_allclose(attn1_tf, attn1_pt, rtol=1e-5, atol=2e-4)
        print("✅ MHA1 (self-attn) parity with TF masks")


    def test_mha2_cross_attention_parity(self):
        x0 = self._tf_embedding_pos(self.x_np)
        gen_tf = self._tf_generator_pre_repeat(self.g_np)
        memory_np = np.repeat(gen_tf[:, None, :], self.W, axis=1).astype(np.float32)

        comb_tf_4d, pad_tf_4d = get_tf_masks_from_memory(memory_np)
        comb_tf_4d_bool = tf.cast(comb_tf_4d, tf.bool)
        pad_tf_4d_bool  = tf.cast(pad_tf_4d, tf.bool)

        # TF out1 (post self-attn + norm)
        attn1_tf, _ = self.tf_td.dec_layers[0].mha1(
            query=x0, key=x0, value=x0,
            attention_mask=comb_tf_4d_bool,
            return_attention_scores=True,
            training=False,
        )
        out1_tf = self.tf_td.dec_layers[0].layernorm1(
            self.tf_td.dec_layers[0].dropout1(attn1_tf, training=False) + x0
        ).numpy()

        # PT out1
        x0_t = torch.from_numpy(x0)
        comb_pt_4d_bool = torch.from_numpy(np.array(comb_tf_4d_bool, dtype=bool))
        with torch.no_grad():
            attn1_pt, _ = self.pt_model.decoder.layers[0].mha1(
                query=x0_t, key=x0_t, value=x0_t,
                attn_mask=comb_pt_4d_bool,
                return_attention_scores=True,
            )
            out1_pt = self.pt_model.decoder.layers[0].norm1(
                x0_t + self.pt_model.decoder.layers[0].dropout1(attn1_pt)
            ).detach().cpu().numpy()

        np.testing.assert_allclose(out1_tf, out1_pt, rtol=1e-5, atol=2e-4)

        # Cross-attn outputs
        with torch.no_grad():
            attn2_tf_out, attn2_tf = self.tf_td.dec_layers[0].mha2(
                query=out1_tf, key=memory_np, value=memory_np,
                attention_mask=pad_tf_4d_bool,
                return_attention_scores=True,
                training=False,
            )
        attn2_tf_out = attn2_tf_out.numpy()

        out1_t = torch.from_numpy(out1_tf)
        mem_t = torch.from_numpy(memory_np)
        pad_pt_4d_bool = torch.from_numpy(np.array(pad_tf_4d_bool, dtype=bool))
        with torch.no_grad():
            attn2_pt_out, attn2_pt = self.pt_model.decoder.layers[0].mha2(
                query=out1_t, key=mem_t, value=mem_t,
                attn_mask=pad_pt_4d_bool,
                return_attention_scores=True,
            )
        attn2_pt_out = attn2_pt_out.detach().cpu().numpy()

        # Optional: per-head diagnostics before asserting
        self.diag_mha2_heads(out1_tf, memory_np, pad_tf_4d_bool)

        print("TF pad_tf_4d_bool shape:", pad_tf_4d_bool.shape, "dtype:", pad_tf_4d_bool.dtype)
        print("PT pad_pt_4d_bool shape:", pad_pt_4d_bool.shape, "dtype:", pad_pt_4d_bool.dtype)

        tf_keep_per_b = np.sum(pad_tf_4d_bool.numpy(), axis=(-1, -2, -3)) # (B,)
        pt_keep_per_b = np.sum(pad_pt_4d_bool.numpy(), axis=(-1, -2, -3)) # (B,)
        print("TF keep sum per batch:", tf_keep_per_b)
        print("PT keep sum per batch:", pt_keep_per_b)

        tf_all_masked_rows = (np.sum(pad_tf_4d_bool.numpy(), axis=-1) == 0) # (B,1,1,Tq)
        pt_all_masked_rows = (np.sum(pad_pt_4d_bool.numpy(), axis=-1) == 0) # (B,1,1,Tq)
        print("TF any all-masked rows per batch:", tf_all_masked_rows.any(axis=(-1,-2,-3)))
        print("PT any all-masked rows per batch:", pt_all_masked_rows.any(axis=(-1,-2,-3)))

        tf_layer = self.tf_td.dec_layers[0]
        qd = getattr(tf_layer.mha2, "query_dense", getattr(tf_layer.mha2, "_query_dense"))
        kd = getattr(tf_layer.mha2, "key_dense", getattr(tf_layer.mha2, "_key_dense"))
        vd = getattr(tf_layer.mha2, "value_dense", getattr(tf_layer.mha2, "_value_dense"))
        Wq, bq = qd.get_weights(); Wk, bk = kd.get_weights(); Wv, bv = vd.get_weights()


        np.testing.assert_allclose(attn2_tf_out, attn2_pt_out, rtol=1e-5, atol=2e-4)
        print("✅ MHA2 (cross-attn) parity with TF masks")

    def test_decoder_layer0_parity(self):
        x0 = self._tf_embedding_pos(self.x_np)
        gen_tf = self._tf_generator_pre_repeat(self.g_np)
        memory_tf = np.repeat(gen_tf[:, None, :], self.W, axis=1)

        comb_mask_pt, pad_mask_pt = self._pt_masks_from_generator(memory_tf)
        comb_mask_tf = tf.convert_to_tensor(comb_mask_pt.numpy(), dtype=tf.bool)
        pad_mask_tf = tf.convert_to_tensor(pad_mask_pt.numpy(), dtype=tf.bool)

        # TF layer call: Keras accepts (B, T, T) and (B, T)
        out3_tf, _, _ = self.tf_td.dec_layers[0](
            x0, memory_tf, training=False, look_ahead_mask=comb_mask_tf, padding_mask=pad_mask_tf
        )
        out3_tf = out3_tf.numpy()

        # PT layer call
        x0_t = torch.from_numpy(x0); mem_t = torch.from_numpy(memory_tf)
        out3_pt, _, _ = self.pt_model.decoder.layers[0](
            x0_t, mem_t, comb_mask_pt, pad_mask_pt,
        )
        out3_pt = out3_pt.detach().cpu().numpy()

        np.testing.assert_allclose(out3_tf, out3_pt, rtol=1e-5, atol=2e-4)
        print("✅ Decoder layer[0] parity") 


    def test_decoder_core_parity(self):
        gen_tf = self._tf_generator_pre_repeat(self.g_np)
        memory_tf = np.repeat(gen_tf[:, None, :], self.W, axis=1)

        comb_mask_pt, pad_mask_pt = self._pt_masks_from_generator(memory_tf)
        comb_mask_tf = tf.convert_to_tensor(comb_mask_pt.numpy(), dtype=tf.bool)
        pad_mask_tf = tf.convert_to_tensor(pad_mask_pt.numpy(), dtype=tf.bool)

        x_tf = tf.convert_to_tensor(self.x_np, dtype=tf.float32)
        y_tf, _att = self.tf_td(x_tf, memory_tf, training=False, look_ahead_mask=comb_mask_tf, padding_mask=pad_mask_tf)
        y_tf = y_tf.numpy()

        x_t = torch.from_numpy(self.x_np); mem_t = torch.from_numpy(memory_tf)
        y_pt, _ = self.pt_model.decoder(x_t, mem_t, comb_mask_pt, pad_mask_pt, training=False)
        y_pt = y_pt.detach().cpu().numpy()

        np.testing.assert_allclose(y_tf, y_pt, rtol=1e-5, atol=2e-4)
        print("✅ Decoder core parity")

    def test_full_decoder_parity(self):
        y_tf_dist, attn_tf = self.tf_model([self.g_np, self.x_np], training=False)
        y_tf = tf.convert_to_tensor(y_tf_dist).numpy()

        with torch.no_grad():
            y_pt, attn_pt = self.pt_model(torch.from_numpy(self.g_np), torch.from_numpy(self.x_np))
            y_pt = y_pt.cpu().numpy()

        np.testing.assert_allclose(y_tf, y_pt, rtol=1e-5, atol=2e-4)
        print("✅ Full decoder parity")

    
    def test_mask_generation_parity(self):
        # Build memory from TF generator
        gen_tf = self._tf_generator_pre_repeat(self.g_np)  # (B, 4*latent)
        memory_np = np.repeat(gen_tf[:, None, :], self.W, axis=1).astype(np.float32)

        # TF masks (mask-out floats or bools as returned by TF)
        comb_tf_4d, pad_tf_4d = get_tf_masks_from_memory(memory_np)  # (B,1,T,T), (B,1,1,T)
        # Convert TF to bool (True=masked-out)
        comb_tf_4d_bool = tf.cast(comb_tf_4d, tf.bool)
        pad_tf_4d_bool = tf.cast(pad_tf_4d, tf.bool)

        # PT masks from our replica
        comb_pt_4d_bool, pad_pt_4d_bool = create_masks_pt(torch.from_numpy(memory_np))

        # Compare 4D directly to avoid squeeze confusion
        np.testing.assert_array_equal(comb_tf_4d_bool.numpy(), comb_pt_4d_bool.numpy())
        np.testing.assert_array_equal(pad_tf_4d_bool.numpy(), pad_pt_4d_bool.numpy())
        print("✅ TF vs PT mask parity (values, 4D) OK")


    def test_keras_mha_mask_shape_acceptance(self):
        x0 = self._tf_embedding_pos(self.x_np)  # (B, T, D)
        gen_tf = self._tf_generator_pre_repeat(self.g_np)
        memory_np = np.repeat(gen_tf[:, None, :], self.W, axis=1).astype(np.float32)

        # TF masks
        comb_tf_4d, pad_tf_4d = get_tf_masks_from_memory(memory_np)  # (B,1,T,T), (B,1,1,T)

        # 3D/2D variants
        comb_tf_3d = tf.squeeze(comb_tf_4d, axis=1)                        # (B,T,T)
        pad_tf_2d  = tf.squeeze(tf.squeeze(pad_tf_4d, axis=1), axis=1)     # (B,T)

        # Probe MHA1
        try:
            _ = self.tf_td.dec_layers[0].mha1(query=x0, key=x0, value=x0, attention_mask=comb_tf_4d, return_attention_scores=True)
            print("✅ Keras MHA1 accepts 4D mask (B,1,T,T)")
        except Exception as e:
            print("❌ Keras MHA1 4D mask rejected:", e)
            _ = self.tf_td.dec_layers[0].mha1(query=x0, key=x0, value=x0, attention_mask=comb_tf_3d, return_attention_scores=True)
            print("✅ Keras MHA1 accepts 3D mask (B,T,T)")

        # Probe MHA2
        try:
            _ = self.tf_td.dec_layers[0].mha2(query=x0, key=memory_np, value=memory_np, attention_mask=pad_tf_4d, return_attention_scores=True)
            print("✅ Keras MHA2 accepts 4D mask (B,1,1,T)")
        except Exception as e:
            print("❌ Keras MHA2 4D mask rejected:", e)
            _ = self.tf_td.dec_layers[0].mha2(query=x0, key=memory_np, value=memory_np, attention_mask=pad_tf_2d, return_attention_scores=True)
            print("✅ Keras MHA2 accepts 2D mask (B,T)")

    def _pick_mask_for_keras_mha1(self, comb_tf_4d):
        # Try 4D then 3D
        comb_tf_3d = tf.squeeze(comb_tf_4d, axis=1)  # (B,T,T)
        x0 = self._tf_embedding_pos(self.x_np)
        try:
            _ = self.tf_td.dec_layers[0].mha1(query=x0, key=x0, value=x0, attention_mask=comb_tf_4d, return_attention_scores=False)
            return comb_tf_4d
        except:
            return comb_tf_3d

    def _pick_mask_for_keras_mha2(self, pad_tf_4d):
        # Try 4D then 2D
        x0 = self._tf_embedding_pos(self.x_np)
        gen_tf = self._tf_generator_pre_repeat(self.g_np)
        memory_np = np.repeat(gen_tf[:, None, :], self.W, axis=1).astype(np.float32)
        pad_tf_2d = tf.squeeze(tf.squeeze(pad_tf_4d, axis=1), axis=1)  # (B,T)
        try:
            _ = self.tf_td.dec_layers[0].mha2(query=x0, key=memory_np, value=memory_np, attention_mask=pad_tf_4d, return_attention_scores=False)
            return pad_tf_4d
        except:
            return pad_tf_2d

    def test_mha_parity_with_exact_tf_masks(self):
        x0 = self._tf_embedding_pos(self.x_np)
        gen_tf = self._tf_generator_pre_repeat(self.g_np)
        memory_np = np.repeat(gen_tf[:, None, :], self.W, axis=1).astype(np.float32)

        # TF masks (mask-out)
        comb_tf_4d, pad_tf_4d = get_tf_masks_from_memory(memory_np)  # (B,1,T,T), (B,1,1,T)
        comb_tf_4d_bool = tf.cast(comb_tf_4d, tf.bool)
        pad_tf_4d_bool  = tf.cast(pad_tf_4d, tf.bool)

        # PT masks
        comb_pt_4d_bool = torch.from_numpy(np.array(comb_tf_4d_bool, dtype=bool))
        pad_pt_4d_bool  = torch.from_numpy(np.array(pad_tf_4d_bool, dtype=bool))

        # MHA1
        attn1_tf, _ = self.tf_td.dec_layers[0].mha1(
            query=x0, key=x0, value=x0,
            attention_mask=comb_tf_4d_bool, return_attention_scores=True,
            training=False,
        )
        attn1_tf = attn1_tf.numpy()

        x0_t = torch.from_numpy(x0)
        attn1_pt, _ = self.pt_model.decoder.layers[0].mha1(
            query=x0_t, key=x0_t, value=x0_t,
            attn_mask=comb_pt_4d_bool, return_attention_scores=True,
        )
        attn1_pt = attn1_pt.detach().cpu().numpy()

        np.testing.assert_allclose(attn1_tf, attn1_pt, rtol=1e-5, atol=2e-4)
        print("✅ MHA1 parity with TF masks (mask-out)")

        # Out1 parity (post norm)
        out1_tf = self.tf_td.dec_layers[0].layernorm1(self.tf_td.dec_layers[0].dropout1(attn1_tf, training=False) + x0).numpy()
        out1_pt = self.pt_model.decoder.layers[0].norm1(x0_t + self.pt_model.decoder.layers[0].dropout1(torch.from_numpy(attn1_pt))).detach().cpu().numpy()
        np.testing.assert_allclose(out1_tf, out1_pt, rtol=1e-5, atol=2e-4)

        # MHA2 (cross-attn)
        attn2_tf, _ = self.tf_td.dec_layers[0].mha2(
            query=out1_tf, key=memory_np, value=memory_np,
            attention_mask=pad_tf_4d_bool, return_attention_scores=True,
            training=False,
        )
        attn2_tf = attn2_tf.numpy()

        out1_t = torch.from_numpy(out1_tf)
        mem_t = torch.from_numpy(memory_np)
        attn2_pt, _ = self.pt_model.decoder.layers[0].mha2(
            query=out1_t, key=mem_t, value=mem_t,
            attn_mask=pad_pt_4d_bool, return_attention_scores=True,
        )
        attn2_pt = attn2_pt.detach().cpu().numpy()

        np.testing.assert_allclose(attn2_tf, attn2_pt, rtol=1e-5, atol=2e-4)
        print("✅ MHA2 parity with TF masks (mask-out)")
    


# Run the unit tests
runner = unittest.TextTestRunner(verbosity=2)
suite = unittest.TestLoader().loadTestsFromTestCase(TestTransformerDecoderPTUnits)
runner.run(suite)

test_decoder_core_parity (__main__.TestTransformerDecoderPTUnits) ... ok
test_decoder_layer0_parity (__main__.TestTransformerDecoderPTUnits) ... 

✅ Decoder core parity


ok
test_embedding_block_parity (__main__.TestTransformerDecoderPTUnits) ... 

✅ Decoder layer[0] parity


ok
test_full_decoder_parity (__main__.TestTransformerDecoderPTUnits) ... 

...

ok
test_positional_encoding_parity (__main__.TestTransformerDecoderPTUnits) ... 

✅ MHA1 parity with TF masks (mask-out)
✅ MHA2 parity with TF masks (mask-out)


ok

----------------------------------------------------------------------
Ran 11 tests in 4.469s

OK


✅ Positional encoding parity


<unittest.runner.TextTestResult run=11 errors=0 failures=0>